In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 0 — SETUP (monter Drive, GPU, dépendances, chemins)
# ══════════════════════════════════════════════════════════════

# ── Monter Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Vérifier GPU ───────────────────────────────────────────────
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Installer les dépendances ──────────────────────────────────
!pip install transformers accelerate sentence-transformers chromadb groq shap mlflow -q
!pip install mitreattack-python stix2 requests aiohttp -q

# ── Secrets Colab (clés API — ne jamais hardcoder) ─────────────
# Dans Colab : icône 🔑 à gauche → "Secrets" → ajoute tes clés
# Puis décommente les lignes ci-dessous :
#
# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"]  = userdata.get('GROQ_API_KEY')
# os.environ["NGROK_TOKEN"]   = userdata.get('NGROK_TOKEN')
# os.environ["VT_API_KEY"]    = userdata.get('VT_API_KEY')
# os.environ["ABUSEIPDB_KEY"] = userdata.get('ABUSEIPDB_KEY')
# os.environ["SLACK_TOKEN"]   = userdata.get('SLACK_TOKEN')
#
# Alternative — charger un .env depuis Drive (jamais dans le repo git) :
#
# from pathlib import Path
# env_path = "/content/drive/MyDrive/phishing_dasec/.env"
# if Path(env_path).exists():
#     with open(env_path) as f:
#         for line in f:
#             line = line.strip()
#             if line and not line.startswith("#") and "=" in line:
#                 key, val = line.split("=", 1)
#                 os.environ[key.strip()] = val.strip()
#     print("✅ Variables d'environnement chargées depuis Drive")

# ── Définir les chemins (une seule fois, partagés dans tout le notebook) ──
import os

BASE_DIR       = "/content/drive/MyDrive/phishing_dasec"
DATA_DIR       = f"{BASE_DIR}/raw"
PROCESSED_DIR  = f"{BASE_DIR}/processed"
MODELS_DIR     = f"{BASE_DIR}/models"
VECTORSTORE_DIR= f"{BASE_DIR}/vectorstore"
IOC_CACHE_DIR  = f"{BASE_DIR}/ioc_cache"
MLFLOW_DIR     = f"{BASE_DIR}/mlflow"

for d in [DATA_DIR, PROCESSED_DIR, MODELS_DIR, VECTORSTORE_DIR, IOC_CACHE_DIR, MLFLOW_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Setup terminé")
print(f"📁 Fichiers disponibles : {os.listdir(DATA_DIR)}")

Mounted at /content/drive
GPU disponible : False
GPU : CPU
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 1 — DONNÉES (fusion 7 datasets, preprocessing, split)
# Prérequis : CELLULE 0 exécutée (BASE_DIR, DATA_DIR, PROCESSED_DIR définis)
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split

# ── 1. Aperçu rapide des colonnes de chaque dataset ──────────
print("📋 Aperçu des colonnes :")
files_all = {
    "ceas08":       "CEAS_08.csv",
    "enron":        "Enron.csv",
    "ling":         "Ling.csv",
    "nazario":      "Nazario.csv",
    "nigerian":     "Nigerian_Fraud.csv",
    "phishing":     "phishing_email.csv",
    "spamassassin": "SpamAssasin.csv",
}
for name, filename in files_all.items():
    path = os.path.join(DATA_DIR, filename)
    df   = pd.read_csv(path, nrows=2)
    print(f"  {name:15} → colonnes : {list(df.columns)}")

# ── 2. Fusion adaptée à chaque structure ─────────────────────
dfs = []

files_subjectbody = {
    "ceas08":       "CEAS_08.csv",
    "enron":        "Enron.csv",
    "ling":         "Ling.csv",
    "nazario":      "Nazario.csv",
    "nigerian":     "Nigerian_Fraud.csv",
    "spamassassin": "SpamAssasin.csv",
}

for name, filename in files_subjectbody.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df["subject"] = df["subject"].fillna("") if "subject" in df.columns else ""
    df["body"]    = df["body"].fillna("")    if "body"    in df.columns else ""
    df["text"]    = df["subject"] + " " + df["body"]
    df["source"]  = name
    dfs.append(df[["text", "label", "source"]])
    print(f"✅ {name:15} → {len(df)} emails | labels : {df['label'].value_counts().to_dict()}")

# phishing_email.csv — structure différente (text_combined)
df_ph = pd.read_csv(os.path.join(DATA_DIR, "phishing_email.csv"))
df_ph = df_ph.rename(columns={"text_combined": "text"})
df_ph["source"] = "phishing"
dfs.append(df_ph[["text", "label", "source"]])
print(f"✅ {'phishing':15} → {len(df_ph)} emails | labels : {df_ph['label'].value_counts().to_dict()}")

# ── 3. Concat + nettoyage ─────────────────────────────────────
full_df = pd.concat(dfs, ignore_index=True)
full_df = full_df.dropna(subset=["text", "label"])
full_df["text"]  = full_df["text"].str.strip()
full_df["label"] = full_df["label"].astype(int)
full_df = full_df[full_df["text"] != ""]

print(f"\n📊 Dataset complet : {len(full_df)} emails")
print(f"Distribution labels :\n{full_df['label'].value_counts().sort_index()}")
print(f"\nDistribution par source :\n{full_df['source'].value_counts()}")

# ── 4. Prétraitement ──────────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    text = re.sub(r'\d+', ' NUM ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

full_df["text_clean"] = full_df["text"].apply(clean_text)

# ── 5. Échantillon 5000 emails équilibré ──────────────────────
df_phishing = full_df[full_df["label"] == 1].sample(2500, random_state=42)
df_legitime = full_df[full_df["label"] == 0].sample(2500, random_state=42)
sample_5000 = pd.concat([df_phishing, df_legitime]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Échantillon 5000 emails")
print(f"Distribution : {sample_5000['label'].value_counts().to_dict()}")

# ── 6. Split 70/15/15 ─────────────────────────────────────────
X = sample_5000["text_clean"]
y = sample_5000["label"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\n📊 Split dataset :")
print(f"  Train : {len(X_train)} emails ({y_train.value_counts().to_dict()})")
print(f"  Val   : {len(X_val)}   emails ({y_val.value_counts().to_dict()})")
print(f"  Test  : {len(X_test)}  emails ({y_test.value_counts().to_dict()})")

# ── 7. Sauvegarde ─────────────────────────────────────────────
# Reconstruire les DataFrames à partir des index (split sur les index de sample_5000)
train_df = sample_5000.iloc[X_train.index].reset_index(drop=True)
val_df   = sample_5000.iloc[X_val.index].reset_index(drop=True)
test_df  = sample_5000.iloc[X_test.index].reset_index(drop=True)

sample_5000.to_csv(f"{PROCESSED_DIR}/sample_5000.csv", index=False)
train_df.to_csv(f"{PROCESSED_DIR}/train.csv", index=False)
val_df.to_csv(f"{PROCESSED_DIR}/val.csv",     index=False)
test_df.to_csv(f"{PROCESSED_DIR}/test.csv",   index=False)

print(f"\n✅ Sauvegardé dans {PROCESSED_DIR}")

# Convertir en listes pour les cellules suivantes
X_train = X_train.tolist()
X_val   = X_val.tolist()
X_test  = X_test.tolist()
y_train = y_train.tolist()
y_val   = y_val.tolist()
y_test  = y_test.tolist()

📋 Aperçu des colonnes :
  ceas08          → colonnes : ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
  enron           → colonnes : ['subject', 'body', 'label']
  ling            → colonnes : ['subject', 'body', 'label']
  nazario         → colonnes : ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
  nigerian        → colonnes : ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
  phishing        → colonnes : ['text_combined', 'label']
  spamassassin    → colonnes : ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
✅ ceas08          → 39154 emails | labels : {1: 21842, 0: 17312}
✅ enron           → 29767 emails | labels : {0: 15791, 1: 13976}
✅ ling            → 2859 emails | labels : {0: 2401, 1: 458}
✅ nazario         → 1565 emails | labels : {1: 1565}
✅ nigerian        → 3332 emails | labels : {1: 3332}
✅ spamassassin    → 5809 emails | labels : {0: 4091, 1: 1718}
✅ phishing        → 82486 emails | labels : {1:

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 2 — DISTILBERT (fine-tuning GPU T4 + évaluation)
# Prérequis : CELLULE 1 exécutée (X_train/val/test, y_train/val/test, PROCESSED_DIR)
# ══════════════════════════════════════════════════════════════

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer, TrainingArguments,
)
from torch.utils.data import Dataset
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

# ── 1. Tokenisation ───────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(X_train, truncation=True, padding=True, max_length=256)
val_enc   = tokenizer(X_val,   truncation=True, padding=True, max_length=256)
test_enc  = tokenizer(X_test,  truncation=True, padding=True, max_length=256)

# ── 2. Dataset PyTorch ────────────────────────────────────────
class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = EmailDataset(train_enc, y_train)
val_dataset   = EmailDataset(val_enc,   y_val)
test_dataset  = EmailDataset(test_enc,  y_test)

# ── 3. Modèle ─────────────────────────────────────────────────
bert_model_train = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

# ── 4. Métriques complètes ────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=1)
    probas = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    return {
        "f1":        round(f1_score(labels, preds), 4),
        "precision": round(precision_score(labels, preds), 4),
        "recall":    round(recall_score(labels, preds), 4),
        "auc":       round(roc_auc_score(labels, probas), 4),
    }

# ── 5. Entraînement GPU ───────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = f"{MODELS_DIR}/distilbert-5000",
    num_train_epochs            = 4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "no",       # évite PermissionError Windows/Drive
    load_best_model_at_end      = False,       # doit être False si save_strategy="no"
    metric_for_best_model       = "f1",
    logging_steps               = 50,
    report_to                   = "none",
)

trainer = Trainer(
    model           = bert_model_train,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print("🚀 Démarrage fine-tuning DistilBERT sur GPU T4...")
print(f"   Train : {len(train_dataset)} | Val : {len(val_dataset)} | Test : {len(test_dataset)}")
trainer.train()

# ── 6. Évaluation finale sur test set ─────────────────────────
print("\n📊 Évaluation sur le TEST SET (jamais vu) :")
preds_output = trainer.predict(test_dataset)
bert_logits  = preds_output.predictions

# IMPORTANT : variables nommées bert_* pour ne pas écraser les baselines TF-IDF ensuite
y_pred_bert  = np.argmax(bert_logits, axis=1)
y_proba_bert = torch.softmax(torch.tensor(bert_logits), dim=1).numpy()[:, 1]

print(classification_report(y_test, y_pred_bert, target_names=["Légitime", "Phishing"]))
bert_f1        = round(f1_score(y_test, y_pred_bert), 4)
bert_precision = round(precision_score(y_test, y_pred_bert), 4)
bert_recall    = round(recall_score(y_test, y_pred_bert), 4)
bert_auc       = round(roc_auc_score(y_test, y_proba_bert), 4)

print(f"  F1        : {bert_f1}")
print(f"  Precision : {bert_precision}")
print(f"  Recall    : {bert_recall}")
print(f"  AUC-ROC   : {bert_auc}")

# Stocker les résultats pour le tableau comparatif (cellule suivante)
distilbert_results = {
    "f1":        bert_f1,
    "precision": bert_precision,
    "recall":    bert_recall,
    "auc":       bert_auc,
}

# ── 7. Matrice de confusion + ROC ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_bert,
    display_labels=["Légitime", "Phishing"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Matrice de confusion — DistilBERT 5000 emails")

RocCurveDisplay.from_predictions(y_test, y_proba_bert, ax=axes[1], name="DistilBERT")
axes[1].set_title("Courbe ROC")
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{PROCESSED_DIR}/evaluation_distilbert_5000.png", dpi=150)
plt.show()

# ── 8. Sauvegarde modèle ──────────────────────────────────────
BERT_PATH = f"{MODELS_DIR}/distilbert-5000/final"
os.makedirs(BERT_PATH, exist_ok=True)
bert_model_train.save_pretrained(BERT_PATH)
tokenizer.save_pretrained(BERT_PATH)
print(f"\n✅ Modèle DistilBERT sauvegardé dans {BERT_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Démarrage fine-tuning DistilBERT sur GPU T4...
   Train : 3500 | Val : 750 | Test : 750


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 3 — BASELINES TF-IDF + MLFLOW + SHAP + PICKLE
# Prérequis : CELLULE 2 exécutée (distilbert_results, y_test, X_train/test définis)
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import mlflow
import mlflow.sklearn
import shap

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)

# ══════════════════════════════════════════════════════════════
# PARTIE A — Baselines TF-IDF
# ══════════════════════════════════════════════════════════════

# ── TF-IDF ────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=10000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

# ── Modèles ───────────────────────────────────────────────────
clf_models = {
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
}

baseline_results = {}
for name, clf in clf_models.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred_clf  = clf.predict(X_test_tfidf)
    y_proba_clf = clf.predict_proba(X_test_tfidf)[:, 1]
    baseline_results[name] = {
        "f1":        round(f1_score(y_test, y_pred_clf), 4),
        "precision": round(precision_score(y_test, y_pred_clf), 4),
        "recall":    round(recall_score(y_test, y_pred_clf), 4),
        "auc":       round(roc_auc_score(y_test, y_proba_clf), 4),
    }
    print(f"\n📊 {name}")
    print(classification_report(y_test, y_pred_clf, target_names=["Légitime", "Phishing"]))

# Références rapides pour SHAP et FastAPI
lr_model = clf_models["LogisticRegression"]
rf_model = clf_models["RandomForest"]

# ── Tableau comparatif ────────────────────────────────────────
comparison = pd.DataFrame({
    "Modèle": [
        "TF-IDF + LogisticRegression",
        "TF-IDF + RandomForest",
        "DistilBERT fine-tuné (5000)",
    ],
    "F1":        [
        baseline_results["LogisticRegression"]["f1"],
        baseline_results["RandomForest"]["f1"],
        distilbert_results["f1"],                    # ← provient de CELLULE 2
    ],
    "Precision": [
        baseline_results["LogisticRegression"]["precision"],
        baseline_results["RandomForest"]["precision"],
        distilbert_results["precision"],
    ],
    "Recall": [
        baseline_results["LogisticRegression"]["recall"],
        baseline_results["RandomForest"]["recall"],
        distilbert_results["recall"],
    ],
    "AUC-ROC": [
        baseline_results["LogisticRegression"]["auc"],
        baseline_results["RandomForest"]["auc"],
        distilbert_results["auc"],
    ],
})

print("\n" + "="*65)
print("📈 COMPARAISON FINALE — tous modèles sur 750 emails de test")
print("="*65)
print(comparison.to_string(index=False))
comparison.to_csv(f"{PROCESSED_DIR}/comparaison_modeles.csv", index=False)
print(f"\n✅ Tableau comparatif sauvegardé")

# ══════════════════════════════════════════════════════════════
# PARTIE B — MLflow tracking
# ══════════════════════════════════════════════════════════════

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.makedirs(MLFLOW_DIR, exist_ok=True)
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment("dasec-phishing-detection")

# Logger les baselines
for model_name, metrics in baseline_results.items():
    with mlflow.start_run(run_name=f"TF-IDF_{model_name}"):
        mlflow.log_param("model_type",   model_name)
        mlflow.log_param("vectorizer",   "TF-IDF")
        mlflow.log_param("max_features", 10000)
        mlflow.log_param("ngram_range",  "(1,2)")
        mlflow.log_param("train_size",   len(X_train))
        mlflow.log_param("test_size",    len(X_test))
        mlflow.log_metric("f1",        metrics["f1"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall",    metrics["recall"])
        mlflow.log_metric("auc",       metrics["auc"])
        print(f"✅ Run loggé : TF-IDF_{model_name}")

# Logger DistilBERT
with mlflow.start_run(run_name="DistilBERT_finetuned_5000"):
    mlflow.log_param("model_type",   "DistilBertForSequenceClassification")
    mlflow.log_param("base_model",   "distilbert-base-uncased")
    mlflow.log_param("max_length",   256)
    mlflow.log_param("batch_size",   32)
    mlflow.log_param("epochs",       4)
    mlflow.log_param("warmup_steps", 100)
    mlflow.log_param("weight_decay", 0.01)
    mlflow.log_param("train_size",   len(X_train))
    mlflow.log_param("test_size",    len(X_test))
    mlflow.log_param("device",       "GPU T4")
    mlflow.log_metric("f1",        distilbert_results["f1"])
    mlflow.log_metric("precision", distilbert_results["precision"])
    mlflow.log_metric("recall",    distilbert_results["recall"])
    mlflow.log_metric("auc",       distilbert_results["auc"])
    mlflow.log_artifact(f"{PROCESSED_DIR}/comparaison_modeles.csv")
    mlflow.log_artifact(f"{PROCESSED_DIR}/evaluation_distilbert_5000.png")
    print("✅ Run loggé : DistilBERT_finetuned_5000")

runs = mlflow.search_runs(experiment_names=["dasec-phishing-detection"])
cols = ["tags.mlflow.runName", "metrics.f1", "metrics.precision", "metrics.recall", "metrics.auc"]
print("\n📊 Résumé MLflow :")
print(runs[cols].sort_values("metrics.f1", ascending=False).to_string(index=False))

# ══════════════════════════════════════════════════════════════
# PARTIE C — SHAP (LogisticRegression + TF-IDF)
# ══════════════════════════════════════════════════════════════

feature_names = tfidf.get_feature_names_out()
explainer     = shap.LinearExplainer(lr_model, X_train_tfidf, feature_perturbation="interventional")

X_test_sample = X_test_tfidf[:200]
shap_values   = explainer.shap_values(X_test_sample)

shap_explanation = shap.Explanation(
    values        = shap_values,
    data          = X_test_sample.toarray(),
    feature_names = feature_names
)

# Beeswarm
print("📊 Génération SHAP beeswarm plot...")
plt.figure()
shap.plots.beeswarm(shap_explanation, max_display=20, show=False)
plt.title("SHAP — Top 20 features les plus influentes (LogisticRegression)")
plt.tight_layout()
plt.savefig(f"{PROCESSED_DIR}/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ shap_beeswarm.png sauvegardé")

# Bar
plt.figure()
shap.plots.bar(shap_explanation, max_display=20, show=False)
plt.title("SHAP — Importance moyenne des features")
plt.tight_layout()
plt.savefig(f"{PROCESSED_DIR}/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ shap_bar.png sauvegardé")

# Waterfall individuel
test_df_reset = pd.read_csv(f"{PROCESSED_DIR}/test.csv").reset_index(drop=True)

idx_phishing = test_df_reset[test_df_reset["label"] == 1].index[0]
idx_legitime = test_df_reset[test_df_reset["label"] == 0].index[0]

for idx, label_name in [(idx_phishing, "PHISHING"), (idx_legitime, "LEGITIME")]:
    if idx >= 200:
        continue
    print(f"\n--- Explication individuelle : {label_name} ---")
    print(f"Texte : {test_df_reset['text_clean'][idx][:120]}...")
    single_shap = shap.Explanation(
        values        = shap_values[idx],
        base_values   = explainer.expected_value,
        data          = X_test_sample[idx].toarray()[0],
        feature_names = feature_names
    )
    plt.figure()
    shap.plots.waterfall(single_shap, max_display=15, show=False)
    plt.title(f"SHAP Waterfall — Email {label_name}")
    plt.tight_layout()
    plt.savefig(f"{PROCESSED_DIR}/shap_waterfall_{label_name.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ shap_waterfall_{label_name.lower()}.png sauvegardé")

mean_shap = np.abs(shap_values).mean(axis=0)
top_idx   = mean_shap.argsort()[-20:][::-1]
print("\n🔑 Top 20 mots les plus influents globalement :")
for i in top_idx:
    print(f"  {feature_names[i]:30} → SHAP moyen : {mean_shap[i]:.4f}")

# ══════════════════════════════════════════════════════════════
# PARTIE D — Sauvegarde artifacts pour FastAPI
# (ici, pas à la fin du notebook — pour survivre à un crash)
# ══════════════════════════════════════════════════════════════

artifacts = {
    "tfidf":    tfidf,
    "lr_model": lr_model,
}
with open(f"{MODELS_DIR}/tfidf_lr_pipeline.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("\n✅ TF-IDF + LR sauvegardé → tfidf_lr_pipeline.pkl")
print(f"✅ DistilBERT sauvegardé  → {MODELS_DIR}/distilbert-5000/final")
print(f"\n📦 Fichiers à télécharger depuis Drive :")
print(f"  {MODELS_DIR}/tfidf_lr_pipeline.pkl")
print(f"  {MODELS_DIR}/distilbert-5000/final/  (dossier complet)")
print(f"  {PROCESSED_DIR}/shap_beeswarm.png")
print(f"  {PROCESSED_DIR}/shap_bar.png")
print(f"  {PROCESSED_DIR}/comparaison_modeles.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE A — INSTALLATION DÉPENDANCES (MITRE + ChromaDB + IoC)
# À exécuter une seule fois par session Colab
# ══════════════════════════════════════════════════════════════

!pip install mitreattack-python stix2 chromadb sentence-transformers -q
!pip install requests aiohttp -q

import requests
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer

# Vérifier que VECTORSTORE_DIR est bien défini (depuis CELLULE 0)
# Si tu relances depuis ici, redéfinis-le manuellement :
# VECTORSTORE_DIR = "/content/drive/MyDrive/phishing_dasec/vectorstore"
# os.makedirs(VECTORSTORE_DIR, exist_ok=True)

print("✅ Dépendances installées")
print(f"📁 Vectorstore : {VECTORSTORE_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE B — CHROMADB (3 collections 100% automatiques depuis STIX)
#   - mitre_techniques  : toutes les techniques ATT&CK
#   - mitre_apt_groups  : tous les groupes APT
#   - threat_intel      : profils APT enrichis (groupe + techniques utilisées)
#
# Prérequis : CELLULE A exécutée, VECTORSTORE_DIR défini
# ══════════════════════════════════════════════════════════════

import requests
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# ── Init ChromaDB + embedder ──────────────────────────────────
embedder      = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

def embed_fn(texts):
    return embedder.encode(texts).tolist()

# ══════════════════════════════════════════════════════════════
# ÉTAPE 1 — Téléchargement MITRE ATT&CK STIX Enterprise
# ══════════════════════════════════════════════════════════════

print("⏳ Téléchargement MITRE ATT&CK Enterprise (STIX)...")
MITRE_URL = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
response  = requests.get(MITRE_URL, timeout=60)
objects   = response.json()["objects"]
print(f"✅ {len(objects)} objets STIX téléchargés")

# ══════════════════════════════════════════════════════════════
# ÉTAPE 2 — Construction des maps STIX (id interne → données)
# Nécessaire pour reconstruire les relations groupe → techniques
# ══════════════════════════════════════════════════════════════

# Map : STIX internal id → objet complet
stix_id_to_obj = {obj["id"]: obj for obj in objects}

# Map : STIX internal id → external ATT&CK id (ex: "attack-pattern--xxx" → "T1566.001")
stix_id_to_attck_id = {}
for obj in objects:
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            stix_id_to_attck_id[obj["id"]] = ref.get("external_id", "")
            break

# Map : STIX internal id → nom de l'objet
stix_id_to_name = {obj["id"]: obj.get("name", "") for obj in objects}

# Relations : source_ref → [(target_ref, relationship_type)]
relations = defaultdict(list)
for obj in objects:
    if obj.get("type") == "relationship":
        src = obj.get("source_ref", "")
        tgt = obj.get("target_ref", "")
        rel = obj.get("relationship_type", "")
        if src and tgt and rel:
            relations[src].append((tgt, rel))

print(f"✅ {len(relations)} relations STIX indexées")

# ══════════════════════════════════════════════════════════════
# ÉTAPE 3 — Collection 1 : mitre_techniques
# ══════════════════════════════════════════════════════════════

techniques      = []
tech_id_to_data = {}  # ATT&CK id → données, pour enrichissement des profils APT

for obj in objects:
    if obj.get("type") != "attack-pattern":
        continue
    if obj.get("revoked", False) or obj.get("x_mitre_deprecated", False):
        continue

    tech_id = stix_id_to_attck_id.get(obj["id"], "")
    if not tech_id:
        continue

    platforms   = ", ".join(obj.get("x_mitre_platforms", []))
    tactics     = ", ".join([
        p["phase_name"].replace("-", " ").title()
        for p in obj.get("kill_chain_phases", [])
        if p.get("kill_chain_name") == "mitre-attack"
    ])
    name        = obj.get("name", "")
    description = obj.get("description", "")[:800]
    detection   = obj.get("x_mitre_detection", "")[:400]

    doc_text = f"""Technique: {tech_id} — {name}
Tactics: {tactics}
Platforms: {platforms}
Description: {description}
Detection: {detection}""".strip()

    data = {
        "id":          tech_id,
        "stix_id":     obj["id"],
        "name":        name,
        "tactics":     tactics,
        "platforms":   platforms,
        "description": description,
        "detection":   detection,
        "doc_text":    doc_text,
    }
    techniques.append(data)
    tech_id_to_data[tech_id] = data

print(f"\n✅ {len(techniques)} techniques MITRE extraites")

col_techniques = chroma_client.get_or_create_collection(
    name="mitre_techniques",
    metadata={"hnsw:space": "cosine"},
)
if col_techniques.count() == 0:
    print("⏳ Indexation des techniques (par batch de 200)...")
    batch_size = 200
    for i in range(0, len(techniques), batch_size):
        batch = techniques[i:i + batch_size]
        col_techniques.add(
            documents  = [t["doc_text"] for t in batch],
            embeddings = embed_fn([t["doc_text"] for t in batch]),
            metadatas  = [{
                "tech_id":     t["id"],
                "name":        t["name"],
                "tactics":     t["tactics"],
                "platforms":   t["platforms"],
                "description": t["description"],
                "detection":   t["detection"],
            } for t in batch],
            ids = [t["id"] for t in batch],
        )
    print(f"✅ {col_techniques.count()} techniques indexées dans mitre_techniques")
else:
    print(f"✅ mitre_techniques déjà peuplée ({col_techniques.count()} entrées)")

# ══════════════════════════════════════════════════════════════
# ÉTAPE 4 — Collection 2 : mitre_apt_groups
# ══════════════════════════════════════════════════════════════

apt_groups      = []
group_stix_ids  = []  # pour l'étape 5

for obj in objects:
    if obj.get("type") != "intrusion-set":
        continue
    if obj.get("revoked", False):
        continue

    group_id = stix_id_to_attck_id.get(obj["id"], "")
    if not group_id:
        continue

    name        = obj.get("name", "")
    aliases     = ", ".join(obj.get("aliases", [name]))
    description = obj.get("description", "")[:600]

    doc_text = f"""APT Group: {group_id} — {name}
Aliases: {aliases}
Description: {description}""".strip()

    apt_groups.append({
        "id":          group_id,
        "stix_id":     obj["id"],
        "name":        name,
        "aliases":     aliases,
        "description": description,
        "doc_text":    doc_text,
    })
    group_stix_ids.append(obj["id"])

print(f"\n✅ {len(apt_groups)} groupes APT extraits")

col_apt = chroma_client.get_or_create_collection(
    name="mitre_apt_groups",
    metadata={"hnsw:space": "cosine"},
)
if col_apt.count() == 0:
    print("⏳ Indexation des groupes APT...")
    col_apt.add(
        documents  = [g["doc_text"] for g in apt_groups],
        embeddings = embed_fn([g["doc_text"] for g in apt_groups]),
        metadatas  = [{
            "group_id": g["id"],
            "name":     g["name"],
            "aliases":  g["aliases"],
        } for g in apt_groups],
        ids = [g["id"] for g in apt_groups],
    )
    print(f"✅ {col_apt.count()} groupes APT indexés dans mitre_apt_groups")
else:
    print(f"✅ mitre_apt_groups déjà peuplée ({col_apt.count()} entrées)")

# ══════════════════════════════════════════════════════════════
# ÉTAPE 5 — Collection 3 : threat_intel
# Profils APT enrichis = groupe + toutes ses techniques utilisées
# Reconstruction automatique via les relations STIX "uses"
# ══════════════════════════════════════════════════════════════
#
# Logique :
#   intrusion-set (APT28) --uses--> attack-pattern (T1566.001)
#   intrusion-set (APT28) --uses--> attack-pattern (T1071.001)
#   → on construit un doc : "APT28 uses T1566.001 Spearphishing, T1071.001 Web Protocols..."
#
# Ce document est sémantiquement riche : un email avec des
# caractéristiques APT28 sera rapproché de ce profil par le RAG.

print(f"\n⏳ Construction des profils APT enrichis depuis les relations STIX...")

threat_intel_docs = []

# Map stix_id → group data pour lookup rapide
stix_to_group = {g["stix_id"]: g for g in apt_groups}

for group in apt_groups:
    stix_id = group["stix_id"]

    # Récupérer toutes les techniques utilisées par ce groupe
    used_techniques = []
    for (target_ref, rel_type) in relations.get(stix_id, []):
        if rel_type != "uses":
            continue
        target_obj = stix_id_to_obj.get(target_ref, {})
        if target_obj.get("type") != "attack-pattern":
            continue
        if target_obj.get("revoked", False):
            continue

        attck_id = stix_id_to_attck_id.get(target_ref, "")
        tech_name = stix_id_to_name.get(target_ref, "")
        if not attck_id:
            continue

        # Récupérer les tactiques depuis tech_id_to_data
        tech_data = tech_id_to_data.get(attck_id, {})
        tactics   = tech_data.get("tactics", "")

        used_techniques.append({
            "id":      attck_id,
            "name":    tech_name,
            "tactics": tactics,
        })

    # Ne garder que les groupes avec au moins 2 techniques (profil utile)
    if len(used_techniques) < 2:
        continue

    # Trier par tactique pour que le document soit lisible
    used_techniques.sort(key=lambda x: x["tactics"])

    # Identifier la tactique dominante (pour le champ "severity")
    tactic_counts = defaultdict(int)
    for t in used_techniques:
        for tac in t["tactics"].split(", "):
            tactic_counts[tac.strip()] += 1
    dominant_tactic = max(tactic_counts, key=tactic_counts.get) if tactic_counts else "unknown"

    # Détecter si ce groupe fait du phishing (pour pertinence SOC email)
    phishing_techs = [
        t for t in used_techniques
        if any(kw in t["name"].lower() for kw in
               ["phish", "spearphish", "credential", "link", "attachment",
                "email", "lure", "masquerad"])
    ]
    is_email_threat = len(phishing_techs) > 0

    # Construire les lignes de techniques
    tech_lines = "\n".join([
        f"  - {t['id']} {t['name']} [{t['tactics']}]"
        for t in used_techniques[:25]  # max 25 techniques pour ne pas saturer l'embedding
    ])

    # Construire le document enrichi
    doc_text = f"""APT Profile: {group['id']} — {group['name']}
Aliases: {group['aliases']}
Email threat actor: {'yes' if is_email_threat else 'no'}
Dominant tactic: {dominant_tactic}
Total techniques: {len(used_techniques)}
Phishing techniques: {', '.join([t['id'] + ' ' + t['name'] for t in phishing_techs[:5]]) or 'none'}
Description: {group['description'][:400]}
Techniques used:
{tech_lines}""".strip()

    # Sévérité basée sur le nombre de techniques (proxy de sophistication)
    if len(used_techniques) >= 30:
        severity = "critical"
    elif len(used_techniques) >= 15:
        severity = "high"
    elif len(used_techniques) >= 5:
        severity = "medium"
    else:
        severity = "low"

    threat_intel_docs.append({
        "id":               f"profile_{group['id']}",
        "campaign":         f"{group['name']} TTPs Profile",
        "actor":            group["name"],
        "group_id":         group["id"],
        "technique_count":  len(used_techniques),
        "severity":         severity,
        "dominant_tactic":  dominant_tactic,
        "is_email_threat":  str(is_email_threat),
        "doc_text":         doc_text,
    })

print(f"✅ {len(threat_intel_docs)} profils APT enrichis construits")
print(f"   dont {sum(1 for d in threat_intel_docs if d['is_email_threat'] == 'True')} groupes avec techniques email/phishing")

# Aperçu des 3 premiers profils
for doc in threat_intel_docs[:3]:
    print(f"\n  [{doc['group_id']}] {doc['actor']}")
    print(f"  Techniques : {doc['technique_count']} | Sévérité : {doc['severity']} | Email threat : {doc['is_email_threat']}")
    print(f"  Extrait    : {doc['doc_text'][:200]}...")

col_ti = chroma_client.get_or_create_collection(
    name="threat_intel",
    metadata={"hnsw:space": "cosine"},
)
if col_ti.count() == 0:
    print("\n⏳ Indexation des profils APT dans threat_intel (par batch de 50)...")
    batch_size = 50
    for i in range(0, len(threat_intel_docs), batch_size):
        batch = threat_intel_docs[i:i + batch_size]
        col_ti.add(
            documents  = [d["doc_text"] for d in batch],
            embeddings = embed_fn([d["doc_text"] for d in batch]),
            metadatas  = [{
                "campaign":        d["campaign"],
                "actor":           d["actor"],
                "group_id":        d["group_id"],
                "technique":       f"{d['technique_count']} techniques",
                "severity":        d["severity"],
                "dominant_tactic": d["dominant_tactic"],
                "is_email_threat": d["is_email_threat"],
            } for d in batch],
            ids = [d["id"] for d in batch],
        )
    print(f"✅ {col_ti.count()} profils indexés dans threat_intel")
else:
    print(f"\n✅ threat_intel déjà peuplée ({col_ti.count()} entrées)")

# ── Résumé final ──────────────────────────────────────────────
print("\n" + "="*55)
print("📦 RÉSUMÉ CHROMADB — 100% automatique depuis STIX")
print("="*55)
print(f"  mitre_techniques  : {col_techniques.count()} techniques ATT&CK")
print(f"  mitre_apt_groups  : {col_apt.count()} groupes APT")
print(f"  threat_intel      : {col_ti.count()} profils APT enrichis")
print(f"  Vectorstore path  : {VECTORSTORE_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE C — TEST RAG (3 collections : techniques, APT, campagnes)
# Prérequis : CELLULE B exécutée (col_techniques, col_apt, col_ti, embedder)
# ══════════════════════════════════════════════════════════════

# ── Fonctions de recherche RAG ────────────────────────────────

def search_mitre(text: str, n: int = 3) -> list:
    """Recherche les techniques MITRE les plus proches sémantiquement."""
    query_emb = embedder.encode([text[:500]]).tolist()
    res = col_techniques.query(query_embeddings=query_emb, n_results=n)
    if not res["ids"][0]:
        return []
    return [
        {
            "tech_id":    res["metadatas"][0][i]["tech_id"],
            "name":       res["metadatas"][0][i]["name"],
            "tactics":    res["metadatas"][0][i]["tactics"],
            "similarity": round(1 - res["distances"][0][i], 3),
        }
        for i in range(len(res["ids"][0]))
    ]


def search_apt(text: str, n: int = 2) -> list:
    """Recherche les groupes APT les plus proches sémantiquement."""
    query_emb = embedder.encode([text[:500]]).tolist()
    res = col_apt.query(query_embeddings=query_emb, n_results=n)
    if not res["ids"][0]:
        return []
    return [
        {
            "group_id":   res["metadatas"][0][i]["group_id"],
            "name":       res["metadatas"][0][i]["name"],
            "aliases":    res["metadatas"][0][i]["aliases"][:50],
            "similarity": round(1 - res["distances"][0][i], 3),
        }
        for i in range(len(res["ids"][0]))
    ]


def search_campaigns(text: str, n: int = 2) -> list:
    """Recherche les campagnes threat intel les plus proches sémantiquement."""
    query_emb = embedder.encode([text[:500]]).tolist()
    res = col_ti.query(query_embeddings=query_emb, n_results=n)
    if not res["ids"][0]:
        return []
    return [
        {
            "campaign":   res["metadatas"][0][i]["campaign"],
            "technique":  res["metadatas"][0][i]["technique"],
            "severity":   res["metadatas"][0][i]["severity"],
            "actor":      res["metadatas"][0][i]["actor"],
            "similarity": round(1 - res["distances"][0][i], 3),
        }
        for i in range(len(res["ids"][0]))
    ]


# ── Tests sur emails types ────────────────────────────────────

test_emails = [
    (
        "Phishing credential O365",
        "URGENT: Your Office 365 account will be suspended. Verify your password immediately at http://secure-login.tk/verify"
    ),
    (
        "BEC / Wire fraud CEO",
        "Dear CEO, please process this urgent wire transfer of $50,000 to our new bank account. Confidential, do not discuss."
    ),
    (
        "Malware attachment macros",
        "Please find attached the invoice document. Enable macros to view the content properly."
    ),
    (
        "PayPal phishing",
        "Your PayPal account has been limited. Click here to verify your identity at http://paypal-secure-alert.net/verify"
    ),
    (
        "Lazarus banking",
        "Please review the attached SWIFT transaction report. Open the document for full details of the international wire transfer."
    ),
]

print("🔍 Test RAG — 3 collections\n")

for label, text in test_emails:
    print(f"{'='*60}")
    print(f"📧 {label}")
    print(f"   {text[:90]}...")

    print("\n  🎯 Techniques MITRE :")
    results = search_mitre(text)
    if results:
        for m in results:
            print(f"    [{m['tech_id']}] {m['name']}")
            print(f"             tactics={m['tactics']} | sim={m['similarity']}")
    else:
        print("    (aucun résultat)")

    print("\n  👥 Groupes APT :")
    results = search_apt(text)
    if results:
        for g in results:
            print(f"    [{g['group_id']}] {g['name']} ({g['aliases']}) | sim={g['similarity']}")
    else:
        print("    (aucun résultat)")

    print("\n  🚨 Campagnes connues :")
    results = search_campaigns(text)
    if results:
        for c in results:
            print(f"    {c['campaign']} | {c['technique']}")
            print(f"    Actor: {c['actor']} | Severity: {c['severity']} | sim={c['similarity']}")
    else:
        print("    (aucun résultat)")

    print()

print("✅ Test RAG terminé — les 3 collections répondent correctement")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE D — IoC FEEDS (URLHaus + OpenPhish)
#   - Téléchargement et cache local des feeds
#   - Indexation OpenPhish dans ChromaDB (recherche sémantique)
#   - Fonctions de lookup : exact (URLHaus) + sémantique (OpenPhish)
#
# Prérequis : CELLULE B exécutée (chroma_client, embedder, embed_fn,
#             VECTORSTORE_DIR, IOC_CACHE_DIR définis)
# ══════════════════════════════════════════════════════════════

import requests
import json
import os
import re
import chromadb
from datetime import datetime
from urllib.parse import urlparse
from collections import defaultdict

# ══════════════════════════════════════════════════════════════
# PARTIE 1 — URLHaus : téléchargement + cache local (lookup exact)
# Format texte : une URL par ligne, lignes # = commentaires
# Pas d'indexation ChromaDB → volume trop élevé (~500k URLs)
# On garde un set Python en mémoire pour un lookup O(1)
# ══════════════════════════════════════════════════════════════

print("⏳ Téléchargement URLHaus feed...")
URLHAUS_URL = "https://urlhaus.abuse.ch/downloads/text/"

try:
    r = requests.get(URLHAUS_URL, timeout=30)
    r.raise_for_status()

    urlhaus_set = set()
    for line in r.text.splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            urlhaus_set.add(line.lower())
            urlhaus_set.add(line)          # garder aussi la casse originale

    # Cache sur Drive
    with open(f"{IOC_CACHE_DIR}/urlhaus.json", "w") as f:
        json.dump(list(urlhaus_set), f)

    print(f"✅ URLHaus : {len(urlhaus_set):,} URLs malveillantes chargées")

except Exception as e:
    print(f"⚠️  URLHaus erreur : {e}")
    # Tenter de charger depuis le cache Drive
    cache_path = f"{IOC_CACHE_DIR}/urlhaus.json"
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            urlhaus_set = set(json.load(f))
        print(f"📦 URLHaus chargé depuis cache : {len(urlhaus_set):,} entrées")
    else:
        urlhaus_set = set()
        print("❌ Aucun cache disponible — URLHaus désactivé")

# ══════════════════════════════════════════════════════════════
# PARTIE 2 — OpenPhish : téléchargement + cache + ChromaDB
# Volume gérable (~5 000 URLs actives) → indexation sémantique
# Permet de trouver des URLs similaires même sans match exact
# ══════════════════════════════════════════════════════════════

print("\n⏳ Téléchargement OpenPhish feed...")
OPENPHISH_URL = "https://openphish.com/feed.txt"

try:
    r = requests.get(OPENPHISH_URL, timeout=30)
    r.raise_for_status()

    openphish_raw = [
        line.strip() for line in r.text.splitlines()
        if line.strip() and line.startswith("http")
    ]
    openphish_set = set(u.lower() for u in openphish_raw)
    openphish_set.update(openphish_raw)   # casse originale aussi

    with open(f"{IOC_CACHE_DIR}/openphish.json", "w") as f:
        json.dump(openphish_raw, f)

    print(f"✅ OpenPhish : {len(openphish_raw):,} URLs phishing actives chargées")

except Exception as e:
    print(f"⚠️  OpenPhish erreur : {e}")
    cache_path = f"{IOC_CACHE_DIR}/openphish.json"
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            openphish_raw = json.load(f)
        openphish_set = set(u.lower() for u in openphish_raw)
        openphish_set.update(openphish_raw)
        print(f"📦 OpenPhish chargé depuis cache : {len(openphish_raw):,} entrées")
    else:
        openphish_raw = []
        openphish_set = set()
        print("❌ Aucun cache disponible — OpenPhish désactivé")

# ══════════════════════════════════════════════════════════════
# PARTIE 3 — Construction des documents RAG depuis OpenPhish
# Chaque URL est transformée en document textuel enrichi
# pour permettre une recherche sémantique par similarité
# ══════════════════════════════════════════════════════════════

def infer_attack_type(url: str, domain: str, path: str) -> str:
    """Infère le type d'attaque depuis l'URL sans API externe."""
    path_l   = path.lower()
    domain_l = domain.lower()

    if any(kw in path_l for kw in ["invoice", "facture", "receipt", "bill", "payment"]):
        return "invoice fraud"
    if any(kw in path_l for kw in ["login", "signin", "sign-in", "logon", "auth"]):
        return "credential harvesting login page"
    if any(kw in path_l for kw in ["verify", "verification", "confirm", "validate", "account"]):
        return "account verification phishing"
    if any(kw in path_l for kw in ["download", "install", "setup", "update", "upgrade"]):
        return "malware delivery"
    if any(kw in path_l for kw in ["tracking", "track", "shipment", "delivery", "dhl", "fedex", "ups"]):
        return "shipping notification phishing"
    if any(brand in domain_l for brand in [
        "paypal", "apple", "microsoft", "amazon", "google", "netflix",
        "facebook", "instagram", "linkedin", "wellsfargo", "chase",
        "bankofamerica", "office365", "outlook", "dropbox", "docusign"
    ]):
        return "brand impersonation phishing"
    if any(kw in path_l for kw in ["prize", "winner", "reward", "gift", "claim", "free"]):
        return "prize scam phishing"
    if any(kw in path_l for kw in ["crypto", "bitcoin", "wallet", "nft", "binance", "coinbase"]):
        return "crypto phishing"
    return "generic phishing"


def infer_brand_target(url: str, domain: str) -> str:
    """Détecte la marque ciblée par typosquatting ou chemin."""
    domain_l = domain.lower()
    url_l    = url.lower()
    brands = [
        "paypal", "apple", "microsoft", "amazon", "google", "netflix",
        "facebook", "instagram", "twitter", "linkedin", "dhl", "fedex",
        "ups", "office365", "outlook", "dropbox", "docusign", "wellsfargo",
        "chase", "bankofamerica", "citibank", "hsbc", "barclays"
    ]
    for brand in brands:
        if brand in domain_l or brand in url_l:
            return brand
    return "unknown"


def url_to_rag_doc(url: str) -> dict:
    """Transforme une URL OpenPhish en document RAG enrichi."""
    try:
        parsed  = urlparse(url)
        domain  = parsed.netloc.lower().replace("www.", "")
        path    = parsed.path.lower()
        tld     = "." + domain.split(".")[-1] if "." in domain else ""
        scheme  = parsed.scheme
    except Exception:
        domain, path, tld, scheme = "", "", "", "http"

    attack_type  = infer_attack_type(url, domain, path)
    brand_target = infer_brand_target(url, domain)

    # Caractéristiques structurelles suspectes
    suspicious_flags = []
    if tld in {".tk", ".xyz", ".click", ".top", ".link", ".info",
               ".biz", ".ru", ".cn", ".pw", ".ga", ".ml", ".cf", ".gq"}:
        suspicious_flags.append(f"suspicious TLD {tld}")
    if len(domain) > 35:
        suspicious_flags.append("long domain")
    if domain.count("-") >= 3:
        suspicious_flags.append("many hyphens in domain")
    if re.match(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", domain):
        suspicious_flags.append("direct IP address")
    if path.count("/") > 5:
        suspicious_flags.append("deep path")

    doc_text = (
        f"OpenPhish live phishing URL. "
        f"Attack type: {attack_type}. "
        f"Brand targeted: {brand_target}. "
        f"Domain: {domain}. "
        f"TLD: {tld}. "
        f"Path pattern: {path[:120]}. "
        f"Suspicious indicators: {', '.join(suspicious_flags) if suspicious_flags else 'none'}. "
        f"Feed date: {datetime.now().strftime('%Y-%m-%d')}. "
        f"Full URL: {url[:200]}."
    )

    uid = f"op_{abs(hash(url)) % 10**9}"

    return {
        "id":            uid,
        "doc_text":      doc_text,
        "url":           url[:500],
        "domain":        domain[:200],
        "tld":           tld,
        "attack_type":   attack_type,
        "brand_target":  brand_target,
        "flags":         ", ".join(suspicious_flags) if suspicious_flags else "none",
        "source":        "OpenPhish",
        "ingested_at":   datetime.now().isoformat(),
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 4 — Indexation OpenPhish dans ChromaDB (collection live_ioc_feeds)
# ══════════════════════════════════════════════════════════════

col_live = chroma_client.get_or_create_collection(
    name="live_ioc_feeds",
    metadata={"hnsw:space": "cosine"},
)

if col_live.count() == 0 and openphish_raw:
    print(f"\n⏳ Indexation OpenPhish dans ChromaDB (max 3 000 URLs)...")

    # Limiter à 3 000 pour ne pas saturer ChromaDB / temps d'embedding
    urls_to_index = openphish_raw[:3000]
    docs = [url_to_rag_doc(u) for u in urls_to_index]

    # Dédupliquer sur l'id (hash collision rare mais possible)
    seen_ids = set()
    docs_dedup = []
    for d in docs:
        if d["id"] not in seen_ids:
            seen_ids.add(d["id"])
            docs_dedup.append(d)

    # Indexation par batch de 200
    batch_size = 200
    for i in range(0, len(docs_dedup), batch_size):
        batch = docs_dedup[i:i + batch_size]
        col_live.add(
            documents  = [d["doc_text"]  for d in batch],
            embeddings = embed_fn([d["doc_text"] for d in batch]),
            metadatas  = [{
                "url":          d["url"],
                "domain":       d["domain"],
                "tld":          d["tld"],
                "attack_type":  d["attack_type"],
                "brand_target": d["brand_target"],
                "flags":        d["flags"],
                "source":       d["source"],
                "ingested_at":  d["ingested_at"],
            } for d in batch],
            ids = [d["id"] for d in batch],
        )
        print(f"  batch {i//batch_size + 1}/{(len(docs_dedup)-1)//batch_size + 1} — {min(i+batch_size, len(docs_dedup))} URLs indexées")

    print(f"✅ {col_live.count()} URLs OpenPhish indexées dans live_ioc_feeds")

    # Stats sur les types d'attaque indexés
    attack_types = defaultdict(int)
    for d in docs_dedup:
        attack_types[d["attack_type"]] += 1
    print("\n📊 Répartition par type d'attaque :")
    for at, count in sorted(attack_types.items(), key=lambda x: -x[1]):
        print(f"  {at:40} : {count}")

elif col_live.count() > 0:
    print(f"\n✅ live_ioc_feeds déjà peuplée ({col_live.count()} entrées)")
else:
    print("\n⚠️  OpenPhish vide — live_ioc_feeds non peuplée")

# ══════════════════════════════════════════════════════════════
# PARTIE 5 — Fonctions de lookup IoC (utilisées par FastAPI)
# ══════════════════════════════════════════════════════════════

def check_url_exact(url: str) -> dict:
    """
    Lookup exact contre URLHaus et OpenPhish.
    O(1) grâce aux sets en mémoire.
    """
    url_norm = url.strip().lower()
    result   = {
        "url":         url,
        "urlhaus":     False,
        "openphish":   False,
        "blacklisted": False,
        "score":       0.0,
        "sources":     [],
    }

    if url_norm in urlhaus_set or url in urlhaus_set:
        result["urlhaus"]     = True
        result["blacklisted"] = True
        result["score"]      += 0.90
        result["sources"].append("URLHaus")

    if url_norm in openphish_set or url in openphish_set:
        result["openphish"]   = True
        result["blacklisted"] = True
        result["score"]      += 0.90
        result["sources"].append("OpenPhish")

    result["score"] = min(result["score"], 1.0)
    return result


def search_similar_iocs(text: str, n: int = 3) -> list:
    """
    Recherche sémantique dans live_ioc_feeds.
    Utile quand l'URL exacte n'est pas dans les listes
    mais ressemble à des patterns connus (même famille d'attaque).
    """
    if col_live.count() == 0:
        return []
    try:
        query_emb = embed_fn([text[:500]])
        res = col_live.query(query_embeddings=query_emb, n_results=n)
        if not res["ids"][0]:
            return []
        return [
            {
                "url":          res["metadatas"][0][i]["url"],
                "domain":       res["metadatas"][0][i]["domain"],
                "attack_type":  res["metadatas"][0][i]["attack_type"],
                "brand_target": res["metadatas"][0][i]["brand_target"],
                "flags":        res["metadatas"][0][i]["flags"],
                "source":       res["metadatas"][0][i]["source"],
                "similarity":   round(1 - res["distances"][0][i], 3),
            }
            for i in range(len(res["ids"][0]))
        ]
    except Exception as e:
        print(f"⚠️  search_similar_iocs erreur : {e}")
        return []


def check_urls_batch(urls: list) -> dict:
    """
    Vérifie une liste d'URLs :
      - lookup exact URLHaus + OpenPhish (toutes les URLs)
      - recherche sémantique ChromaDB (sur le texte combiné des URLs)
    Retourne un dict agrégé avec score global et détails.
    """
    exact_results = [check_url_exact(u) for u in urls]
    blacklisted   = [r for r in exact_results if r["blacklisted"]]

    # Score exact : max des scores individuels
    exact_score = max((r["score"] for r in exact_results), default=0.0)

    # Recherche sémantique sur la concaténation des URLs
    combined_text = " ".join(urls[:10])
    semantic_hits = search_similar_iocs(combined_text, n=3)

    # Score sémantique pondéré (moins fort qu'un match exact)
    semantic_score = 0.0
    if semantic_hits:
        best_sim = semantic_hits[0]["similarity"]
        if best_sim > 0.80:
            semantic_score = 0.70
        elif best_sim > 0.65:
            semantic_score = 0.45
        elif best_sim > 0.50:
            semantic_score = 0.25

    # Score combiné : exact prime sur sémantique
    final_score = max(exact_score, semantic_score * 0.6)

    return {
        "total_checked":    len(urls),
        "blacklisted_count":len(blacklisted),
        "score":            round(min(final_score, 1.0), 4),
        "exact_matches":    [r["url"] for r in blacklisted],
        "exact_sources":    list({s for r in blacklisted for s in r["sources"]}),
        "semantic_similar": semantic_hits,
        "details":          exact_results,
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 6 — Sauvegarde métadonnées + tests
# ══════════════════════════════════════════════════════════════

metadata = {
    "updated_at":       datetime.now().isoformat(),
    "urlhaus_count":    len(urlhaus_set) // 2,  # divisé par 2 car double (orig + lower)
    "openphish_count":  len(openphish_raw),
    "chromadb_indexed": col_live.count(),
}
with open(f"{IOC_CACHE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ── Tests lookup exact ────────────────────────────────────────
print("\n" + "="*60)
print("🔍 Test 1 — Lookup exact (URLHaus + OpenPhish)")
print("="*60)

test_urls_exact = [
    "https://google.com",
    "http://secure-account-verify.tk/login",
    "http://paypal-secure-alert.net/verify",
    "http://microsoft-login.xyz/auth/office365",
]

for url in test_urls_exact:
    r      = check_url_exact(url)
    status = "🔴 BLACKLISTÉ" if r["blacklisted"] else "🟢 Clean"
    src    = f" | sources: {r['sources']}" if r["sources"] else ""
    print(f"  {status} | score={r['score']:.2f}{src}")
    print(f"           {url[:70]}")

# ── Tests recherche sémantique ────────────────────────────────
print("\n" + "="*60)
print("🔍 Test 2 — Recherche sémantique (OpenPhish ChromaDB)")
print("="*60)

test_queries_semantic = [
    ("Credential harvest O365",
     "URGENT Your Office 365 account has been suspended verify your password http://secure-login-microsoft.tk/verify"),
    ("PayPal phishing",
     "Your PayPal account is limited click here to restore access http://paypal-alert-secure.xyz/account/verify"),
    ("DHL delivery phishing",
     "Your DHL package could not be delivered please update your delivery address http://dhl-tracking.click/update"),
]

for label, text in test_queries_semantic:
    print(f"\n  📧 {label}")
    hits = search_similar_iocs(text, n=3)
    if hits:
        for h in hits:
            print(f"    sim={h['similarity']:.3f} | {h['attack_type']} | brand={h['brand_target']}")
            print(f"             {h['url'][:70]}")
    else:
        print("    (aucun résultat — collection vide)")

# ── Test batch complet ────────────────────────────────────────
print("\n" + "="*60)
print("🔍 Test 3 — Batch check (exact + sémantique combinés)")
print("="*60)

batch_test = [
    "https://google.com",
    "http://paypal-secure-alert.net/verify",
    "http://office365-login.xyz/auth",
    "http://dhl-delivery-update.click/track",
]

result = check_urls_batch(batch_test)
print(f"  URLs vérifiées  : {result['total_checked']}")
print(f"  Blacklistées    : {result['blacklisted_count']}")
print(f"  Score global    : {result['score']:.4f}")
print(f"  Sources exactes : {result['exact_sources']}")
print(f"  Hits sémantiques: {len(result['semantic_similar'])}")
if result["semantic_similar"]:
    best = result["semantic_similar"][0]
    print(f"  Meilleur hit    : {best['attack_type']} | sim={best['similarity']:.3f} | {best['domain']}")

# ── Résumé final ──────────────────────────────────────────────
print("\n" + "="*60)
print("📦 RÉSUMÉ CELLULE D")
print("="*60)
print(f"  URLHaus    : {len(urlhaus_set)//2:,} URLs (lookup exact, set mémoire)")
print(f"  OpenPhish  : {len(openphish_raw):,} URLs (cache Drive)")
print(f"  ChromaDB   : {col_live.count()} URLs indexées (recherche sémantique)")
print(f"  Cache      : {IOC_CACHE_DIR}")
print(f"\n✅ Cellule D terminée — fonctions disponibles :")
print(f"   check_url_exact(url)         → lookup exact URLHaus + OpenPhish")
print(f"   search_similar_iocs(text, n) → RAG sémantique sur live_ioc_feeds")
print(f"   check_urls_batch(urls)       → combiné exact + sémantique")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Sauvegarde des artifacts pour FastAPI
# (à exécuter dans Colab avant de coder l'API en local)
# ══════════════════════════════════════════════════════════════
import pickle

# Sauvegarder TF-IDF + LogisticRegression (légers, parfaits pour API)
artifacts = {
    "tfidf":   tfidf,
    "lr_model": lr_model,
}
with open(f"{BASE_DIR}/models/tfidf_lr_pipeline.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("✅ TF-IDF + LR sauvegardé")
print(f"✅ DistilBERT déjà sauvegardé dans {BASE_DIR}/models/distilbert-5000/final")
print("\n📦 Télécharge depuis Drive vers ton PC :")
print(f"  - {BASE_DIR}/models/tfidf_lr_pipeline.pkl")
print(f"  - {BASE_DIR}/models/distilbert-5000/final/  (dossier complet)")
print(f"  - {PROCESSED_DIR}/shap_beeswarm.png")
print(f"  - {PROCESSED_DIR}/shap_bar.png")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Installation de toutes les dépendances
# ══════════════════════════════════════════════════════════════
!pip install fastapi uvicorn pyngrok nest-asyncio -q
!pip install python-whois vt-py python-magic-bin oletools pytesseract Pillow -q
!pip install python-Levenshtein slack-sdk -q
!apt-get install -y tesseract-ocr -q

import nest_asyncio
nest_asyncio.apply()
print("✅ Toutes les dépendances installées")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE E — THREAT INTELLIGENCE EXTERNE (AbuseIPDB + VirusTotal)
#   - AbuseIPDB : réputation des IPs extraites des en-têtes SMTP
#   - VirusTotal : analyse des URLs et hashes de pièces jointes
#   - Fonctions prêtes à brancher dans FastAPI (cellule F)
#
# Prérequis : CELLULE D exécutée
# Clés API nécessaires dans les Secrets Colab :
#   ABUSEIPDB_KEY  → https://www.abuseipdb.com/account/api
#   VT_API_KEY     → https://www.virustotal.com/gui/my-apikey
# ══════════════════════════════════════════════════════════════

import os
import re
import json
import time
import requests
from datetime import datetime, timezone

# ── Chargement des clés depuis Colab Secrets ─────────────────
# Si pas encore fait dans CELLULE 0, décommente :
# from google.colab import userdata
# os.environ["ABUSEIPDB_KEY"] = userdata.get('ABUSEIPDB_KEY')
# os.environ["VT_API_KEY"]    = userdata.get('VT_API_KEY')

from google.colab import userdata

try:
    ABUSEIPDB_KEY = userdata.get('ABUSEIPDB_KEY')
except Exception:
    ABUSEIPDB_KEY = ""

try:
    VT_API_KEY = userdata.get('VT_API_KEY')
except Exception:
    VT_API_KEY = ""



# ══════════════════════════════════════════════════════════════
# PARTIE 1 — AbuseIPDB
# Endpoint : GET https://api.abuseipdb.com/api/v2/check
# Gratuit  : 1 000 req/jour
# Retourne : abuseConfidenceScore (0-100), pays, ISP, nb signalements
# ══════════════════════════════════════════════════════════════

# Cache mémoire pour éviter de requêter deux fois la même IP
_abuseipdb_cache = {}

def check_ip_abuseipdb(ip: str, max_age_days: int = 30) -> dict:
    """
    Vérifie la réputation d'une IP via AbuseIPDB.
    Retourne un dict normalisé avec score 0-1.

    Score interprétation :
      0-25   → propre / peu signalée
      25-50  → suspecte
      50-75  → probablement malveillante
      75-100 → malveillante confirmée
    """
    # Valeur par défaut si API non disponible
    default = {
        "ip":             ip,
        "available":      False,
        "abuse_score":    0,       # 0-100 AbuseIPDB natif
        "score":          0.0,     # normalisé 0-1 pour le pipeline
        "country":        "unknown",
        "isp":            "unknown",
        "reports_count":  0,
        "is_tor":         False,
        "last_reported":  None,
        "verdict":        "unknown",
        "source":         "AbuseIPDB",
    }

    # Validation IP basique
    if not re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", ip):
        return default

    # IPs privées → skip
    private_ranges = ["10.", "192.168.", "172.16.", "172.17.", "172.18.",
                      "172.19.", "172.20.", "172.21.", "172.22.", "172.23.",
                      "172.24.", "172.25.", "172.26.", "172.27.", "172.28.",
                      "172.29.", "172.30.", "172.31.", "127.", "0."]
    if any(ip.startswith(r) for r in private_ranges):
        default["verdict"] = "private"
        return default

    # Cache hit
    if ip in _abuseipdb_cache:
        return _abuseipdb_cache[ip]

    if not ABUSEIPDB_KEY:
        # Mode mock — simuler une réponse pour les tests
        mock = default.copy()
        mock["available"] = False
        mock["verdict"]   = "api_key_missing"
        return mock

    try:
        resp = requests.get(
            "https://api.abuseipdb.com/api/v2/check",
            headers={
                "Key":    ABUSEIPDB_KEY,
                "Accept": "application/json",
            },
            params={
                "ipAddress":   ip,
                "maxAgeInDays": max_age_days,
                "verbose":     "",
            },
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json().get("data", {})

        abuse_score   = data.get("abuseConfidenceScore", 0)
        reports_count = data.get("totalReports", 0)
        country       = data.get("countryCode", "unknown")
        isp           = data.get("isp", "unknown")
        is_tor        = data.get("isTor", False)
        last_reported = data.get("lastReportedAt", None)

        # Normalisation score 0-1
        normalized = round(abuse_score / 100, 4)

        # Verdict
        if abuse_score >= 75:
            verdict = "malicious"
        elif abuse_score >= 50:
            verdict = "suspicious"
        elif abuse_score >= 25:
            verdict = "low_risk"
        else:
            verdict = "clean"

        result = {
            "ip":            ip,
            "available":     True,
            "abuse_score":   abuse_score,
            "score":         normalized,
            "country":       country,
            "isp":           isp,
            "reports_count": reports_count,
            "is_tor":        is_tor,
            "last_reported": last_reported,
            "verdict":       verdict,
            "source":        "AbuseIPDB",
        }

        _abuseipdb_cache[ip] = result
        return result

    except requests.exceptions.Timeout:
        default["verdict"] = "timeout"
        return default
    except Exception as e:
        default["verdict"] = f"error: {str(e)[:60]}"
        return default


def check_ips_batch(ips: list) -> dict:
    """
    Vérifie une liste d'IPs et retourne le score agrégé.
    Utilisé par FastAPI pour les IPs extraites des en-têtes Received.
    """
    if not ips:
        return {"score": 0.0, "results": [], "malicious_ips": []}

    results      = [check_ip_abuseipdb(ip) for ip in ips[:5]]  # max 5 IPs
    malicious    = [r for r in results if r["verdict"] in ("malicious", "suspicious")]
    max_score    = max((r["score"] for r in results), default=0.0)

    return {
        "score":        round(max_score, 4),
        "results":      results,
        "malicious_ips":[r["ip"] for r in malicious],
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 2 — VirusTotal
# Plan gratuit : 4 req/min, 500 req/jour
# Endpoints utilisés :
#   POST /urls          → soumettre une URL à l'analyse
#   GET  /urls/{id}     → récupérer le rapport d'analyse
#   GET  /files/{hash}  → rapport d'un hash (MD5/SHA256)
# ══════════════════════════════════════════════════════════════

VT_BASE = "https://www.virustotal.com/api/v3"
_vt_cache = {}   # cache mémoire URL/hash → résultat

def _vt_headers() -> dict:
    return {"x-apikey": VT_API_KEY, "Accept": "application/json"}


def _vt_default(identifier: str, kind: str = "url") -> dict:
    return {
        "identifier":    identifier,
        "kind":          kind,
        "available":     False,
        "malicious":     0,
        "suspicious":    0,
        "harmless":      0,
        "undetected":    0,
        "total_engines": 0,
        "score":         0.0,
        "verdict":       "unknown",
        "source":        "VirusTotal",
    }


def check_url_virustotal(url: str) -> dict:
    """
    Analyse une URL via VirusTotal.
    Stratégie : POST pour soumettre → attendre 15s → GET rapport.
    Plan gratuit : 4 req/min → on rate-limite entre les appels.
    """
    if url in _vt_cache:
        return _vt_cache[url]

    if not VT_API_KEY:
        result = _vt_default(url, "url")
        result["verdict"] = "api_key_missing"
        return result

    try:
        import base64

        # Encoder l'URL en base64 URL-safe sans padding (format VT)
        url_id = base64.urlsafe_b64encode(url.encode()).decode().rstrip("=")

        # Essayer d'abord GET direct (URL déjà connue de VT)
        resp = requests.get(
            f"{VT_BASE}/urls/{url_id}",
            headers=_vt_headers(),
            timeout=15,
        )

        if resp.status_code == 404:
            # URL inconnue → soumettre pour analyse
            sub = requests.post(
                f"{VT_BASE}/urls",
                headers={**_vt_headers(), "Content-Type": "application/x-www-form-urlencoded"},
                data=f"url={requests.utils.quote(url, safe='')}",
                timeout=15,
            )
            sub.raise_for_status()
            time.sleep(15)   # attendre l'analyse (gratuit → pas de polling)

            # Récupérer le rapport
            resp = requests.get(
                f"{VT_BASE}/urls/{url_id}",
                headers=_vt_headers(),
                timeout=15,
            )

        resp.raise_for_status()
        stats = resp.json()["data"]["attributes"]["last_analysis_stats"]

        malicious  = stats.get("malicious", 0)
        suspicious = stats.get("suspicious", 0)
        harmless   = stats.get("harmless", 0)
        undetected = stats.get("undetected", 0)
        total      = malicious + suspicious + harmless + undetected

        # Score normalisé 0-1
        score = round((malicious + suspicious * 0.5) / total, 4) if total > 0 else 0.0

        if malicious >= 5:
            verdict = "malicious"
        elif malicious >= 2 or suspicious >= 5:
            verdict = "suspicious"
        elif malicious >= 1:
            verdict = "low_risk"
        else:
            verdict = "clean"

        result = {
            "identifier":    url,
            "kind":          "url",
            "available":     True,
            "malicious":     malicious,
            "suspicious":    suspicious,
            "harmless":      harmless,
            "undetected":    undetected,
            "total_engines": total,
            "score":         score,
            "verdict":       verdict,
            "source":        "VirusTotal",
        }

        _vt_cache[url] = result
        time.sleep(15)   # rate limit plan gratuit (4 req/min)
        return result

    except requests.exceptions.Timeout:
        r = _vt_default(url, "url")
        r["verdict"] = "timeout"
        return r
    except Exception as e:
        r = _vt_default(url, "url")
        r["verdict"] = f"error: {str(e)[:60]}"
        return r


def check_hash_virustotal(file_hash: str) -> dict:
    """
    Vérifie un hash MD5/SHA256 de pièce jointe via VirusTotal.
    Retourne le résultat d'analyse si le fichier est connu de VT.
    """
    if file_hash in _vt_cache:
        return _vt_cache[file_hash]

    if not VT_API_KEY:
        result = _vt_default(file_hash, "hash")
        result["verdict"] = "api_key_missing"
        return result

    try:
        resp = requests.get(
            f"{VT_BASE}/files/{file_hash}",
            headers=_vt_headers(),
            timeout=15,
        )

        if resp.status_code == 404:
            result = _vt_default(file_hash, "hash")
            result["verdict"] = "not_found"
            return result

        resp.raise_for_status()
        stats = resp.json()["data"]["attributes"]["last_analysis_stats"]

        malicious  = stats.get("malicious", 0)
        suspicious = stats.get("suspicious", 0)
        harmless   = stats.get("harmless", 0)
        undetected = stats.get("undetected", 0)
        total      = malicious + suspicious + harmless + undetected
        score      = round((malicious + suspicious * 0.5) / total, 4) if total > 0 else 0.0

        if malicious >= 10:
            verdict = "malicious"
        elif malicious >= 3 or suspicious >= 8:
            verdict = "suspicious"
        elif malicious >= 1:
            verdict = "low_risk"
        else:
            verdict = "clean"

        result = {
            "identifier":    file_hash,
            "kind":          "hash",
            "available":     True,
            "malicious":     malicious,
            "suspicious":    suspicious,
            "harmless":      harmless,
            "undetected":    undetected,
            "total_engines": total,
            "score":         score,
            "verdict":       verdict,
            "source":        "VirusTotal",
        }

        _vt_cache[file_hash] = result
        time.sleep(15)
        return result

    except requests.exceptions.Timeout:
        r = _vt_default(file_hash, "hash")
        r["verdict"] = "timeout"
        return r
    except Exception as e:
        r = _vt_default(file_hash, "hash")
        r["verdict"] = f"error: {str(e)[:60]}"
        return r


def check_urls_virustotal_batch(urls: list, max_urls: int = 3) -> dict:
    """
    Analyse un lot d'URLs suspectes via VirusTotal.
    Limité à max_urls pour respecter le quota gratuit (500/jour).
    Prend en priorité les URLs les plus suspectes (non-HTTPS, TLD suspects).
    """
    if not urls:
        return {"score": 0.0, "results": [], "malicious_urls": []}

    # Prioriser les URLs suspectes
    suspicious_tlds = {".tk", ".xyz", ".click", ".top", ".link",
                       ".biz", ".pw", ".ga", ".ml", ".cf", ".gq"}

    def suspicion_key(url):
        score = 0
        if not url.startswith("https"):
            score += 1
        if any(url.lower().endswith(t) or f"{t}/" in url.lower() for t in suspicious_tlds):
            score += 2
        if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
            score += 3
        return -score   # négatif pour trier décroissant

    sorted_urls = sorted(urls, key=suspicion_key)[:max_urls]

    results   = [check_url_virustotal(u) for u in sorted_urls]
    malicious = [r for r in results if r["verdict"] in ("malicious", "suspicious")]
    max_score = max((r["score"] for r in results), default=0.0)

    return {
        "score":         round(max_score, 4),
        "results":       results,
        "malicious_urls":[r["identifier"] for r in malicious],
        "checked_count": len(results),
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 3 — Fonction agrégée threat_intel_check
# Point d'entrée unique pour FastAPI
# Reçoit URLs + IPs + hashes → retourne dict complet
# ══════════════════════════════════════════════════════════════

def threat_intel_check(
    urls:   list = None,
    ips:    list = None,
    hashes: list = None,
) -> dict:
    """
    Agrège AbuseIPDB + VirusTotal pour URLs, IPs et hashes.
    Retourne un dict avec scores individuels et score global threat_intel.

    Pondération interne :
      VirusTotal URLs   → 0.50
      AbuseIPDB IPs     → 0.30
      VirusTotal hashes → 0.20
    """
    urls    = urls    or []
    ips     = ips     or []
    hashes  = hashes  or []

    results = {
        "virustotal_urls":   {"score": 0.0, "results": [], "malicious_urls": []},
        "abuseipdb":         {"score": 0.0, "results": [], "malicious_ips": []},
        "virustotal_hashes": {"score": 0.0, "results": [], "malicious_urls": []},
    }

    if urls:
        results["virustotal_urls"] = check_urls_virustotal_batch(urls, max_urls=3)

    if ips:
        results["abuseipdb"] = check_ips_batch(ips)

    if hashes:
        hash_results = [check_hash_virustotal(h) for h in hashes[:2]]
        max_h_score  = max((r["score"] for r in hash_results), default=0.0)
        mal_hashes   = [r["identifier"] for r in hash_results
                        if r["verdict"] in ("malicious", "suspicious")]
        results["virustotal_hashes"] = {
            "score":         round(max_h_score, 4),
            "results":       hash_results,
            "malicious_urls": mal_hashes,
        }

    # Score global pondéré
    global_score = round(
        results["virustotal_urls"]["score"]   * 0.50 +
        results["abuseipdb"]["score"]          * 0.30 +
        results["virustotal_hashes"]["score"]  * 0.20,
        4
    )

    # Consolidation des IoCs détectés
    all_iocs = (
        results["virustotal_urls"]["malicious_urls"] +
        results["abuseipdb"]["malicious_ips"] +
        results["virustotal_hashes"]["malicious_urls"]
    )

    return {
        "score":           global_score,
        "iocs_detected":   all_iocs,
        "details":         results,
        "apis_used":       {
            "abuseipdb":   bool(ABUSEIPDB_KEY),
            "virustotal":  bool(VT_API_KEY),
        },
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 4 — Tests
# ══════════════════════════════════════════════════════════════

print("=" * 60)
print("🔍 CELLULE E — Tests AbuseIPDB + VirusTotal")
print("=" * 60)

# ── Test AbuseIPDB ────────────────────────────────────────────
print("\n📡 AbuseIPDB — test IPs")
test_ips = [
    "8.8.8.8",         # Google DNS — propre
    "1.1.1.1",         # Cloudflare — propre
    "185.220.101.45",  # Tor exit node connu — suspect
    "192.168.1.1",     # IP privée → skip
]

for ip in test_ips:
    r = check_ip_abuseipdb(ip)
    if r["available"]:
        print(f"  {ip:20} → score={r['abuse_score']:3d}/100 | {r['verdict']:12} | {r['country']} | {r['isp'][:30]}")
    else:
        print(f"  {ip:20} → {r['verdict']} (clé API : {'✅' if ABUSEIPDB_KEY else '❌ manquante'})")

# ── Test VirusTotal URLs ──────────────────────────────────────
print("\n🦠 VirusTotal — test URLs")
test_urls_vt = [
    "https://google.com",
    "http://malware-test.tk/payload",
]

for url in test_urls_vt:
    r = check_url_virustotal(url)
    if r["available"]:
        print(f"  {url[:50]:50} → {r['verdict']:12} | malicious={r['malicious']}/{r['total_engines']}")
    else:
        print(f"  {url[:50]:50} → {r['verdict']} (clé API : {'✅' if VT_API_KEY else '❌ manquante'})")

# ── Test agrégé complet ───────────────────────────────────────
print("\n🔬 threat_intel_check — test agrégé")
result = threat_intel_check(
    urls   = ["http://paypal-secure-alert.net/verify", "https://google.com"],
    ips    = ["8.8.8.8", "185.220.101.45"],
    hashes = [],
)

print(f"  Score global threat_intel : {result['score']:.4f}")
print(f"  IoCs détectés             : {result['iocs_detected'] or 'aucun'}")
print(f"  APIs disponibles          : AbuseIPDB={result['apis_used']['abuseipdb']} | VT={result['apis_used']['virustotal']}")

# ── Résumé ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📦 RÉSUMÉ CELLULE E")
print("=" * 60)
print(f"  AbuseIPDB  : {'✅ actif' if ABUSEIPDB_KEY else '⚠️  mode mock (clé manquante)'}")
print(f"  VirusTotal : {'✅ actif' if VT_API_KEY else '⚠️  mode mock (clé manquante)'}")
print(f"\n✅ Cellule E terminée — fonctions disponibles :")
print(f"   check_ip_abuseipdb(ip)              → réputation IP")
print(f"   check_ips_batch(ips)                → lot d'IPs")
print(f"   check_url_virustotal(url)           → analyse URL")
print(f"   check_hash_virustotal(hash)         → analyse hash")
print(f"   check_urls_virustotal_batch(urls)   → lot d'URLs")
print(f"   threat_intel_check(urls, ips, hashes) → agrégé complet ← FastAPI")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE G — THREAT INTEL EXTERNE : AbuseIPDB + VirusTotal
# Définit threat_intel_check() dans __main__, déjà appelée par api_v2.py
# ══════════════════════════════════════════════════════════════

!pip install requests -q

import os, time, requests
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

def _get_secret(name):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
        return val or ""
    except Exception:
        return os.getenv(name, "")

VT_API_KEY      = _get_secret("VT_API_KEY")
ABUSEIPDB_KEY   = _get_secret("ABUSEIPDB_KEY")

print(f"VirusTotal : {'✅ configuré' if VT_API_KEY else '⚠️ absent (Secrets Colab : VT_API_KEY)'}")
print(f"AbuseIPDB  : {'✅ configuré' if ABUSEIPDB_KEY else '⚠️ absent (Secrets Colab : ABUSEIPDB_KEY)'}")

# ── Cache mémoire pour éviter de re-interroger les mêmes IOCs ──
_ti_cache = {}
_ti_executor = ThreadPoolExecutor(max_workers=4)

VT_BASE_URL  = "https://www.virustotal.com/api/v3"
ABUSE_BASE_URL = "https://api.abuseipdb.com/api/v2/check"

# ══════════════════════════════════════════════════════════════
# VIRUSTOTAL — hash + URL lookup
# ══════════════════════════════════════════════════════════════
def _vt_check_hash(sha256: str, timeout: int = 5) -> dict:
    """Interroge VT pour un hash de fichier. Cache 24h en mémoire."""
    cache_key = f"vt_hash:{sha256}"
    if cache_key in _ti_cache:
        return _ti_cache[cache_key]

    if not VT_API_KEY:
        return {"checked": False, "malicious": False, "reason": "no_api_key"}

    try:
        headers = {"x-apikey": VT_API_KEY}
        url = f"{VT_BASE_URL}/files/{sha256}"
        r = requests.get(url, headers=headers, timeout=timeout)

        if r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt",
                      "malicious_count": 0, "total_engines": 0}
        elif r.status_code == 200:
            data  = r.json()["data"]["attributes"]
            stats = data.get("last_analysis_stats", {})
            malicious  = stats.get("malicious", 0)
            suspicious = stats.get("suspicious", 0)
            total      = sum(stats.values()) if stats else 0
            result = {
                "checked": True,
                "malicious": (malicious + suspicious) > 0,
                "malicious_count": malicious,
                "suspicious_count": suspicious,
                "total_engines": total,
                "names": data.get("names", [])[:3],
                "type_description": data.get("type_description", ""),
            }
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}

        _ti_cache[cache_key] = result
        return result

    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _vt_check_url(target_url: str, timeout: int = 5) -> dict:
    """Interroge VT pour une URL. Nécessite encodage base64 url-safe sans padding."""
    cache_key = f"vt_url:{target_url}"
    if cache_key in _ti_cache:
        return _ti_cache[cache_key]

    if not VT_API_KEY:
        return {"checked": False, "malicious": False, "reason": "no_api_key"}

    try:
        import base64
        url_id = base64.urlsafe_b64encode(target_url.encode()).decode().strip("=")
        headers = {"x-apikey": VT_API_KEY}
        r = requests.get(f"{VT_BASE_URL}/urls/{url_id}", headers=headers, timeout=timeout)

        if r.status_code == 200:
            stats = r.json()["data"]["attributes"].get("last_analysis_stats", {})
            malicious  = stats.get("malicious", 0)
            suspicious = stats.get("suspicious", 0)
            result = {
                "checked": True,
                "malicious": (malicious + suspicious) > 0,
                "malicious_count": malicious,
                "suspicious_count": suspicious,
                "total_engines": sum(stats.values()) if stats else 0,
            }
        elif r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt"}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}

        _ti_cache[cache_key] = result
        return result

    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

# ══════════════════════════════════════════════════════════════
# ABUSEIPDB — réputation IP
# ══════════════════════════════════════════════════════════════
def _abuseipdb_check(ip: str, timeout: int = 5) -> dict:
    """Score de confiance d'abus 0-100 pour une IP. Cache en mémoire."""
    cache_key = f"abuse:{ip}"
    if cache_key in _ti_cache:
        return _ti_cache[cache_key]

    if not ABUSEIPDB_KEY:
        return {"checked": False, "malicious": False, "reason": "no_api_key"}

    try:
        headers = {"Key": ABUSEIPDB_KEY, "Accept": "application/json"}
        params  = {"ipAddress": ip, "maxAgeInDays": 90}
        r = requests.get(ABUSE_BASE_URL, headers=headers, params=params, timeout=timeout)

        if r.status_code == 200:
            data  = r.json()["data"]
            score = data.get("abuseConfidenceScore", 0)
            result = {
                "checked":        True,
                "malicious":      score >= 50,
                "abuse_score":    score,
                "total_reports":  data.get("totalReports", 0),
                "country":        data.get("countryCode", ""),
                "isp":            data.get("isp", ""),
                "is_tor":         data.get("isTor", False),
            }
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}

        _ti_cache[cache_key] = result
        return result

    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

# ══════════════════════════════════════════════════════════════
# FONCTION PRINCIPALE — appelée par api_v2.py via __main__
# ══════════════════════════════════════════════════════════════
def threat_intel_check(urls: list = None, ips: list = None, hashes: list = None) -> dict:
    """
    Agrège VirusTotal (hashes + URLs) et AbuseIPDB (IPs).
    Timeout global 8s pour ne pas bloquer l'API si les services tiers sont lents.
    """
    urls   = urls or []
    ips    = ips or []
    hashes = hashes or []

    iocs_detected = []
    details       = {"urls": [], "ips": [], "hashes": []}
    apis_used     = {"virustotal": bool(VT_API_KEY), "abuseipdb": bool(ABUSEIPDB_KEY)}
    score         = 0.0

    def _run_with_timeout(fn, arg, timeout=6):
        try:
            future = _ti_executor.submit(fn, arg)
            return future.result(timeout=timeout)
        except FuturesTimeout:
            return {"checked": False, "malicious": False, "reason": "timeout"}
        except Exception as e:
            return {"checked": False, "malicious": False, "reason": str(e)[:80]}

    # Hashes (max 2 — limite VT free tier 4/min)
    for h in hashes[:2]:
        r = _run_with_timeout(_vt_check_hash, h)
        details["hashes"].append({"hash": h, **r})
        if r.get("malicious"):
            score += 0.95
            iocs_detected.append(f"Hash malveillant (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {h[:16]}...")

    # URLs (max 3)
    for u in urls[:3]:
        r = _run_with_timeout(_vt_check_url, u)
        details["urls"].append({"url": u, **r})
        if r.get("malicious"):
            score += 0.85
            iocs_detected.append(f"URL malveillante (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {u[:50]}")

    # IPs (max 3)
    for ip in ips[:3]:
        r = _run_with_timeout(_abuseipdb_check, ip)
        details["ips"].append({"ip": ip, **r})
        if r.get("malicious"):
            score += 0.70
            iocs_detected.append(f"IP malveillante (AbuseIPDB {r.get('abuse_score',0)}%) : {ip}")

    return {
        "score":          round(min(score, 1.0), 4),
        "iocs_detected":  iocs_detected,
        "details":        details,
        "apis_used":      apis_used,
    }

# ══════════════════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════════════════
print("\n🔍 Test threat_intel_check()...")
test_result = threat_intel_check(
    urls   = ["http://secure-login.tk/verify"],
    ips    = ["185.220.101.45"],   # IP de test connue Tor/malveillante
    hashes = [],
)
import json
print(json.dumps(test_result, indent=2, ensure_ascii=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE KILL — À exécuter AVANT de relancer la cellule F
# ══════════════════════════════════════════════════════════════
import os, signal, subprocess
from pyngrok import ngrok

# Tuer ngrok
try:
    ngrok.kill()
    print("✅ ngrok tué")
except Exception as e:
    print(f"ngrok : {e}")

# Libérer le port 8000
try:
    result = subprocess.run(
        ["fuser", "-k", "8000/tcp"],
        capture_output=True, text=True
    )
    print(f"✅ Port 8000 libéré : {result.stdout or 'OK'}")
except Exception as e:
    print(f"Port 8000 : {e}")

# Double vérification
import time
time.sleep(2)
result = subprocess.run(["fuser", "8000/tcp"], capture_output=True, text=True)
if result.stdout.strip():
    print(f"⚠️ Port encore occupé par PID : {result.stdout.strip()}")
    os.kill(int(result.stdout.strip()), signal.SIGKILL)
    print("✅ Process tué de force")
else:
    print("✅ Port 8000 libre")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE F — FASTAPI v3.0 CONSOLIDÉE (Groq + JSON fix + Sigma natif)
# Remplace : ancienne Cellule F + patch JSON + patch Groq
# Prérequis : CELLULE KILL exécutée juste avant, CELLULE G exécutée avant (threat_intel_check)
# ══════════════════════════════════════════════════════════════

!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart pyyaml -q
!pip install python-whois python-magic oletools pytesseract Pillow -q
!pip install python-Levenshtein slack-sdk google-generativeai groq -q
!apt-get install -y tesseract-ocr libmagic1 -q

import os, sys, time, asyncio, threading, subprocess
from google.colab import userdata
from pyngrok import ngrok

def _get_secret(name):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
        return val or ""
    except Exception:
        return os.getenv(name, "")

NGROK_TOKEN    = _get_secret("NGROK_TOKEN")
GEMINI_API_KEY = _get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = _get_secret("GROQ_API_KEY")
SLACK_TOKEN    = _get_secret("SLACK_TOKEN")
VT_API_KEY     = _get_secret("VT_API_KEY")
ABUSEIPDB_KEY  = _get_secret("ABUSEIPDB_KEY")

if not NGROK_TOKEN:
    raise ValueError("❌ NGROK_TOKEN manquant dans les Secrets Colab")

try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)
print("✅ Dépendances installées, port libéré")

# ══════════════════════════════════════════════════════════════
api_code = r'''
import re, os, sys, json, pickle, hashlib, time, uuid, yaml
import torch
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from email import policy
from email.parser import BytesParser
from io import BytesIO
from typing import Optional
from datetime import datetime

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import pytesseract
from PIL import Image
import Levenshtein
import whois

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
BASE_DIR        = "/content/drive/MyDrive/phishing_dasec"
VECTORSTORE_DIR = f"{BASE_DIR}/vectorstore"
IOC_CACHE_DIR   = f"{BASE_DIR}/ioc_cache"
SLACK_TOKEN     = os.getenv("SLACK_TOKEN", "")
SLACK_CHANNEL   = "#soc-alerts"
SOC_THRESHOLD   = 0.65

KNOWN_BRANDS = [
    "paypal","microsoft","google","apple","amazon","netflix",
    "facebook","instagram","twitter","linkedin","dhl","fedex",
    "bnpparibas","societegenerale","creditagricole","orange","sfr",
    "docusign","dropbox","wellsfargo","chase","bankofamerica",
]
SUSPICIOUS_TLDS = {
    ".tk",".xyz",".click",".top",".link",".info",
    ".biz",".ru",".cn",".pw",".ga",".ml",".cf",".gq",
}

# ══════════════════════════════════════════════════════════════
# CHARGEMENT MODÈLES
# ══════════════════════════════════════════════════════════════
print("⏳ Chargement des modèles...")

with open(f"{BASE_DIR}/models/tfidf_lr_pipeline.pkl", "rb") as f:
    artifacts = pickle.load(f)
tfidf    = artifacts["tfidf"]
lr_model = artifacts["lr_model"]

BERT_PATH  = f"{BASE_DIR}/models/distilbert-5000/final"
tokenizer  = DistilBertTokenizerFast.from_pretrained(BERT_PATH)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_PATH)
bert_model.eval()

embedder      = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

def embed_fn(texts):
    return embedder.encode(texts).tolist()

def _get_col(name):
    try:
        return chroma_client.get_collection(name)
    except Exception:
        return None

col_techniques = _get_col("mitre_techniques")
col_apt        = _get_col("mitre_apt_groups")
col_ti         = _get_col("threat_intel")
col_live       = _get_col("live_ioc_feeds")
RAG_OK         = all([col_techniques, col_apt, col_ti])

import json as _json
def _load_set(path):
    try:
        with open(path) as f:
            return set(_json.load(f))
    except Exception:
        return set()

urlhaus_set   = _load_set(f"{IOC_CACHE_DIR}/urlhaus.json")
openphish_set = _load_set(f"{IOC_CACHE_DIR}/openphish.json")

# ── LLM : Groq en priorité, Gemini en fallback (modèle 2.5 stable) ──
llm_client  = None
llm_backend = None
GROQ_KEY   = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY = os.getenv("GEMINI_API_KEY", "")

if GROQ_KEY:
    try:
        from groq import Groq
        llm_client  = Groq(api_key=GROQ_KEY)
        llm_backend = "groq"
        print("✅ LLM : Groq (llama-3.1-8b-instant)")
    except Exception as e:
        print(f"⚠️ Groq : {e}")

if not llm_client and GEMINI_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_KEY)
        llm_client  = genai.GenerativeModel("gemini-2.5-flash")
        llm_backend = "gemini"
        print("✅ LLM : Gemini (gemini-2.5-flash)")
    except Exception as e:
        print(f"⚠️ Gemini : {e}")

if not llm_client:
    print("⚠️ Aucun LLM — mode rapport structuré")

print(f"✅ Chargé | RAG={'OK' if RAG_OK else 'KO'} | LLM={llm_backend or 'none'} "
      f"| URLHaus={len(urlhaus_set):,} | OpenPhish={len(openphish_set):,}")

# ══════════════════════════════════════════════════════════════
# THREAT INTEL — AbuseIPDB + VirusTotal (intégré nativement, plus de __main__ import)
# ══════════════════════════════════════════════════════════════
VT_API_KEY    = os.getenv("VT_API_KEY", "")
ABUSEIPDB_KEY = os.getenv("ABUSEIPDB_KEY", "")
_ti_cache     = {}
_ti_executor  = ThreadPoolExecutor(max_workers=4)
VT_BASE_URL    = "https://www.virustotal.com/api/v3"
ABUSE_BASE_URL = "https://api.abuseipdb.com/api/v2/check"

def _vt_check_hash(sha256: str, timeout: int = 5) -> dict:
    cache_key = f"vt_hash:{sha256}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        headers = {"x-apikey": VT_API_KEY}
        r = _rq.get(f"{VT_BASE_URL}/files/{sha256}", headers=headers, timeout=timeout)
        if r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt",
                      "malicious_count": 0, "total_engines": 0}
        elif r.status_code == 200:
            data  = r.json()["data"]["attributes"]
            stats = data.get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0,
                      "names": data.get("names", [])[:3]}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _vt_check_url(target_url: str, timeout: int = 5) -> dict:
    cache_key = f"vt_url:{target_url}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq, base64
        url_id = base64.urlsafe_b64encode(target_url.encode()).decode().strip("=")
        headers = {"x-apikey": VT_API_KEY}
        r = _rq.get(f"{VT_BASE_URL}/urls/{url_id}", headers=headers, timeout=timeout)
        if r.status_code == 200:
            stats = r.json()["data"]["attributes"].get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0}
        elif r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt"}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _abuseipdb_check(ip: str, timeout: int = 5) -> dict:
    cache_key = f"abuse:{ip}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not ABUSEIPDB_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        headers = {"Key": ABUSEIPDB_KEY, "Accept": "application/json"}
        params  = {"ipAddress": ip, "maxAgeInDays": 90}
        r = _rq.get(ABUSE_BASE_URL, headers=headers, params=params, timeout=timeout)
        if r.status_code == 200:
            data  = r.json()["data"]
            score = data.get("abuseConfidenceScore", 0)
            result = {"checked": True, "malicious": score >= 50, "abuse_score": score,
                      "total_reports": data.get("totalReports", 0),
                      "country": data.get("countryCode", ""), "isp": data.get("isp", ""),
                      "is_tor": data.get("isTor", False)}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def threat_intel_check(urls=None, ips=None, hashes=None) -> dict:
    urls, ips, hashes = urls or [], ips or [], hashes or []
    iocs_detected, details = [], {"urls": [], "ips": [], "hashes": []}
    apis_used = {"virustotal": bool(VT_API_KEY), "abuseipdb": bool(ABUSEIPDB_KEY)}
    score = 0.0

    def _run(fn, arg, timeout=6):
        try:
            return _ti_executor.submit(fn, arg).result(timeout=timeout)
        except Exception as e:
            return {"checked": False, "malicious": False, "reason": str(e)[:80]}

    for h in hashes[:2]:
        r = _run(_vt_check_hash, h)
        details["hashes"].append({"hash": h, **r})
        if r.get("malicious"):
            score += 0.95
            iocs_detected.append(f"Hash malveillant (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {h[:16]}...")

    for u in urls[:3]:
        r = _run(_vt_check_url, u)
        details["urls"].append({"url": u, **r})
        if r.get("malicious"):
            score += 0.85
            iocs_detected.append(f"URL malveillante (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {u[:50]}")

    for ip in ips[:3]:
        r = _run(_abuseipdb_check, ip)
        details["ips"].append({"ip": ip, **r})
        if r.get("malicious"):
            score += 0.70
            iocs_detected.append(f"IP malveillante (AbuseIPDB {r.get('abuse_score',0)}%) : {ip}")

    return {"score": round(min(score, 1.0), 4), "iocs_detected": iocs_detected,
            "details": details, "apis_used": apis_used}

# ══════════════════════════════════════════════════════════════
# UTILITAIRES
# ══════════════════════════════════════════════════════════════
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_urls(text: str) -> list:
    return list(set(re.findall(r"https?://[^\s<>\"\']+|www\.[^\s<>\"\']+", text)))

def get_domain(url: str) -> str:
    m = re.search(r"(?:https?://)?(?:www\.)?([^/\s]+)", url)
    return m.group(1).lower() if m else ""

# ══════════════════════════════════════════════════════════════
# MODULE 1 — PARSER .EML
# ══════════════════════════════════════════════════════════════
def parse_eml(raw_bytes: bytes) -> dict:
    msg = BytesParser(policy=policy.default).parsebytes(raw_bytes)
    body_text, body_html = "", ""
    images, attachments  = [], []
    for part in msg.walk():
        ct = part.get_content_type()
        cd = str(part.get("Content-Disposition", ""))
        if ct == "text/plain" and "attachment" not in cd:
            body_text += part.get_content() or ""
        elif ct == "text/html" and "attachment" not in cd:
            body_html += str(part.get_content() or "")
        elif ct.startswith("image/"):
            images.append({"content_type": ct, "data": part.get_payload(decode=True),
                           "name": part.get_filename("image")})
        elif "attachment" in cd or part.get_filename():
            payload = part.get_payload(decode=True)
            if payload:
                attachments.append({
                    "filename": part.get_filename("unknown"), "content_type": ct,
                    "data": payload, "md5": hashlib.md5(payload).hexdigest(),
                    "sha256": hashlib.sha256(payload).hexdigest(), "size": len(payload),
                })
    body_html_clean = re.sub(r"<[^>]+>", " ", body_html)
    full_text = f"{body_text} {body_html_clean}".strip()
    return {
        "headers": dict(msg.items()), "from": msg.get("From", ""),
        "reply_to": msg.get("Reply-To", ""), "return_path": msg.get("Return-Path", ""),
        "subject": msg.get("Subject", ""), "date": msg.get("Date", ""),
        "message_id": msg.get("Message-ID", ""), "x_mailer": msg.get("X-Mailer", ""),
        "body_text": body_text, "body_html": body_html, "full_text": full_text,
        "images": images, "attachments": attachments,
        "urls_in_html": extract_urls(body_html), "urls_in_text": extract_urls(body_text),
    }

# ══════════════════════════════════════════════════════════════
# MODULE 2 — HEADERS SMTP
# ══════════════════════════════════════════════════════════════
def analyze_headers(parsed: dict) -> dict:
    headers, findings, score = parsed["headers"], [], 0.0
    auth = str(headers.get("Authentication-Results", "")).lower()
    spf  = str(headers.get("Received-SPF", "")).lower()
    dkim = "dkim-signature" in {k.lower() for k in headers}
    spf_pass, dkim_pass, dmarc_pass = ("pass" in spf or "pass" in auth), (dkim and "dkim=pass" in auth), ("dmarc=pass" in auth)

    if not spf_pass: findings.append("SPF absent ou échoué"); score += 0.20
    if not dkim_pass: findings.append("DKIM absent ou échoué"); score += 0.20
    if not dmarc_pass: findings.append("DMARC absent ou échoué"); score += 0.10

    from_addr, reply_to = parsed["from"], parsed["reply_to"]
    if reply_to and from_addr and reply_to != from_addr:
        findings.append(f"Reply-To ≠ From : {reply_to}"); score += 0.20

    return_path = parsed["return_path"]
    if return_path and from_addr:
        fd, rd = get_domain(from_addr), get_domain(return_path)
        if fd and rd and fd != rd:
            findings.append(f"Return-Path ({rd}) ≠ From ({fd})"); score += 0.15

    mailer = parsed["x_mailer"].lower()
    for sm in ["mailchimp","sendgrid","phpmailer","massmailer","bulk"]:
        if sm in mailer:
            findings.append(f"X-Mailer suspect : {parsed['x_mailer']}"); score += 0.10; break

    msg_id = parsed["message_id"]
    if not msg_id or not re.match(r"<.+@.+>", msg_id):
        findings.append("Message-ID absent ou invalide"); score += 0.10

    received  = str(headers.get("Received", ""))
    ips_found = list(set(re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", received)))

    return {"score": round(min(score, 1.0), 3), "spf_pass": spf_pass, "dkim_pass": dkim_pass,
            "dmarc_pass": dmarc_pass, "findings": findings, "ips_found": ips_found[:5]}

# ══════════════════════════════════════════════════════════════
# MODULE 3 — URLs (whois via ThreadPoolExecutor — safe en thread)
# ══════════════════════════════════════════════════════════════
_whois_executor = ThreadPoolExecutor(max_workers=2)

def _whois_safe(domain: str) -> dict:
    try:
        return _whois_executor.submit(whois.whois, domain).result(timeout=3)
    except Exception:
        return {}

def analyze_urls(urls: list) -> dict:
    findings, suspicious, score = [], [], 0.0
    for url in urls[:10]:
        domain = get_domain(url)
        if not domain: continue

        for tld in SUSPICIOUS_TLDS:
            if domain.endswith(tld):
                findings.append(f"TLD suspect : {domain}"); score += 0.15; suspicious.append(url); break

        domain_base = domain.split(".")[0]
        for brand in KNOWN_BRANDS:
            if 0 < Levenshtein.distance(domain_base, brand) <= 2:
                findings.append(f"Typosquatting : {domain} ≈ {brand}"); score += 0.25; suspicious.append(url); break

        if len(domain) > 40: findings.append(f"Domaine trop long : {domain}"); score += 0.10
        if domain.count("-") >= 3: findings.append(f"Beaucoup de tirets : {domain}"); score += 0.10
        if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
            findings.append(f"IP directe : {url[:60]}"); score += 0.30; suspicious.append(url)

        url_norm = url.lower()
        if url_norm in urlhaus_set or url in urlhaus_set:
            findings.append(f"URLHaus blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)
        if url_norm in openphish_set or url in openphish_set:
            findings.append(f"OpenPhish blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)

        try:
            w = _whois_safe(domain)
            if w and w.get("creation_date"):
                creation = w["creation_date"]
                if isinstance(creation, list): creation = creation[0]
                if creation and (datetime.now() - creation).days < 30:
                    findings.append(f"Domaine très récent : {domain}"); score += 0.30; suspicious.append(url)
        except Exception:
            pass

    return {"score": round(min(score, 1.0), 3), "total_urls": len(urls),
            "suspicious": list(set(suspicious))[:5], "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 4 — OCR IMAGES
# ══════════════════════════════════════════════════════════════
def analyze_images(images: list) -> dict:
    ocr_texts, findings, score = [], [], 0.0
    if len(images) > 3:
        findings.append(f"{len(images)} images — contournement filtre"); score += 0.15
    for img_data in images[:5]:
        try:
            raw = img_data["data"]
            if not raw: continue
            img = Image.open(BytesIO(raw))
            w, h = img.size
            if w <= 3 and h <= 3:
                findings.append("Pixel tracking"); score += 0.20; continue
            text = pytesseract.image_to_string(img, lang="eng+fra")
            if text.strip():
                ocr_texts.append(text.strip())
                kw = [k for k in ["verify","account","password","click","urgent",
                                  "suspended","login","security","bank","credit card"]
                      if k in text.lower()]
                if kw: findings.append(f"Mots-clés phishing image : {kw}"); score += 0.20
        except Exception:
            pass
    return {"score": round(min(score, 1.0), 3), "ocr_text": " ".join(ocr_texts)[:500],
            "images_count": len(images), "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 5 — PIÈCES JOINTES
# ══════════════════════════════════════════════════════════════
def analyze_attachments(attachments: list) -> dict:
    findings, suspicious_hashes, score = [], [], 0.0
    DANGEROUS_EXT = {".exe",".bat",".cmd",".ps1",".vbs",".js",".jar",".scr",".pif",".com",".msi"}
    DOUBLE_EXT    = re.compile(r"\.(pdf|docx?|xlsx?|zip)\.(exe|bat|cmd|scr)$", re.I)
    for att in attachments:
        fname, ct = att["filename"].lower(), att["content_type"]
        for ext in DANGEROUS_EXT:
            if fname.endswith(ext):
                findings.append(f"Extension dangereuse : {att['filename']}"); score += 0.40; break
        if DOUBLE_EXT.search(fname):
            findings.append(f"Double extension : {att['filename']}"); score += 0.50
        try:
            import magic
            real_mime = magic.from_buffer(att["data"][:1024], mime=True)
            if real_mime and real_mime not in ct:
                findings.append(f"MIME réel ({real_mime}) ≠ déclaré ({ct})"); score += 0.35
        except Exception:
            pass
        if fname.endswith((".doc",".docx",".xls",".xlsx",".ppt",".pptx")):
            try:
                from oletools.olevba import VBA_Parser
                vba = VBA_Parser(att["filename"], data=att["data"])
                if vba.detect_vba_macros():
                    findings.append(f"Macros VBA : {att['filename']}"); score += 0.45
            except Exception:
                pass
        suspicious_hashes.append({"filename": att["filename"], "md5": att["md5"],
                                  "sha256": att["sha256"], "size": att["size"]})
    return {"score": round(min(score, 1.0), 3), "attachments_count": len(attachments),
            "suspicious_hashes": suspicious_hashes, "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 6 — NLP
# ══════════════════════════════════════════════════════════════
def analyze_text(text: str) -> dict:
    text_clean = clean_text(text)
    if not text_clean.strip():
        return {"bert_score":0.0,"lr_score":0.0,"score":0.0,"keywords":[]}
    inputs = tokenizer(text_clean, truncation=True, padding=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    bert_score = float(torch.softmax(logits, dim=1).numpy()[0][1])
    lr_score   = float(lr_model.predict_proba(tfidf.transform([text_clean]))[0][1])
    keywords = [kw for kw in [
        "verify","account","password","click here","urgent","confirm","suspended","login",
        "update","security","limited","expire","winner","prize","claim","free","bank",
        "credit card","wire transfer",
    ] if kw in text.lower()]
    return {"bert_score": round(bert_score, 4), "lr_score": round(lr_score, 4),
            "score": round(bert_score * 0.6 + lr_score * 0.4, 4), "keywords": keywords}

# ══════════════════════════════════════════════════════════════
# MODULE 7 — RAG
# ══════════════════════════════════════════════════════════════
def rag_search(text: str) -> dict:
    if not RAG_OK:
        return {"techniques":[],"apt_groups":[],"campaigns":[],"live_iocs":[]}
    q_emb = embed_fn([text[:500]])
    def _query(col, n):
        if not col: return None
        try: return col.query(query_embeddings=q_emb, n_results=n)
        except Exception: return None
    def _parse(res, fields):
        if not res or not res["ids"][0]: return []
        out = []
        for i in range(len(res["ids"][0])):
            item = {"similarity": round(1 - res["distances"][0][i], 3)}
            for f in fields: item[f] = res["metadatas"][0][i].get(f, "")
            out.append(item)
        return out
    return {
        "techniques": _parse(_query(col_techniques, 3), ["tech_id","name","tactics"]),
        "apt_groups": _parse(_query(col_apt, 2),        ["group_id","name","aliases"]),
        "campaigns":  _parse(_query(col_ti, 2),         ["campaign","actor","technique","severity"]),
        "live_iocs":  _parse(_query(col_live, 2),       ["url","attack_type","brand_target"]) if col_live else [],
    }

# ══════════════════════════════════════════════════════════════
# MODULE 8 — RAPPORT LLM (JSON sérialisé proprement — fix intégré)
# ══════════════════════════════════════════════════════════════
def _call_llm(prompt: str) -> str:
    if llm_backend == "groq":
        resp = llm_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role":"user","content":prompt}],
            temperature=0.1, max_tokens=800,
        )
        return resp.choices[0].message.content
    elif llm_backend == "gemini":
        return llm_client.generate_content(prompt).text
    return ""

def generate_report(parsed, header_r, url_r, image_r, attach_r,
                    text_r, rag_r, ti_r, score_global) -> dict:
    top_tech     = rag_r["techniques"][0] if rag_r["techniques"] else {}
    top_campaign = rag_r["campaigns"][0]  if rag_r["campaigns"]  else {}
    top_apt      = rag_r["apt_groups"][0] if rag_r["apt_groups"] else {}
    all_findings = (header_r["findings"] + url_r["findings"] + image_r["findings"] + attach_r["findings"])
    iocs_detected = ti_r.get("iocs_detected", [])

    if not llm_client:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {
            "verdict": verdict, "confiance": "haute" if score_global >= 0.75 else "moyenne",
            "score_final": score_global,
            "campagne_probable": top_campaign.get("campaign","Inconnue"),
            "technique_attck": f"{top_tech.get('tech_id','')} — {top_tech.get('name','N/A')}",
            "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
            "indicateurs_cles": all_findings[:5],
            "explication": f"Score {score_global:.2f}. {len(all_findings)} indicateurs. LLM indisponible.",
            "recommandation": "Bloquer et quarantaine." if score_global >= 0.65 else "Vérification manuelle.",
            "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                     "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
            "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else "",
        }

    # ── Sérialisation JSON propre (FIX) — listes Python → JSON valide ──
    urls_json   = _json.dumps(url_r["suspicious"])
    ips_json    = _json.dumps(header_r["ips_found"])
    hashes_json = _json.dumps([h["sha256"] for h in attach_r["suspicious_hashes"]])
    ti_block    = f"IoCs confirmés externes : {iocs_detected}" if iocs_detected else "Aucun IoC externe confirmé"

    prompt = f"""Tu es un analyste SOC expert en détection de phishing.

=== EMAIL ===
De      : {parsed.get("from","")}
Sujet   : {parsed.get("subject","")}
Extrait : {parsed.get("full_text","")[:300]}

=== SCORES ===
Headers : {header_r["score"]} | SPF={header_r["spf_pass"]} DKIM={header_r["dkim_pass"]} DMARC={header_r["dmarc_pass"]}
URLs    : {url_r["score"]} ({url_r["total_urls"]} URLs, {len(url_r["suspicious"])} suspectes)
OCR     : {image_r["score"]} | Attachments : {attach_r["score"]}
NLP     : BERT={text_r["bert_score"]} LR={text_r["lr_score"]}
TI      : {ti_r.get("score",0)} | Global : {score_global}

=== INDICATEURS ===
{chr(10).join(all_findings) if all_findings else "Aucun"}

=== THREAT INTEL EXTERNE (VirusTotal / AbuseIPDB) ===
{ti_block}

=== RAG ===
MITRE   : {top_tech.get("tech_id","")} {top_tech.get("name","")} (sim={top_tech.get("similarity",0)})
APT     : {top_apt.get("name","")} (sim={top_apt.get("similarity",0)})
Campagne: {top_campaign.get("campaign","")} — {top_campaign.get("actor","")} (sim={top_campaign.get("similarity",0)})

Réponds STRICTEMENT avec un JSON valide (syntaxe JSON pure, pas Python), sans texte avant/après, sans markdown ni backticks. Respecte exactement ce format :

{{"verdict": "PHISHING", "confiance": "haute", "score_final": {score_global}, "campagne_probable": "nom ou Aucune", "technique_attck": "T#### - nom", "vecteur_attaque": "lien", "indicateurs_cles": ["max 5, mentionne les hits VT/AbuseIPDB si presents"], "explication": "2-3 phrases SOC mentionnant les IoCs externes confirmés si presents", "recommandation": "action", "iocs": {{"urls_suspectes": {urls_json}, "ips_malveillantes": {ips_json}, "hashes": {hashes_json}}}, "sigma_hint": "valeur"}}"""

    try:
        raw = _call_llm(prompt)
        raw = re.sub(r"```json|```", "", raw).strip()
        m   = re.search(r"\{.*\}", raw, re.DOTALL)
        return _json.loads(m.group() if m else raw)
    except Exception as e:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {
            "verdict": verdict, "confiance": "moyenne", "score_final": score_global,
            "explication": f"Erreur LLM : {str(e)[:100]}",
            "campagne_probable": top_campaign.get("campaign","N/A"),
            "technique_attck": f"{top_tech.get('tech_id','')} {top_tech.get('name','')}",
            "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
            "indicateurs_cles": all_findings[:5],
            "recommandation": "Bloquer et quarantaine." if score_global >= 0.65 else "Vérification manuelle.",
            "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                     "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
            "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else "",
        }

# ══════════════════════════════════════════════════════════════
# MODULE 9 — SIGMA RULES (intégré nativement)
# ══════════════════════════════════════════════════════════════
def _mitre_to_tag(technique_attck: str) -> str:
    m = re.search(r"T(\d{4})(?:\.(\d{3}))?", technique_attck or "")
    if not m: return "attack.t1566"
    base = f"attack.t{m.group(1)}"
    return f"{base}.{m.group(2)}" if m.group(2) else base

def _tactic_tag(tech_id: str) -> str:
    TACTIC_MAP = {"T1566":"attack.initial_access","T1598":"attack.reconnaissance",
                  "T1556":"attack.credential_access","T1539":"attack.credential_access",
                  "T1204":"attack.execution","T1027":"attack.defense_evasion"}
    base = tech_id.split(".")[0] if tech_id else ""
    return TACTIC_MAP.get(base, "attack.initial_access")

def _level_from_score(score: float) -> str:
    if score >= 0.85: return "critical"
    if score >= 0.65: return "high"
    if score >= 0.40: return "medium"
    if score >= 0.20: return "low"
    return "informational"

def generate_sigma_rules(report: dict, score_global: float, email_meta: dict = None) -> dict:
    email_meta = email_meta or {}
    rules = []
    level = _level_from_score(score_global)
    tech_id = (report.get("technique_attck","") or "").split(" ")[0].strip("—-").strip()
    tag_technique, tag_tactic = _mitre_to_tag(tech_id), _tactic_tag(tech_id)
    rule_date = datetime.now().strftime("%Y/%m/%d")

    iocs = report.get("iocs", {})
    urls_susp, ips_mal, hashes_mal = iocs.get("urls_suspectes",[]), iocs.get("ips_malveillantes",[]), iocs.get("hashes",[])
    common_tags = ["attack.phishing", tag_tactic, tag_technique]
    common_refs = [f"https://attack.mitre.org/techniques/{tech_id}"] if tech_id else []

    if urls_susp:
        domains = list(set([re.sub(r"^https?://","",u).split("/")[0] for u in urls_susp]))
        rules.append({"type":"proxy_network","data":{
            "title": f"Accès à un domaine de phishing détecté par DASEC ({report.get('campagne_probable','Inconnue')})",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Connexion vers domaine malveillant. Score : {score_global:.2f}. Vecteur : {report.get('vecteur_attaque','lien')}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"proxy"},
            "detection":{"selection":{"c-uri|contains": domains if len(domains)>1 else domains[0]},"condition":"selection"},
            "falsepositives":["Domaine réutilisé légitimement","Faux positif AV tiers"],"level":level,
        }})

    sender, subject = email_meta.get("from",""), email_meta.get("subject","")
    if sender or subject:
        selection = {}
        if sender:
            selection["sender|contains"] = sender.split("@")[-1] if "@" in sender else sender
        if subject:
            selection["subject|contains"] = subject[:60]
        rules.append({"type":"mail_gateway","data":{
            "title": f"Email de phishing détecté par DASEC — {report.get('campagne_probable','Inconnue')}",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Caractéristiques phishing : {', '.join(report.get('indicateurs_cles',[])[:3]) or 'analyse NLP'}. MITRE : {report.get('technique_attck','N/A')}.",
            "references": common_refs,
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"mail"},
            "detection":{"selection":selection,"condition":"selection"},
            "falsepositives":["Email légitime — vérifier SPF/DKIM/DMARC"],"level":level,
        }})

    if hashes_mal:
        rules.append({"type":"file_hash","data":{
            "title":"Pièce jointe malveillante détectée par DASEC (hash confirmé VirusTotal)",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Hash confirmé malveillant par VirusTotal. Score : {score_global:.2f}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags+["attack.execution"],
            "logsource":{"category":"file_event"},
            "detection":{"selection":{"Hashes|contains":hashes_mal},"condition":"selection"},
            "falsepositives":["Aucun connu"],"level":"critical",
        }})

    yaml_blocks = []
    for r in rules:
        clean = {k:v for k,v in r["data"].items() if v is not None}
        yaml_str = yaml.dump(clean, sort_keys=False, allow_unicode=True, default_flow_style=False, width=100)
        yaml_blocks.append(yaml_str)
        r["yaml"] = yaml_str

    combined = "\n---\n".join(yaml_blocks) if yaml_blocks else "# Aucune règle générée"
    return {"rules_count": len(rules), "rules": [{"type":r["type"],"yaml":r["yaml"]} for r in rules],
            "combined_yaml": combined}

# ══════════════════════════════════════════════════════════════
# MODULE 10 — ALERTE SOC
# ══════════════════════════════════════════════════════════════
def send_soc_alert(report: dict, parsed: dict, score: float) -> bool:
    if not SLACK_TOKEN or score < SOC_THRESHOLD:
        return False
    try:
        from slack_sdk import WebClient
        client = WebClient(token=SLACK_TOKEN)
        verdict = report.get("verdict","?")
        emoji = "🚨" if verdict == "PHISHING" else "⚠️"
        msg = (f"{emoji} *DASEC — {verdict}*\n>Score : {score:.2f} | Confiance : {report.get('confiance','?')}\n"
               f">De : {parsed.get('from','')}\n>Sujet : {parsed.get('subject','')}\n"
               f">Campagne : {report.get('campagne_probable','N/A')}\n>Technique : {report.get('technique_attck','N/A')}\n"
               f">Action : {report.get('recommandation','N/A')}\n>Indicateurs : {', '.join(report.get('indicateurs_cles',[])[:3])}")
        client.chat_postMessage(channel=SLACK_CHANNEL, text=msg)
        return True
    except Exception as e:
        print(f"Slack error: {e}"); return False

# ══════════════════════════════════════════════════════════════
# SCORE GLOBAL
# ══════════════════════════════════════════════════════════════
def compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full") -> float:
    if mode == "text_only":
        return round(text_r["score"]*0.50 + url_r["score"]*0.35 + ti_r.get("score",0.0)*0.10 +
                    (len(url_r["suspicious"])>0)*0.05, 4)
    return round(header_r["score"]*0.25 + text_r["score"]*0.20 + url_r["score"]*0.20 +
                attach_r["score"]*0.10 + ti_r.get("score",0.0)*0.15 + image_r["score"]*0.05 +
                (len(url_r["suspicious"])>0)*0.05, 4)

# ══════════════════════════════════════════════════════════════
# FASTAPI
# ══════════════════════════════════════════════════════════════
app = FastAPI(title="DASEC — Phishing Detection API",
             description="Headers+URLs+OCR+Attachments+NLP+RAG+LLM+ThreatIntel+Sigma+SOC", version="3.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class TextRequest(BaseModel):
    text: str
    subject: Optional[str] = ""
    sender: Optional[str] = ""

EMPTY_HEADER = {"score":0.0,"spf_pass":None,"dkim_pass":None,"dmarc_pass":None,"findings":[],"ips_found":[]}
EMPTY_IMAGE  = {"score":0.0,"ocr_text":"","images_count":0,"findings":[]}
EMPTY_ATTACH = {"score":0.0,"attachments_count":0,"suspicious_hashes":[],"findings":[]}

@app.get("/")
def root():
    return {"message":"DASEC v3.0","status":"running","rag":RAG_OK,"llm_backend":llm_backend or "none"}

@app.get("/health")
def health():
    return {
        "status":"ok","models":["distilbert","lr","sentence-transformers"],
        "rag_collections":{
            "mitre_techniques": col_techniques.count() if col_techniques else 0,
            "mitre_apt_groups": col_apt.count() if col_apt else 0,
            "threat_intel": col_ti.count() if col_ti else 0,
            "live_ioc_feeds": col_live.count() if col_live else 0,
        },
        "ioc_feeds":{"urlhaus":len(urlhaus_set),"openphish":len(openphish_set)},
        "threat_intel_apis":{"virustotal":bool(VT_API_KEY),"abuseipdb":bool(ABUSEIPDB_KEY)},
        "llm_backend":llm_backend or "none","soc_alert":bool(SLACK_TOKEN),"soc_threshold":SOC_THRESHOLD,
    }

@app.post("/analyze/eml")
async def analyze_eml(file: UploadFile = File(...)):
    if not file.filename.endswith(".eml"):
        raise HTTPException(400, "Fichier .eml requis")
    raw = await file.read()
    parsed = parse_eml(raw)
    all_urls = list(set(parsed["urls_in_html"] + parsed["urls_in_text"]))
    header_r = analyze_headers(parsed)
    url_r    = analyze_urls(all_urls)
    image_r  = analyze_images(parsed["images"])
    attach_r = analyze_attachments(parsed["attachments"])
    full_text = parsed["full_text"] + " " + image_r["ocr_text"]
    text_r = analyze_text(full_text)
    rag_r  = rag_search(full_text)
    ti_r = threat_intel_check(
        urls=url_r["suspicious"][:3], ips=header_r["ips_found"][:3],
        hashes=[a["sha256"] for a in parsed["attachments"]][:2],
    )
    score_global = compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full")
    report = generate_report(parsed, header_r, url_r, image_r, attach_r, text_r, rag_r, ti_r, score_global)
    sigma_r = generate_sigma_rules(report, score_global, {"from":parsed["from"],"subject":parsed["subject"]})
    alert_sent = send_soc_alert(report, parsed, score_global)
    return {
        "score_global": score_global, "verdict": report.get("verdict","?"),
        "modules":{"headers":header_r,"urls":url_r,"images":image_r,"attachments":attach_r,
                   "nlp":text_r,"rag":rag_r,"threat_intel":ti_r},
        "report": report, "sigma_rules": sigma_r, "alert_sent": alert_sent,
        "email_meta":{"from":parsed["from"],"subject":parsed["subject"],"date":parsed["date"]},
    }

@app.post("/analyze/text")
async def analyze_text_endpoint(req: TextRequest):
    text_r = analyze_text(req.text)
    url_r  = analyze_urls(extract_urls(req.text))
    rag_r  = rag_search(req.text)
    ti_r   = threat_intel_check(urls=url_r["suspicious"][:2])
    score_global = compute_global_score(EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, ti_r, mode="text_only")
    report = generate_report(
        {"from":req.sender,"subject":req.subject,"full_text":req.text},
        EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, rag_r, ti_r, score_global,
    )
    sigma_r = generate_sigma_rules(report, score_global, {"from":req.sender,"subject":req.subject})
    return {
        "score_global": score_global, "verdict": report.get("verdict","?"),
        "nlp": text_r, "urls": url_r, "rag": rag_r, "threat_intel": ti_r,
        "report": report, "sigma_rules": sigma_r,
    }
'''

with open("/content/api_v2.py", "w") as f:
    f.write(api_code)
print("✅ api_v2.py v3.0 écrit (ThreatIntel + Sigma natifs, plus de patch)")

# ══════════════════════════════════════════════════════════════
sys.path.insert(0, "/content")
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f"\n🌍 API : {PUBLIC_URL}")
print(f"📖 Docs : {PUBLIC_URL}/docs")
print(f"🔎 Health : {PUBLIC_URL}/health")

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    import uvicorn
    config = uvicorn.Config("api_v2:app", host="0.0.0.0", port=8000, log_level="warning", loop="asyncio")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(4)

print("\n✅ Serveur DASEC v3.0 démarré (tout natif, zéro patch à chaud) !")

In [ ]:
import requests

r = requests.post(
    "https://sliced-tinderbox-crock.ngrok-free.dev/analyze/text",
    json={
        "text":    "URGENT verify your account password now http://secure-login.tk/verify",
        "subject": "Security Alert",
        "sender":  "security@paypa1.com"
    }
)

print("Score  :", r.json()["score_global"])
print("Verdict:", r.json()["verdict"])
print("Rapport:", r.json()["report"])

Score  : 0.6775
Verdict: PHISHING
Rapport: {'verdict': 'PHISHING', 'confiance': 'haute', 'score_final': 0.6775, 'campagne_probable': 'Threat Group-1314', 'technique_attck': 'T1556 - Modify Authentication Process', 'vecteur_attaque': 'lien', 'indicateurs_cles': ['TLD suspect : secure-login.tk', 'URL malveillante (VT 9/92) : http://secure-login.tk/verify'], 'explication': "L'email contient un lien suspect (http://secure-login.tk/verify) qui a été identifié comme malveillant par VirusTotal (9/92). De plus, le TLD 'secure-login.tk' est suspect.", 'recommandation': "Vérifier l'authenticité de l'email et ne pas cliquer sur le lien suspect.", 'iocs': {'urls_suspectes': ['http://secure-login.tk/verify'], 'ips_malveillantes': [], 'hashes': []}, 'sigma_hint': 'T1556 Modify Authentication Process'}


In [ ]:
# ══════════════════════════════════════════════════════════════
# PATCH À CHAUD — Bascule Gemini → Groq (modèle stable)
# ══════════════════════════════════════════════════════════════
import os
from google.colab import userdata

GROQ_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_KEY

# Recharge le module api_v2 déjà importé en mémoire par uvicorn
import api_v2
from groq import Groq

api_v2.llm_client  = Groq(api_key=GROQ_KEY)
api_v2.llm_backend = "groq"

print(f"✅ LLM basculé sur : {api_v2.llm_backend}")

In [ ]:
import requests, json

r = requests.post("https://sliced-tinderbox-crock.ngrok-free.dev/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com"
})
result = r.json()
print("Score  :", result["score_global"])
print("Verdict:", result["verdict"])
print("Threat Intel:", json.dumps(result["threat_intel"], indent=2, ensure_ascii=False))
print("\nRapport:", json.dumps(result["report"], indent=2, ensure_ascii=False))

In [ ]:
import requests, json

# PUBLIC_URL a été affiché par la cellule F au démarrage
# Copie-colle la valeur ici — ne laisse JAMAIS une URL en dur
URL = "https://sliced-tinderbox-crock.ngrok-free.dev"  # ← remplace par ce que la cellule F a affiché

r = requests.post(f"{URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com"
})
result = r.json()
print("Score  :", result["score_global"])
print("Verdict:", result["verdict"])
print("Sigma  :", result["sigma_rules"]["combined_yaml"])
print("Rapport:", json.dumps(result["report"], indent=2, ensure_ascii=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE H — SQLite PERSISTENCE
# Log chaque analyse dans Drive/phishing_dasec/dasec.db
# Prérequis : CELLULE 0 exécutée (BASE_DIR défini)
# ══════════════════════════════════════════════════════════════

import sqlite3
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

DB_PATH = f"{BASE_DIR}/dasec.db"

# ══════════════════════════════════════════════════════════════
# PARTIE 1 — Initialisation (idempotente — safe à ré-exécuter)
# ══════════════════════════════════════════════════════════════

def init_db():
    """Crée les tables si elles n'existent pas encore."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS analyses (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id     TEXT    UNIQUE NOT NULL,
            analyzed_at     TEXT    NOT NULL,
            mode            TEXT    NOT NULL,          -- 'eml' ou 'text'
            score_global    REAL    NOT NULL,
            verdict         TEXT    NOT NULL,
            confidence      TEXT,
            sender          TEXT,
            subject         TEXT,
            technique_attck TEXT,
            campaign        TEXT,
            vecteur_attaque TEXT,
            sigma_count     INTEGER DEFAULT 0,
            alert_sent      INTEGER DEFAULT 0,         -- 0/1
            report_json     TEXT    NOT NULL           -- dict complet JSON
        );

        CREATE TABLE IF NOT EXISTS iocs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id TEXT NOT NULL,
            ioc_type    TEXT NOT NULL,                 -- 'url', 'ip', 'hash'
            ioc_value   TEXT NOT NULL,
            source      TEXT,                          -- 'URLHaus', 'VT', etc.
            detected_at TEXT NOT NULL,
            FOREIGN KEY (analysis_id) REFERENCES analyses(analysis_id)
        );

        CREATE INDEX IF NOT EXISTS idx_analyses_verdict   ON analyses(verdict);
        CREATE INDEX IF NOT EXISTS idx_analyses_score     ON analyses(score_global);
        CREATE INDEX IF NOT EXISTS idx_analyses_date      ON analyses(analyzed_at);
        CREATE INDEX IF NOT EXISTS idx_iocs_analysis_id   ON iocs(analysis_id);
        CREATE INDEX IF NOT EXISTS idx_iocs_type          ON iocs(ioc_type);
    """)
    con.commit()
    con.close()
    print(f"✅ DB initialisée : {DB_PATH}")

init_db()

# ══════════════════════════════════════════════════════════════
# PARTIE 2 — Fonctions CRUD
# ══════════════════════════════════════════════════════════════

def save_analysis(
    result: dict,
    mode: str = "text",
    sender: str = "",
    subject: str = "",
    alert_sent: bool = False,
) -> str:
    """
    Persiste une analyse complète (retour de /analyze/eml ou /analyze/text).
    Retourne l'analysis_id généré (UUID4).
    """
    analysis_id = str(uuid.uuid4())
    now         = datetime.now(timezone.utc).isoformat()

    report      = result.get("report", {})
    sigma_r     = result.get("sigma_rules", {})
    iocs        = report.get("iocs", {})
    ti          = result.get("threat_intel", result.get("modules", {}).get("threat_intel", {}))

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    cur.execute("""
        INSERT OR IGNORE INTO analyses
          (analysis_id, analyzed_at, mode, score_global, verdict, confidence,
           sender, subject, technique_attck, campaign, vecteur_attaque,
           sigma_count, alert_sent, report_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        analysis_id,
        now,
        mode,
        result.get("score_global", 0.0),
        report.get("verdict",  result.get("verdict", "UNKNOWN")),
        report.get("confiance", ""),
        sender  or report.get("iocs", {}).get("sender", ""),
        subject or "",
        report.get("technique_attck", ""),
        report.get("campagne_probable", ""),
        report.get("vecteur_attaque", ""),
        sigma_r.get("rules_count", 0),
        int(alert_sent),
        json.dumps(result, ensure_ascii=False),
    ))

    # IoCs détectés
    rows = []
    for url in iocs.get("urls_suspectes", []):
        rows.append((analysis_id, "url", url, "pipeline", now))
    for ip in iocs.get("ips_malveillantes", []):
        rows.append((analysis_id, "ip", ip, "AbuseIPDB", now))
    for h in iocs.get("hashes", []):
        rows.append((analysis_id, "hash", h, "VirusTotal", now))
    # IoCs confirmés par TI externe
    for ioc_str in ti.get("iocs_detected", []):
        src = "VT" if "VT" in ioc_str else ("AbuseIPDB" if "AbuseIPDB" in ioc_str else "TI")
        rows.append((analysis_id, "ioc_external", ioc_str[:200], src, now))

    if rows:
        cur.executemany("""
            INSERT INTO iocs (analysis_id, ioc_type, ioc_value, source, detected_at)
            VALUES (?,?,?,?,?)
        """, rows)

    con.commit()
    con.close()
    return analysis_id


def get_recent_analyses(limit: int = 20, verdict_filter: str = None) -> list:
    """Retourne les N dernières analyses (dict), filtrables par verdict."""
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()

    if verdict_filter:
        cur.execute("""
            SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                   sender, subject, technique_attck, campaign, sigma_count, alert_sent
            FROM analyses
            WHERE verdict = ?
            ORDER BY analyzed_at DESC
            LIMIT ?
        """, (verdict_filter.upper(), limit))
    else:
        cur.execute("""
            SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                   sender, subject, technique_attck, campaign, sigma_count, alert_sent
            FROM analyses
            ORDER BY analyzed_at DESC
            LIMIT ?
        """, (limit,))

    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows


def get_analysis_by_id(analysis_id: str) -> dict | None:
    """Récupère une analyse complète (avec report_json parsé) par son ID."""
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM analyses WHERE analysis_id = ?", (analysis_id,))
    row = cur.fetchone()
    con.close()
    if not row:
        return None
    d = dict(row)
    try:
        d["report"] = json.loads(d.pop("report_json"))
    except Exception:
        d["report"] = {}
    return d


def get_iocs_by_analysis(analysis_id: str) -> list:
    """Retourne tous les IoCs d'une analyse."""
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("""
        SELECT ioc_type, ioc_value, source, detected_at
        FROM iocs WHERE analysis_id = ?
        ORDER BY ioc_type
    """, (analysis_id,))
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows


def get_stats() -> dict:
    """Stats globales de la DB (dashboard)."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    cur.execute("SELECT COUNT(*) FROM analyses")
    total = cur.fetchone()[0]

    cur.execute("SELECT verdict, COUNT(*) FROM analyses GROUP BY verdict")
    by_verdict = dict(cur.fetchall())

    cur.execute("SELECT AVG(score_global) FROM analyses")
    avg_score = round(cur.fetchone()[0] or 0.0, 4)

    cur.execute("SELECT COUNT(*) FROM iocs")
    total_iocs = cur.fetchone()[0]

    cur.execute("""
        SELECT technique_attck, COUNT(*) as cnt
        FROM analyses
        WHERE technique_attck != ''
        GROUP BY technique_attck
        ORDER BY cnt DESC
        LIMIT 5
    """)
    top_techniques = cur.fetchall()

    cur.execute("""
        SELECT campaign, COUNT(*) as cnt
        FROM analyses
        WHERE campaign != ''
        GROUP BY campaign
        ORDER BY cnt DESC
        LIMIT 5
    """)
    top_campaigns = cur.fetchall()

    con.close()
    return {
        "total_analyses": total,
        "by_verdict":     by_verdict,
        "avg_score":      avg_score,
        "total_iocs":     total_iocs,
        "top_techniques": [{"technique": t, "count": c} for t, c in top_techniques],
        "top_campaigns":  [{"campaign": c, "count": n} for c, n in top_campaigns],
    }


# ══════════════════════════════════════════════════════════════
# PARTIE 3 — Intégration dans api_v2.py (patch à chaud)
# On injecte save_analysis dans les endpoints /analyze/*
# ══════════════════════════════════════════════════════════════

import api_v2

# Injecter les fonctions dans le module api_v2
api_v2.save_analysis        = save_analysis
api_v2.get_recent_analyses  = get_recent_analyses
api_v2.get_analysis_by_id   = get_analysis_by_id
api_v2.get_iocs_by_analysis = get_iocs_by_analysis
api_v2.get_stats            = get_stats
api_v2.DB_PATH              = DB_PATH

# ── Patch endpoint /analyze/eml ──────────────────────────────
_orig_eml = api_v2.analyze_eml.__wrapped__ if hasattr(api_v2.analyze_eml, "__wrapped__") else None

from fastapi import UploadFile, File
from functools import wraps

original_analyze_eml  = api_v2.app.routes  # on va patcher via le router

# Approche propre : on redéfinit les fonctions métier dans api_v2
# et on les wrape avec save_analysis

_orig_eml_fn  = None
_orig_text_fn = None

# Récupère les fonctions handler depuis les routes FastAPI
for route in api_v2.app.routes:
    if hasattr(route, "path"):
        if route.path == "/analyze/eml":
            _orig_eml_fn  = route.endpoint
        elif route.path == "/analyze/text":
            _orig_text_fn = route.endpoint

print(f"✅ Handlers trouvés : eml={_orig_eml_fn is not None}, text={_orig_text_fn is not None}")

# Wrapper eml
if _orig_eml_fn:
    async def _patched_eml(file: UploadFile = File(...)):
        result = await _orig_eml_fn(file=file)
        meta   = result.get("email_meta", {})
        aid    = save_analysis(
            result,
            mode="eml",
            sender=meta.get("from", ""),
            subject=meta.get("subject", ""),
            alert_sent=result.get("alert_sent", False),
        )
        result["analysis_id"] = aid
        return result

    for route in api_v2.app.routes:
        if hasattr(route, "path") and route.path == "/analyze/eml":
            route.endpoint = _patched_eml
            break

# Wrapper text
if _orig_text_fn:
    from api_v2 import TextRequest

    async def _patched_text(req: TextRequest):
        result = await _orig_text_fn(req=req)
        aid    = save_analysis(
            result,
            mode="text",
            sender=req.sender or "",
            subject=req.subject or "",
            alert_sent=False,
        )
        result["analysis_id"] = aid
        return result

    for route in api_v2.app.routes:
        if hasattr(route, "path") and route.path == "/analyze/text":
            route.endpoint = _patched_text
            break

print("✅ Endpoints patchés avec persistence SQLite")

# ── Ajout des endpoints de consultation ──────────────────────
from fastapi import Query
from fastapi.responses import JSONResponse

@api_v2.app.get("/history")
def history(limit: int = Query(20, ge=1, le=100), verdict: str = Query(None)):
    """Dernières analyses (filtrables par verdict : PHISHING / SUSPECT / LEGITIME)."""
    return get_recent_analyses(limit=limit, verdict_filter=verdict)

@api_v2.app.get("/history/{analysis_id}")
def history_detail(analysis_id: str):
    """Détail complet d'une analyse (report + IoCs)."""
    row = get_analysis_by_id(analysis_id)
    if not row:
        raise HTTPException(404, f"Analyse {analysis_id} introuvable")
    row["iocs_detail"] = get_iocs_by_analysis(analysis_id)
    return row

@api_v2.app.get("/stats")
def stats():
    """Stats globales de toutes les analyses loggées."""
    return get_stats()

print("✅ Endpoints /history, /history/{id}, /stats ajoutés")

# ══════════════════════════════════════════════════════════════
# PARTIE 4 — Test end-to-end
# ══════════════════════════════════════════════════════════════

import requests, time

URL = PUBLIC_URL   # défini par la cellule F

# Analyse test
r = requests.post(f"{URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
result = r.json()
aid = result.get("analysis_id", "")
print(f"✅ Analyse loggée : {aid}")
print(f"   Verdict : {result['verdict']} | Score : {result['score_global']}")

# Vérification via /history
time.sleep(0.5)
h = requests.get(f"{URL}/history?limit=5").json()
print(f"\n📋 /history — {len(h)} entrées récentes :")
for row in h:
    print(f"  [{row['verdict']:8}] {row['score_global']:.3f} | {row['subject'] or '(no subject)'} | {row['analyzed_at'][:19]}")

# Détail via /history/{id}
if aid:
    d = requests.get(f"{URL}/history/{aid}").json()
    print(f"\n🔍 /history/{aid[:8]}... :")
    print(f"  Technique : {d.get('technique_attck','')}")
    print(f"  Campagne  : {d.get('campaign','')}")
    print(f"  IoCs      : {d.get('iocs_detail',[])}")

# Stats
s = requests.get(f"{URL}/stats").json()
print(f"\n📊 /stats :")
print(f"  Total analyses : {s['total_analyses']}")
print(f"  Par verdict    : {s['by_verdict']}")
print(f"  Score moyen    : {s['avg_score']}")
print(f"  Total IoCs     : {s['total_iocs']}")

In [ ]:
!git clone https://github.com/ChanezNouioua-gif/phishing_dasec.git
!cd phishing_dasec

Cloning into 'phishing_dasec'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 391.75 KiB | 1.32 MiB/s, done.


In [ ]:
%cd /content/phishing_dasec
!ls

/content/phishing_dasec
api_v2.py  phishing_detection.ipynb


In [ ]:
import nbformat

file = "phishing_detection.ipynb"

with open(file, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# supprimer metadata cassées (root + cellules)
nb.metadata.pop("widgets", None)

for cell in nb.cells:
    cell.metadata.pop("widgets", None)

fixed_file = "phishing_detection_fixed.ipynb"

with open(fixed_file, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("✔ Notebook réparé :", fixed_file)

✔ Notebook réparé : phishing_detection_fixed.ipynb


In [ ]:
!ls

api_v2.py  phishing_detection_fixed.ipynb  phishing_detection.ipynb


In [ ]:

# ══════════════════════════════════════════════════════════════
# DIAGNOSTIC — Cellule H
# ══════════════════════════════════════════════════════════════
import sqlite3

# 1. Vérifier le contenu réel de la DB
con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("SELECT COUNT(*) FROM analyses")
print("Lignes dans 'analyses' :", cur.fetchone()[0])
cur.execute("SELECT analysis_id, verdict, score_global FROM analyses")
for row in cur.fetchall():
    print(row)
con.close()

# 2. Vérifier que le endpoint patché est bien celui qui est actif
for route in api_v2.app.routes:
    if hasattr(route, "path") and route.path == "/analyze/text":
        print("\nEndpoint actif pour /analyze/text :", route.endpoint.__name__)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSTIC 2 — appeler save_analysis directement, voir l'erreur réelle
# ══════════════════════════════════════════════════════════════
import requests

r = requests.post(f"{PUBLIC_URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
result = r.json()
print("Clés de result :", list(result.keys()))
print("analysis_id présent ?", "analysis_id" in result)

# Appel direct, hors FastAPI, pour voir le traceback complet
try:
    aid = save_analysis(
        result,
        mode="text",
        sender="security@paypa1.com",
        subject="Security Alert",
        alert_sent=False,
    )
    print("✅ save_analysis OK, id =", aid)
except Exception as e:
    import traceback
    print("❌ ERREUR dans save_analysis :")
    traceback.print_exc()

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIX — Patch complet du endpoint (route.endpoint + route.dependant.call)
# ══════════════════════════════════════════════════════════════

from api_v2 import TextRequest

# Wrapper text — identique à avant
async def _patched_text(req: TextRequest):
    result = await _orig_text_fn(req=req)
    aid    = save_analysis(
        result,
        mode="text",
        sender=req.sender or "",
        subject=req.subject or "",
        alert_sent=False,
    )
    result["analysis_id"] = aid
    return result

# Wrapper eml — identique à avant
async def _patched_eml(file: UploadFile = File(...)):
    result = await _orig_eml_fn(file=file)
    meta   = result.get("email_meta", {})
    aid    = save_analysis(
        result,
        mode="eml",
        sender=meta.get("from", ""),
        subject=meta.get("subject", ""),
        alert_sent=result.get("alert_sent", False),
    )
    result["analysis_id"] = aid
    return result

# Patch RÉEL — remplace endpoint ET dependant.call pour chaque route
for route in api_v2.app.routes:
    if not hasattr(route, "path"):
        continue
    if route.path == "/analyze/text":
        route.endpoint = _patched_text
        route.dependant.call = _patched_text   # ← le vrai fix
        print("✅ /analyze/text patché (endpoint + dependant.call)")
    elif route.path == "/analyze/eml":
        route.endpoint = _patched_eml
        route.dependant.call = _patched_eml    # ← le vrai fix
        print("✅ /analyze/eml patché (endpoint + dependant.call)")

In [ ]:
import requests, time

r = requests.post(f"{PUBLIC_URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
result = r.json()
print("Clés :", list(result.keys()))
print("analysis_id :", result.get("analysis_id", "MANQUANT"))

time.sleep(0.5)
h = requests.get(f"{PUBLIC_URL}/history?limit=5").json()
print(f"\n/history — {len(h)} entrées")
for row in h:
    print(f"  [{row['verdict']}] {row['score_global']} | {row['subject']}")

s = requests.get(f"{PUBLIC_URL}/stats").json()
print(f"\n/stats : {s}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIX — installer chromadb manquant, puis relancer le serveur
# ══════════════════════════════════════════════════════════════
!pip install chromadb sentence-transformers -q

import subprocess, time
from pyngrok import ngrok

# Tuer le tunnel et libérer le port avant de relancer
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)
print("✅ chromadb installé, port libéré — relance la Cellule F maintenant")

In [ ]:

# ══════════════════════════════════════════════════════════════
# CELLULE F — FASTAPI v4.0 CONSOLIDÉE (Sigma + ThreatIntel + SQLite natifs)
# Remplace : ancienne Cellule F + Cellule H entière
# Prérequis : CELLULE KILL exécutée juste avant
# ══════════════════════════════════════════════════════════════

!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart pyyaml -q
!pip install python-whois python-magic oletools pytesseract Pillow -q
!pip install python-Levenshtein slack-sdk google-generativeai groq -q
!apt-get install -y tesseract-ocr libmagic1 -q

import os, sys, time, asyncio, threading, subprocess
from google.colab import userdata
from pyngrok import ngrok

def _get_secret(name):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
        return val or ""
    except Exception:
        return os.getenv(name, "")

NGROK_TOKEN    = _get_secret("NGROK_TOKEN")
GEMINI_API_KEY = _get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = _get_secret("GROQ_API_KEY")
SLACK_TOKEN    = _get_secret("SLACK_TOKEN")
VT_API_KEY     = _get_secret("VT_API_KEY")
ABUSEIPDB_KEY  = _get_secret("ABUSEIPDB_KEY")

if not NGROK_TOKEN:
    raise ValueError("❌ NGROK_TOKEN manquant dans les Secrets Colab")

try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)
print("✅ Dépendances installées, port libéré")

# ══════════════════════════════════════════════════════════════
api_code = r'''
import re, os, sys, json, pickle, hashlib, time, uuid, yaml, sqlite3
import torch
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from email import policy
from email.parser import BytesParser
from io import BytesIO
from typing import Optional
from datetime import datetime, timezone

from fastapi import FastAPI, UploadFile, File, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import pytesseract
from PIL import Image
import Levenshtein
import whois

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
BASE_DIR        = "/content/drive/MyDrive/phishing_dasec"
VECTORSTORE_DIR = f"{BASE_DIR}/vectorstore"
IOC_CACHE_DIR   = f"{BASE_DIR}/ioc_cache"
DB_PATH         = f"{BASE_DIR}/dasec.db"
SLACK_TOKEN     = os.getenv("SLACK_TOKEN", "")
SLACK_CHANNEL   = "#soc-alerts"
SOC_THRESHOLD   = 0.65

KNOWN_BRANDS = [
    "paypal","microsoft","google","apple","amazon","netflix",
    "facebook","instagram","twitter","linkedin","dhl","fedex",
    "bnpparibas","societegenerale","creditagricole","orange","sfr",
    "docusign","dropbox","wellsfargo","chase","bankofamerica",
]
SUSPICIOUS_TLDS = {
    ".tk",".xyz",".click",".top",".link",".info",
    ".biz",".ru",".cn",".pw",".ga",".ml",".cf",".gq",
}

# ══════════════════════════════════════════════════════════════
# SQLITE — INITIALISATION + CRUD (natif)
# ══════════════════════════════════════════════════════════════
def init_db():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS analyses (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id     TEXT    UNIQUE NOT NULL,
            analyzed_at     TEXT    NOT NULL,
            mode            TEXT    NOT NULL,
            score_global    REAL    NOT NULL,
            verdict         TEXT    NOT NULL,
            confidence      TEXT,
            sender          TEXT,
            subject         TEXT,
            technique_attck TEXT,
            campaign        TEXT,
            vecteur_attaque TEXT,
            sigma_count     INTEGER DEFAULT 0,
            alert_sent      INTEGER DEFAULT 0,
            report_json     TEXT    NOT NULL
        );
        CREATE TABLE IF NOT EXISTS iocs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id TEXT NOT NULL,
            ioc_type    TEXT NOT NULL,
            ioc_value   TEXT NOT NULL,
            source      TEXT,
            detected_at TEXT NOT NULL,
            FOREIGN KEY (analysis_id) REFERENCES analyses(analysis_id)
        );
        CREATE INDEX IF NOT EXISTS idx_analyses_verdict ON analyses(verdict);
        CREATE INDEX IF NOT EXISTS idx_analyses_score   ON analyses(score_global);
        CREATE INDEX IF NOT EXISTS idx_analyses_date    ON analyses(analyzed_at);
        CREATE INDEX IF NOT EXISTS idx_iocs_analysis_id ON iocs(analysis_id);
        CREATE INDEX IF NOT EXISTS idx_iocs_type        ON iocs(ioc_type);
    """)
    con.commit()
    con.close()

init_db()

def save_analysis(result: dict, mode: str = "text", sender: str = "",
                  subject: str = "", alert_sent: bool = False) -> str:
    analysis_id = str(uuid.uuid4())
    now = datetime.now(timezone.utc).isoformat()
    report  = result.get("report", {})
    sigma_r = result.get("sigma_rules", {})
    iocs    = report.get("iocs", {})
    ti      = result.get("threat_intel", result.get("modules", {}).get("threat_intel", {}))

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("""
        INSERT OR IGNORE INTO analyses
          (analysis_id, analyzed_at, mode, score_global, verdict, confidence,
           sender, subject, technique_attck, campaign, vecteur_attaque,
           sigma_count, alert_sent, report_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        analysis_id, now, mode, result.get("score_global", 0.0),
        report.get("verdict", result.get("verdict", "UNKNOWN")),
        report.get("confiance", ""), sender, subject,
        report.get("technique_attck", ""), report.get("campagne_probable", ""),
        report.get("vecteur_attaque", ""), sigma_r.get("rules_count", 0),
        int(alert_sent), json.dumps(result, ensure_ascii=False),
    ))

    rows = []
    for url in iocs.get("urls_suspectes", []):
        rows.append((analysis_id, "url", url, "pipeline", now))
    for ip in iocs.get("ips_malveillantes", []):
        rows.append((analysis_id, "ip", ip, "AbuseIPDB", now))
    for h in iocs.get("hashes", []):
        rows.append((analysis_id, "hash", h, "VirusTotal", now))
    for ioc_str in ti.get("iocs_detected", []):
        src = "VT" if "VT" in ioc_str else ("AbuseIPDB" if "AbuseIPDB" in ioc_str else "TI")
        rows.append((analysis_id, "ioc_external", ioc_str[:200], src, now))
    if rows:
        cur.executemany(
            "INSERT INTO iocs (analysis_id, ioc_type, ioc_value, source, detected_at) VALUES (?,?,?,?,?)",
            rows
        )
    con.commit()
    con.close()
    return analysis_id

def get_recent_analyses(limit: int = 20, verdict_filter: str = None) -> list:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    if verdict_filter:
        cur.execute("""SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                       sender, subject, technique_attck, campaign, sigma_count, alert_sent
                       FROM analyses WHERE verdict = ? ORDER BY analyzed_at DESC LIMIT ?""",
                   (verdict_filter.upper(), limit))
    else:
        cur.execute("""SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                       sender, subject, technique_attck, campaign, sigma_count, alert_sent
                       FROM analyses ORDER BY analyzed_at DESC LIMIT ?""", (limit,))
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows

def get_analysis_by_id(analysis_id: str):
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM analyses WHERE analysis_id = ?", (analysis_id,))
    row = cur.fetchone()
    con.close()
    if not row:
        return None
    d = dict(row)
    try:
        d["report"] = json.loads(d.pop("report_json"))
    except Exception:
        d["report"] = {}
    return d

def get_iocs_by_analysis(analysis_id: str) -> list:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("""SELECT ioc_type, ioc_value, source, detected_at FROM iocs
                   WHERE analysis_id = ? ORDER BY ioc_type""", (analysis_id,))
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows

def get_stats() -> dict:
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("SELECT COUNT(*) FROM analyses")
    total = cur.fetchone()[0]
    cur.execute("SELECT verdict, COUNT(*) FROM analyses GROUP BY verdict")
    by_verdict = dict(cur.fetchall())
    cur.execute("SELECT AVG(score_global) FROM analyses")
    avg_score = round(cur.fetchone()[0] or 0.0, 4)
    cur.execute("SELECT COUNT(*) FROM iocs")
    total_iocs = cur.fetchone()[0]
    cur.execute("""SELECT technique_attck, COUNT(*) c FROM analyses
                   WHERE technique_attck != '' GROUP BY technique_attck
                   ORDER BY c DESC LIMIT 5""")
    top_techniques = cur.fetchall()
    cur.execute("""SELECT campaign, COUNT(*) c FROM analyses
                   WHERE campaign != '' GROUP BY campaign
                   ORDER BY c DESC LIMIT 5""")
    top_campaigns = cur.fetchall()
    con.close()
    return {
        "total_analyses": total, "by_verdict": by_verdict, "avg_score": avg_score,
        "total_iocs": total_iocs,
        "top_techniques": [{"technique": t, "count": c} for t, c in top_techniques],
        "top_campaigns": [{"campaign": c, "count": n} for c, n in top_campaigns],
    }

# ══════════════════════════════════════════════════════════════
# CHARGEMENT MODÈLES
# ══════════════════════════════════════════════════════════════
print("⏳ Chargement des modèles...")

with open(f"{BASE_DIR}/models/tfidf_lr_pipeline.pkl", "rb") as f:
    artifacts = pickle.load(f)
tfidf    = artifacts["tfidf"]
lr_model = artifacts["lr_model"]

BERT_PATH  = f"{BASE_DIR}/models/distilbert-5000/final"
tokenizer  = DistilBertTokenizerFast.from_pretrained(BERT_PATH)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_PATH)
bert_model.eval()

embedder      = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

def embed_fn(texts):
    return embedder.encode(texts).tolist()

def _get_col(name):
    try:
        return chroma_client.get_collection(name)
    except Exception:
        return None

col_techniques = _get_col("mitre_techniques")
col_apt        = _get_col("mitre_apt_groups")
col_ti         = _get_col("threat_intel")
col_live       = _get_col("live_ioc_feeds")
RAG_OK         = all([col_techniques, col_apt, col_ti])

import json as _json
def _load_set(path):
    try:
        with open(path) as f:
            return set(_json.load(f))
    except Exception:
        return set()

urlhaus_set   = _load_set(f"{IOC_CACHE_DIR}/urlhaus.json")
openphish_set = _load_set(f"{IOC_CACHE_DIR}/openphish.json")

llm_client  = None
llm_backend = None
GROQ_KEY   = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY = os.getenv("GEMINI_API_KEY", "")

if GROQ_KEY:
    try:
        from groq import Groq
        llm_client  = Groq(api_key=GROQ_KEY)
        llm_backend = "groq"
        print("✅ LLM : Groq (llama-3.1-8b-instant)")
    except Exception as e:
        print(f"⚠️ Groq : {e}")

if not llm_client and GEMINI_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_KEY)
        llm_client  = genai.GenerativeModel("gemini-2.5-flash")
        llm_backend = "gemini"
        print("✅ LLM : Gemini (gemini-2.5-flash)")
    except Exception as e:
        print(f"⚠️ Gemini : {e}")

if not llm_client:
    print("⚠️ Aucun LLM — mode rapport structuré")

print(f"✅ Chargé | RAG={'OK' if RAG_OK else 'KO'} | LLM={llm_backend or 'none'} "
      f"| URLHaus={len(urlhaus_set):,} | OpenPhish={len(openphish_set):,} | DB={DB_PATH}")

# ══════════════════════════════════════════════════════════════
# THREAT INTEL — AbuseIPDB + VirusTotal
# ══════════════════════════════════════════════════════════════
VT_API_KEY    = os.getenv("VT_API_KEY", "")
ABUSEIPDB_KEY = os.getenv("ABUSEIPDB_KEY", "")
_ti_cache     = {}
_ti_executor  = ThreadPoolExecutor(max_workers=4)
VT_BASE_URL    = "https://www.virustotal.com/api/v3"
ABUSE_BASE_URL = "https://api.abuseipdb.com/api/v2/check"

def _vt_check_hash(sha256: str, timeout: int = 5) -> dict:
    cache_key = f"vt_hash:{sha256}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        headers = {"x-apikey": VT_API_KEY}
        r = _rq.get(f"{VT_BASE_URL}/files/{sha256}", headers=headers, timeout=timeout)
        if r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt",
                      "malicious_count": 0, "total_engines": 0}
        elif r.status_code == 200:
            data  = r.json()["data"]["attributes"]
            stats = data.get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0,
                      "names": data.get("names", [])[:3]}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _vt_check_url(target_url: str, timeout: int = 5) -> dict:
    cache_key = f"vt_url:{target_url}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq, base64
        url_id = base64.urlsafe_b64encode(target_url.encode()).decode().strip("=")
        headers = {"x-apikey": VT_API_KEY}
        r = _rq.get(f"{VT_BASE_URL}/urls/{url_id}", headers=headers, timeout=timeout)
        if r.status_code == 200:
            stats = r.json()["data"]["attributes"].get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0}
        elif r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt"}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _abuseipdb_check(ip: str, timeout: int = 5) -> dict:
    cache_key = f"abuse:{ip}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not ABUSEIPDB_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        headers = {"Key": ABUSEIPDB_KEY, "Accept": "application/json"}
        params  = {"ipAddress": ip, "maxAgeInDays": 90}
        r = _rq.get(ABUSE_BASE_URL, headers=headers, params=params, timeout=timeout)
        if r.status_code == 200:
            data  = r.json()["data"]
            score = data.get("abuseConfidenceScore", 0)
            result = {"checked": True, "malicious": score >= 50, "abuse_score": score,
                      "total_reports": data.get("totalReports", 0),
                      "country": data.get("countryCode", ""), "isp": data.get("isp", ""),
                      "is_tor": data.get("isTor", False)}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def threat_intel_check(urls=None, ips=None, hashes=None) -> dict:
    urls, ips, hashes = urls or [], ips or [], hashes or []
    iocs_detected, details = [], {"urls": [], "ips": [], "hashes": []}
    apis_used = {"virustotal": bool(VT_API_KEY), "abuseipdb": bool(ABUSEIPDB_KEY)}
    score = 0.0

    def _run(fn, arg, timeout=6):
        try:
            return _ti_executor.submit(fn, arg).result(timeout=timeout)
        except Exception as e:
            return {"checked": False, "malicious": False, "reason": str(e)[:80]}

    for h in hashes[:2]:
        r = _run(_vt_check_hash, h)
        details["hashes"].append({"hash": h, **r})
        if r.get("malicious"):
            score += 0.95
            iocs_detected.append(f"Hash malveillant (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {h[:16]}...")

    for u in urls[:3]:
        r = _run(_vt_check_url, u)
        details["urls"].append({"url": u, **r})
        if r.get("malicious"):
            score += 0.85
            iocs_detected.append(f"URL malveillante (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {u[:50]}")

    for ip in ips[:3]:
        r = _run(_abuseipdb_check, ip)
        details["ips"].append({"ip": ip, **r})
        if r.get("malicious"):
            score += 0.70
            iocs_detected.append(f"IP malveillante (AbuseIPDB {r.get('abuse_score',0)}%) : {ip}")

    return {"score": round(min(score, 1.0), 4), "iocs_detected": iocs_detected,
            "details": details, "apis_used": apis_used}

# ══════════════════════════════════════════════════════════════
# UTILITAIRES
# ══════════════════════════════════════════════════════════════
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_urls(text: str) -> list:
    return list(set(re.findall(r"https?://[^\s<>\"\']+|www\.[^\s<>\"\']+", text)))

def get_domain(url: str) -> str:
    m = re.search(r"(?:https?://)?(?:www\.)?([^/\s]+)", url)
    return m.group(1).lower() if m else ""

# ══════════════════════════════════════════════════════════════
# MODULE 1 — PARSER .EML
# ══════════════════════════════════════════════════════════════
def parse_eml(raw_bytes: bytes) -> dict:
    msg = BytesParser(policy=policy.default).parsebytes(raw_bytes)
    body_text, body_html = "", ""
    images, attachments  = [], []
    for part in msg.walk():
        ct = part.get_content_type()
        cd = str(part.get("Content-Disposition", ""))
        if ct == "text/plain" and "attachment" not in cd:
            body_text += part.get_content() or ""
        elif ct == "text/html" and "attachment" not in cd:
            body_html += str(part.get_content() or "")
        elif ct.startswith("image/"):
            images.append({"content_type": ct, "data": part.get_payload(decode=True),
                           "name": part.get_filename("image")})
        elif "attachment" in cd or part.get_filename():
            payload = part.get_payload(decode=True)
            if payload:
                attachments.append({
                    "filename": part.get_filename("unknown"), "content_type": ct,
                    "data": payload, "md5": hashlib.md5(payload).hexdigest(),
                    "sha256": hashlib.sha256(payload).hexdigest(), "size": len(payload),
                })
    body_html_clean = re.sub(r"<[^>]+>", " ", body_html)
    full_text = f"{body_text} {body_html_clean}".strip()
    return {
        "headers": dict(msg.items()), "from": msg.get("From", ""),
        "reply_to": msg.get("Reply-To", ""), "return_path": msg.get("Return-Path", ""),
        "subject": msg.get("Subject", ""), "date": msg.get("Date", ""),
        "message_id": msg.get("Message-ID", ""), "x_mailer": msg.get("X-Mailer", ""),
        "body_text": body_text, "body_html": body_html, "full_text": full_text,
        "images": images, "attachments": attachments,
        "urls_in_html": extract_urls(body_html), "urls_in_text": extract_urls(body_text),
    }

# ══════════════════════════════════════════════════════════════
# MODULE 2 — HEADERS SMTP
# ══════════════════════════════════════════════════════════════
def analyze_headers(parsed: dict) -> dict:
    headers, findings, score = parsed["headers"], [], 0.0
    auth = str(headers.get("Authentication-Results", "")).lower()
    spf  = str(headers.get("Received-SPF", "")).lower()
    dkim = "dkim-signature" in {k.lower() for k in headers}
    spf_pass, dkim_pass, dmarc_pass = ("pass" in spf or "pass" in auth), (dkim and "dkim=pass" in auth), ("dmarc=pass" in auth)

    if not spf_pass: findings.append("SPF absent ou échoué"); score += 0.20
    if not dkim_pass: findings.append("DKIM absent ou échoué"); score += 0.20
    if not dmarc_pass: findings.append("DMARC absent ou échoué"); score += 0.10

    from_addr, reply_to = parsed["from"], parsed["reply_to"]
    if reply_to and from_addr and reply_to != from_addr:
        findings.append(f"Reply-To ≠ From : {reply_to}"); score += 0.20

    return_path = parsed["return_path"]
    if return_path and from_addr:
        fd, rd = get_domain(from_addr), get_domain(return_path)
        if fd and rd and fd != rd:
            findings.append(f"Return-Path ({rd}) ≠ From ({fd})"); score += 0.15

    mailer = parsed["x_mailer"].lower()
    for sm in ["mailchimp","sendgrid","phpmailer","massmailer","bulk"]:
        if sm in mailer:
            findings.append(f"X-Mailer suspect : {parsed['x_mailer']}"); score += 0.10; break

    msg_id = parsed["message_id"]
    if not msg_id or not re.match(r"<.+@.+>", msg_id):
        findings.append("Message-ID absent ou invalide"); score += 0.10

    received  = str(headers.get("Received", ""))
    ips_found = list(set(re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", received)))

    return {"score": round(min(score, 1.0), 3), "spf_pass": spf_pass, "dkim_pass": dkim_pass,
            "dmarc_pass": dmarc_pass, "findings": findings, "ips_found": ips_found[:5]}

# ══════════════════════════════════════════════════════════════
# MODULE 3 — URLs
# ══════════════════════════════════════════════════════════════
_whois_executor = ThreadPoolExecutor(max_workers=2)

def _whois_safe(domain: str) -> dict:
    try:
        return _whois_executor.submit(whois.whois, domain).result(timeout=3)
    except Exception:
        return {}

def analyze_urls(urls: list) -> dict:
    findings, suspicious, score = [], [], 0.0
    for url in urls[:10]:
        domain = get_domain(url)
        if not domain: continue

        for tld in SUSPICIOUS_TLDS:
            if domain.endswith(tld):
                findings.append(f"TLD suspect : {domain}"); score += 0.15; suspicious.append(url); break

        domain_base = domain.split(".")[0]
        for brand in KNOWN_BRANDS:
            if 0 < Levenshtein.distance(domain_base, brand) <= 2:
                findings.append(f"Typosquatting : {domain} ≈ {brand}"); score += 0.25; suspicious.append(url); break

        if len(domain) > 40: findings.append(f"Domaine trop long : {domain}"); score += 0.10
        if domain.count("-") >= 3: findings.append(f"Beaucoup de tirets : {domain}"); score += 0.10
        if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
            findings.append(f"IP directe : {url[:60]}"); score += 0.30; suspicious.append(url)

        url_norm = url.lower()
        if url_norm in urlhaus_set or url in urlhaus_set:
            findings.append(f"URLHaus blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)
        if url_norm in openphish_set or url in openphish_set:
            findings.append(f"OpenPhish blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)

        try:
            w = _whois_safe(domain)
            if w and w.get("creation_date"):
                creation = w["creation_date"]
                if isinstance(creation, list): creation = creation[0]
                if creation and (datetime.now() - creation).days < 30:
                    findings.append(f"Domaine très récent : {domain}"); score += 0.30; suspicious.append(url)
        except Exception:
            pass

    return {"score": round(min(score, 1.0), 3), "total_urls": len(urls),
            "suspicious": list(set(suspicious))[:5], "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 4 — OCR IMAGES
# ══════════════════════════════════════════════════════════════
def analyze_images(images: list) -> dict:
    ocr_texts, findings, score = [], [], 0.0
    if len(images) > 3:
        findings.append(f"{len(images)} images — contournement filtre"); score += 0.15
    for img_data in images[:5]:
        try:
            raw = img_data["data"]
            if not raw: continue
            img = Image.open(BytesIO(raw))
            w, h = img.size
            if w <= 3 and h <= 3:
                findings.append("Pixel tracking"); score += 0.20; continue
            text = pytesseract.image_to_string(img, lang="eng+fra")
            if text.strip():
                ocr_texts.append(text.strip())
                kw = [k for k in ["verify","account","password","click","urgent",
                                  "suspended","login","security","bank","credit card"]
                      if k in text.lower()]
                if kw: findings.append(f"Mots-clés phishing image : {kw}"); score += 0.20
        except Exception:
            pass
    return {"score": round(min(score, 1.0), 3), "ocr_text": " ".join(ocr_texts)[:500],
            "images_count": len(images), "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 5 — PIÈCES JOINTES
# ══════════════════════════════════════════════════════════════
def analyze_attachments(attachments: list) -> dict:
    findings, suspicious_hashes, score = [], [], 0.0
    DANGEROUS_EXT = {".exe",".bat",".cmd",".ps1",".vbs",".js",".jar",".scr",".pif",".com",".msi"}
    DOUBLE_EXT    = re.compile(r"\.(pdf|docx?|xlsx?|zip)\.(exe|bat|cmd|scr)$", re.I)
    for att in attachments:
        fname, ct = att["filename"].lower(), att["content_type"]
        for ext in DANGEROUS_EXT:
            if fname.endswith(ext):
                findings.append(f"Extension dangereuse : {att['filename']}"); score += 0.40; break
        if DOUBLE_EXT.search(fname):
            findings.append(f"Double extension : {att['filename']}"); score += 0.50
        try:
            import magic
            real_mime = magic.from_buffer(att["data"][:1024], mime=True)
            if real_mime and real_mime not in ct:
                findings.append(f"MIME réel ({real_mime}) ≠ déclaré ({ct})"); score += 0.35
        except Exception:
            pass
        if fname.endswith((".doc",".docx",".xls",".xlsx",".ppt",".pptx")):
            try:
                from oletools.olevba import VBA_Parser
                vba = VBA_Parser(att["filename"], data=att["data"])
                if vba.detect_vba_macros():
                    findings.append(f"Macros VBA : {att['filename']}"); score += 0.45
            except Exception:
                pass
        suspicious_hashes.append({"filename": att["filename"], "md5": att["md5"],
                                  "sha256": att["sha256"], "size": att["size"]})
    return {"score": round(min(score, 1.0), 3), "attachments_count": len(attachments),
            "suspicious_hashes": suspicious_hashes, "findings": findings}

# ══════════════════════════════════════════════════════════════
# MODULE 6 — NLP
# ══════════════════════════════════════════════════════════════
def analyze_text(text: str) -> dict:
    text_clean = clean_text(text)
    if not text_clean.strip():
        return {"bert_score":0.0,"lr_score":0.0,"score":0.0,"keywords":[]}
    inputs = tokenizer(text_clean, truncation=True, padding=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    bert_score = float(torch.softmax(logits, dim=1).numpy()[0][1])
    lr_score   = float(lr_model.predict_proba(tfidf.transform([text_clean]))[0][1])
    keywords = [kw for kw in [
        "verify","account","password","click here","urgent","confirm","suspended","login",
        "update","security","limited","expire","winner","prize","claim","free","bank",
        "credit card","wire transfer",
    ] if kw in text.lower()]
    return {"bert_score": round(bert_score, 4), "lr_score": round(lr_score, 4),
            "score": round(bert_score * 0.6 + lr_score * 0.4, 4), "keywords": keywords}

# ══════════════════════════════════════════════════════════════
# MODULE 7 — RAG
# ══════════════════════════════════════════════════════════════
def rag_search(text: str) -> dict:
    if not RAG_OK:
        return {"techniques":[],"apt_groups":[],"campaigns":[],"live_iocs":[]}
    q_emb = embed_fn([text[:500]])
    def _query(col, n):
        if not col: return None
        try: return col.query(query_embeddings=q_emb, n_results=n)
        except Exception: return None
    def _parse(res, fields):
        if not res or not res["ids"][0]: return []
        out = []
        for i in range(len(res["ids"][0])):
            item = {"similarity": round(1 - res["distances"][0][i], 3)}
            for f in fields: item[f] = res["metadatas"][0][i].get(f, "")
            out.append(item)
        return out
    return {
        "techniques": _parse(_query(col_techniques, 3), ["tech_id","name","tactics"]),
        "apt_groups": _parse(_query(col_apt, 2),        ["group_id","name","aliases"]),
        "campaigns":  _parse(_query(col_ti, 2),         ["campaign","actor","technique","severity"]),
        "live_iocs":  _parse(_query(col_live, 2),       ["url","attack_type","brand_target"]) if col_live else [],
    }

# ══════════════════════════════════════════════════════════════
# MODULE 8 — RAPPORT LLM
# ══════════════════════════════════════════════════════════════
def _call_llm(prompt: str) -> str:
    if llm_backend == "groq":
        resp = llm_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role":"user","content":prompt}],
            temperature=0.1, max_tokens=800,
        )
        return resp.choices[0].message.content
    elif llm_backend == "gemini":
        return llm_client.generate_content(prompt).text
    return ""

def generate_report(parsed, header_r, url_r, image_r, attach_r,
                    text_r, rag_r, ti_r, score_global) -> dict:
    top_tech     = rag_r["techniques"][0] if rag_r["techniques"] else {}
    top_campaign = rag_r["campaigns"][0]  if rag_r["campaigns"]  else {}
    top_apt      = rag_r["apt_groups"][0] if rag_r["apt_groups"] else {}
    all_findings = (header_r["findings"] + url_r["findings"] + image_r["findings"] + attach_r["findings"])
    iocs_detected = ti_r.get("iocs_detected", [])

    if not llm_client:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {
            "verdict": verdict, "confiance": "haute" if score_global >= 0.75 else "moyenne",
            "score_final": score_global,
            "campagne_probable": top_campaign.get("campaign","Inconnue"),
            "technique_attck": f"{top_tech.get('tech_id','')} — {top_tech.get('name','N/A')}",
            "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
            "indicateurs_cles": all_findings[:5],
            "explication": f"Score {score_global:.2f}. {len(all_findings)} indicateurs. LLM indisponible.",
            "recommandation": "Bloquer et quarantaine." if score_global >= 0.65 else "Vérification manuelle.",
            "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                     "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
            "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else "",
        }

    urls_json   = _json.dumps(url_r["suspicious"])
    ips_json    = _json.dumps(header_r["ips_found"])
    hashes_json = _json.dumps([h["sha256"] for h in attach_r["suspicious_hashes"]])
    ti_block    = f"IoCs confirmés externes : {iocs_detected}" if iocs_detected else "Aucun IoC externe confirmé"

    prompt = f"""Tu es un analyste SOC expert en détection de phishing.

=== EMAIL ===
De      : {parsed.get("from","")}
Sujet   : {parsed.get("subject","")}
Extrait : {parsed.get("full_text","")[:300]}

=== SCORES ===
Headers : {header_r["score"]} | SPF={header_r["spf_pass"]} DKIM={header_r["dkim_pass"]} DMARC={header_r["dmarc_pass"]}
URLs    : {url_r["score"]} ({url_r["total_urls"]} URLs, {len(url_r["suspicious"])} suspectes)
OCR     : {image_r["score"]} | Attachments : {attach_r["score"]}
NLP     : BERT={text_r["bert_score"]} LR={text_r["lr_score"]}
TI      : {ti_r.get("score",0)} | Global : {score_global}

=== INDICATEURS ===
{chr(10).join(all_findings) if all_findings else "Aucun"}

=== THREAT INTEL EXTERNE (VirusTotal / AbuseIPDB) ===
{ti_block}

=== RAG ===
MITRE   : {top_tech.get("tech_id","")} {top_tech.get("name","")} (sim={top_tech.get("similarity",0)})
APT     : {top_apt.get("name","")} (sim={top_apt.get("similarity",0)})
Campagne: {top_campaign.get("campaign","")} — {top_campaign.get("actor","")} (sim={top_campaign.get("similarity",0)})

Réponds STRICTEMENT avec un JSON valide (syntaxe JSON pure, pas Python), sans texte avant/après, sans markdown ni backticks. Respecte exactement ce format :

{{"verdict": "PHISHING", "confiance": "haute", "score_final": {score_global}, "campagne_probable": "nom ou Aucune", "technique_attck": "T#### - nom", "vecteur_attaque": "lien", "indicateurs_cles": ["max 5, mentionne les hits VT/AbuseIPDB si presents"], "explication": "2-3 phrases SOC mentionnant les IoCs externes confirmés si presents", "recommandation": "action", "iocs": {{"urls_suspectes": {urls_json}, "ips_malveillantes": {ips_json}, "hashes": {hashes_json}}}, "sigma_hint": "valeur"}}"""

    try:
        raw = _call_llm(prompt)
        raw = re.sub(r"```json|```", "", raw).strip()
        m   = re.search(r"\{.*\}", raw, re.DOTALL)
        return _json.loads(m.group() if m else raw)
    except Exception as e:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {
            "verdict": verdict, "confiance": "moyenne", "score_final": score_global,
            "explication": f"Erreur LLM : {str(e)[:100]}",
            "campagne_probable": top_campaign.get("campaign","N/A"),
            "technique_attck": f"{top_tech.get('tech_id','')} {top_tech.get('name','')}",
            "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
            "indicateurs_cles": all_findings[:5],
            "recommandation": "Bloquer et quarantaine." if score_global >= 0.65 else "Vérification manuelle.",
            "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                     "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
            "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else "",
        }

# ══════════════════════════════════════════════════════════════
# MODULE 9 — SIGMA RULES
# ══════════════════════════════════════════════════════════════
def _mitre_to_tag(technique_attck: str) -> str:
    m = re.search(r"T(\d{4})(?:\.(\d{3}))?", technique_attck or "")
    if not m: return "attack.t1566"
    base = f"attack.t{m.group(1)}"
    return f"{base}.{m.group(2)}" if m.group(2) else base

def _tactic_tag(tech_id: str) -> str:
    TACTIC_MAP = {"T1566":"attack.initial_access","T1598":"attack.reconnaissance",
                  "T1556":"attack.credential_access","T1539":"attack.credential_access",
                  "T1204":"attack.execution","T1027":"attack.defense_evasion"}
    base = tech_id.split(".")[0] if tech_id else ""
    return TACTIC_MAP.get(base, "attack.initial_access")

def _level_from_score(score: float) -> str:
    if score >= 0.85: return "critical"
    if score >= 0.65: return "high"
    if score >= 0.40: return "medium"
    if score >= 0.20: return "low"
    return "informational"

def generate_sigma_rules(report: dict, score_global: float, email_meta: dict = None) -> dict:
    email_meta = email_meta or {}
    rules = []
    level = _level_from_score(score_global)
    tech_id = (report.get("technique_attck","") or "").split(" ")[0].strip("—-").strip()
    tag_technique, tag_tactic = _mitre_to_tag(tech_id), _tactic_tag(tech_id)
    rule_date = datetime.now().strftime("%Y/%m/%d")

    iocs = report.get("iocs", {})
    urls_susp, ips_mal, hashes_mal = iocs.get("urls_suspectes",[]), iocs.get("ips_malveillantes",[]), iocs.get("hashes",[])
    common_tags = ["attack.phishing", tag_tactic, tag_technique]
    common_refs = [f"https://attack.mitre.org/techniques/{tech_id}"] if tech_id else []

    if urls_susp:
        domains = list(set([re.sub(r"^https?://","",u).split("/")[0] for u in urls_susp]))
        rules.append({"type":"proxy_network","data":{
            "title": f"Accès à un domaine de phishing détecté par DASEC ({report.get('campagne_probable','Inconnue')})",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Connexion vers domaine malveillant. Score : {score_global:.2f}. Vecteur : {report.get('vecteur_attaque','lien')}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"proxy"},
            "detection":{"selection":{"c-uri|contains": domains if len(domains)>1 else domains[0]},"condition":"selection"},
            "falsepositives":["Domaine réutilisé légitimement","Faux positif AV tiers"],"level":level,
        }})

    sender, subject = email_meta.get("from",""), email_meta.get("subject","")
    if sender or subject:
        selection = {}
        if sender:
            selection["sender|contains"] = sender.split("@")[-1] if "@" in sender else sender
        if subject:
            selection["subject|contains"] = subject[:60]
        rules.append({"type":"mail_gateway","data":{
            "title": f"Email de phishing détecté par DASEC — {report.get('campagne_probable','Inconnue')}",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Caractéristiques phishing : {', '.join(report.get('indicateurs_cles',[])[:3]) or 'analyse NLP'}. MITRE : {report.get('technique_attck','N/A')}.",
            "references": common_refs,
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"mail"},
            "detection":{"selection":selection,"condition":"selection"},
            "falsepositives":["Email légitime — vérifier SPF/DKIM/DMARC"],"level":level,
        }})

    if hashes_mal:
        rules.append({"type":"file_hash","data":{
            "title":"Pièce jointe malveillante détectée par DASEC (hash confirmé VirusTotal)",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Hash confirmé malveillant par VirusTotal. Score : {score_global:.2f}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags+["attack.execution"],
            "logsource":{"category":"file_event"},
            "detection":{"selection":{"Hashes|contains":hashes_mal},"condition":"selection"},
            "falsepositives":["Aucun connu"],"level":"critical",
        }})

    yaml_blocks = []
    for r in rules:
        clean = {k:v for k,v in r["data"].items() if v is not None}
        yaml_str = yaml.dump(clean, sort_keys=False, allow_unicode=True, default_flow_style=False, width=100)
        yaml_blocks.append(yaml_str)
        r["yaml"] = yaml_str

    combined = "\n---\n".join(yaml_blocks) if yaml_blocks else "# Aucune règle générée"
    return {"rules_count": len(rules), "rules": [{"type":r["type"],"yaml":r["yaml"]} for r in rules],
            "combined_yaml": combined}

# ══════════════════════════════════════════════════════════════
# MODULE 10 — ALERTE SOC
# ══════════════════════════════════════════════════════════════
def send_soc_alert(report: dict, parsed: dict, score: float) -> bool:
    if not SLACK_TOKEN or score < SOC_THRESHOLD:
        return False
    try:
        from slack_sdk import WebClient
        client = WebClient(token=SLACK_TOKEN)
        verdict = report.get("verdict","?")
        emoji = "🚨" if verdict == "PHISHING" else "⚠️"
        msg = (f"{emoji} *DASEC — {verdict}*\n>Score : {score:.2f} | Confiance : {report.get('confiance','?')}\n"
               f">De : {parsed.get('from','')}\n>Sujet : {parsed.get('subject','')}\n"
               f">Campagne : {report.get('campagne_probable','N/A')}\n>Technique : {report.get('technique_attck','N/A')}\n"
               f">Action : {report.get('recommandation','N/A')}\n>Indicateurs : {', '.join(report.get('indicateurs_cles',[])[:3])}")
        client.chat_postMessage(channel=SLACK_CHANNEL, text=msg)
        return True
    except Exception as e:
        print(f"Slack error: {e}"); return False

# ══════════════════════════════════════════════════════════════
# SCORE GLOBAL
# ══════════════════════════════════════════════════════════════
def compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full") -> float:
    if mode == "text_only":
        return round(text_r["score"]*0.50 + url_r["score"]*0.35 + ti_r.get("score",0.0)*0.10 +
                    (len(url_r["suspicious"])>0)*0.05, 4)
    return round(header_r["score"]*0.25 + text_r["score"]*0.20 + url_r["score"]*0.20 +
                attach_r["score"]*0.10 + ti_r.get("score",0.0)*0.15 + image_r["score"]*0.05 +
                (len(url_r["suspicious"])>0)*0.05, 4)

# ══════════════════════════════════════════════════════════════
# FASTAPI
# ══════════════════════════════════════════════════════════════
app = FastAPI(title="DASEC — Phishing Detection API",
             description="Headers+URLs+OCR+Attachments+NLP+RAG+LLM+ThreatIntel+Sigma+SOC+SQLite", version="4.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class TextRequest(BaseModel):
    text: str
    subject: Optional[str] = ""
    sender: Optional[str] = ""

EMPTY_HEADER = {"score":0.0,"spf_pass":None,"dkim_pass":None,"dmarc_pass":None,"findings":[],"ips_found":[]}
EMPTY_IMAGE  = {"score":0.0,"ocr_text":"","images_count":0,"findings":[]}
EMPTY_ATTACH = {"score":0.0,"attachments_count":0,"suspicious_hashes":[],"findings":[]}

@app.get("/")
def root():
    return {"message":"DASEC v4.0","status":"running","rag":RAG_OK,"llm_backend":llm_backend or "none"}

@app.get("/health")
def health():
    return {
        "status":"ok","models":["distilbert","lr","sentence-transformers"],
        "rag_collections":{
            "mitre_techniques": col_techniques.count() if col_techniques else 0,
            "mitre_apt_groups": col_apt.count() if col_apt else 0,
            "threat_intel": col_ti.count() if col_ti else 0,
            "live_ioc_feeds": col_live.count() if col_live else 0,
        },
        "ioc_feeds":{"urlhaus":len(urlhaus_set),"openphish":len(openphish_set)},
        "threat_intel_apis":{"virustotal":bool(VT_API_KEY),"abuseipdb":bool(ABUSEIPDB_KEY)},
        "llm_backend":llm_backend or "none","soc_alert":bool(SLACK_TOKEN),"soc_threshold":SOC_THRESHOLD,
        "db_path": DB_PATH,
    }

@app.post("/analyze/eml")
async def analyze_eml(file: UploadFile = File(...)):
    if not file.filename.endswith(".eml"):
        raise HTTPException(400, "Fichier .eml requis")
    raw = await file.read()
    parsed = parse_eml(raw)
    all_urls = list(set(parsed["urls_in_html"] + parsed["urls_in_text"]))
    header_r = analyze_headers(parsed)
    url_r    = analyze_urls(all_urls)
    image_r  = analyze_images(parsed["images"])
    attach_r = analyze_attachments(parsed["attachments"])
    full_text = parsed["full_text"] + " " + image_r["ocr_text"]
    text_r = analyze_text(full_text)
    rag_r  = rag_search(full_text)
    ti_r = threat_intel_check(
        urls=url_r["suspicious"][:3], ips=header_r["ips_found"][:3],
        hashes=[a["sha256"] for a in parsed["attachments"]][:2],
    )
    score_global = compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full")
    report = generate_report(parsed, header_r, url_r, image_r, attach_r, text_r, rag_r, ti_r, score_global)
    sigma_r = generate_sigma_rules(report, score_global, {"from":parsed["from"],"subject":parsed["subject"]})
    alert_sent = send_soc_alert(report, parsed, score_global)

    result = {
        "score_global": score_global, "verdict": report.get("verdict","?"),
        "modules":{"headers":header_r,"urls":url_r,"images":image_r,"attachments":attach_r,
                   "nlp":text_r,"rag":rag_r,"threat_intel":ti_r},
        "report": report, "sigma_rules": sigma_r, "alert_sent": alert_sent,
        "email_meta":{"from":parsed["from"],"subject":parsed["subject"],"date":parsed["date"]},
    }
    aid = save_analysis(result, mode="eml", sender=parsed["from"],
                        subject=parsed["subject"], alert_sent=alert_sent)
    result["analysis_id"] = aid
    return result

@app.post("/analyze/text")
async def analyze_text_endpoint(req: TextRequest):
    text_r = analyze_text(req.text)
    url_r  = analyze_urls(extract_urls(req.text))
    rag_r  = rag_search(req.text)
    ti_r   = threat_intel_check(urls=url_r["suspicious"][:2])
    score_global = compute_global_score(EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, ti_r, mode="text_only")
    report = generate_report(
        {"from":req.sender,"subject":req.subject,"full_text":req.text},
        EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, rag_r, ti_r, score_global,
    )
    sigma_r = generate_sigma_rules(report, score_global, {"from":req.sender,"subject":req.subject})

    result = {
        "score_global": score_global, "verdict": report.get("verdict","?"),
        "nlp": text_r, "urls": url_r, "rag": rag_r, "threat_intel": ti_r,
        "report": report, "sigma_rules": sigma_r,
    }
    aid = save_analysis(result, mode="text", sender=req.sender or "",
                        subject=req.subject or "", alert_sent=False)
    result["analysis_id"] = aid
    return result

@app.get("/history")
def history(limit: int = Query(20, ge=1, le=100), verdict: str = Query(None)):
    return get_recent_analyses(limit=limit, verdict_filter=verdict)

@app.get("/history/{analysis_id}")
def history_detail(analysis_id: str):
    row = get_analysis_by_id(analysis_id)
    if not row:
        raise HTTPException(404, f"Analyse {analysis_id} introuvable")
    row["iocs_detail"] = get_iocs_by_analysis(analysis_id)
    return row

@app.get("/stats")
def stats():
    return get_stats()
'''

with open("/content/api_v2.py", "w") as f:
    f.write(api_code)
print("✅ api_v2.py v4.0 écrit (Sigma + ThreatIntel + SQLite natifs)")

# ══════════════════════════════════════════════════════════════
sys.path.insert(0, "/content")
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f"\n🌍 API : {PUBLIC_URL}")
print(f"📖 Docs : {PUBLIC_URL}/docs")
print(f"🔎 Health : {PUBLIC_URL}/health")
print(f"📜 History : {PUBLIC_URL}/history")
print(f"📊 Stats : {PUBLIC_URL}/stats")

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    import uvicorn
    config = uvicorn.Config("api_v2:app", host="0.0.0.0", port=8000, log_level="warning", loop="asyncio")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(4)

print("\n✅ Serveur DASEC v4.0 démarré — tout natif, zéro patch à chaud !")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSTIC — relance avec capture d'erreur visible
# ══════════════════════════════════════════════════════════════
import threading, traceback, time

_server_error = []

def run_server_debug():
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        import uvicorn
        config = uvicorn.Config("api_v2:app", host="0.0.0.0", port=8000,
                                log_level="info", loop="asyncio")  # info au lieu de warning pour + de logs
        server = uvicorn.Server(config)
        loop.run_until_complete(server.serve())
    except Exception as e:
        _server_error.append(traceback.format_exc())
        print("❌ ERREUR DANS LE THREAD SERVEUR :")
        print(traceback.format_exc())

thread = threading.Thread(target=run_server_debug, daemon=True)
thread.start()
time.sleep(6)

if _server_error:
    print("\n🔴 Le serveur a planté, voir le traceback ci-dessus")
else:
    import subprocess
    check = subprocess.run(["fuser", "8000/tcp"], capture_output=True, text=True)
    print("✅ Port 8000 actif :", bool(check.stdout.strip()))

In [ ]:
import requests, json, time

r = requests.post(f"{PUBLIC_URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
print(r.status_code)
print(r.text)

if r.ok:
    result = r.json()
    print("Score :", result["score_global"])
else:
    print("Erreur API")
print("Score :", result["score_global"], "| Verdict:", result["verdict"])
print("analysis_id :", result.get("analysis_id"))

time.sleep(0.5)
h = requests.get(f"{PUBLIC_URL}/history?limit=5").json()
print(f"\n/history — {len(h)} entrées")

s = requests.get(f"{PUBLIC_URL}/stats").json()
print(f"\n/stats : {s}")

In [ ]:
import subprocess
result = subprocess.run(["fuser", "8000/tcp"], capture_output=True, text=True)
print("PID sur port 8000 :", result.stdout.strip() or "AUCUN — rien n'écoute")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE J — LANGGRAPH PLANNER AGENT
# Orchestration conditionnelle : active uniquement les modules pertinents
# Prérequis : api_v2.py chargé en mémoire (Cellule F exécutée — ou import direct des fonctions)
# ══════════════════════════════════════════════════════════════

!pip install langgraph -q

from typing import TypedDict, Optional, List, Literal
from langgraph.graph import StateGraph, START, END

# ══════════════════════════════════════════════════════════════
# PARTIE 1 — État partagé du graphe
# ══════════════════════════════════════════════════════════════

class DASECState(TypedDict, total=False):
    # Entrée
    mode:           str            # "eml" ou "text"
    raw_text:       str
    raw_eml_bytes:  Optional[bytes]
    sender:         str
    subject:        str

    # Parsing
    parsed:         dict
    full_text:      str
    urls:           list
    has_images:     bool
    has_attachments: bool

    # Résultats modules (remplis conditionnellement)
    header_r:       dict
    url_r:           dict
    image_r:         dict
    attach_r:        dict
    text_r:          dict
    rag_r:           dict
    ti_r:            dict

    # Sortie
    score_global:    float
    report:          dict
    sigma_r:         dict
    alert_sent:      bool
    analysis_id:     str
    modules_run:     list           # trace de ce qui a été exécuté (debug/démo)


# ══════════════════════════════════════════════════════════════
# PARTIE 2 — Nœuds du graphe
# Réutilise les fonctions déjà définies dans api_v2 (import direct)
# ══════════════════════════════════════════════════════════════

import api_v2 as dasec

EMPTY_HEADER = dasec.EMPTY_HEADER
EMPTY_IMAGE  = dasec.EMPTY_IMAGE
EMPTY_ATTACH = dasec.EMPTY_ATTACH

def node_parse_input(state: DASECState) -> DASECState:
    """Parse l'email (.eml) ou prépare le texte brut, détecte ce qui est présent."""
    modules_run = ["parse_input"]

    if state["mode"] == "eml":
        parsed = dasec.parse_eml(state["raw_eml_bytes"])
        urls = list(set(parsed["urls_in_html"] + parsed["urls_in_text"]))
        return {
            **state,
            "parsed": parsed,
            "full_text": parsed["full_text"],
            "urls": urls,
            "has_images": len(parsed["images"]) > 0,
            "has_attachments": len(parsed["attachments"]) > 0,
            "sender": parsed["from"],
            "subject": parsed["subject"],
            "modules_run": modules_run,
        }
    else:
        urls = dasec.extract_urls(state["raw_text"])
        return {
            **state,
            "parsed": {"from": state.get("sender",""), "subject": state.get("subject",""),
                      "full_text": state["raw_text"]},
            "full_text": state["raw_text"],
            "urls": urls,
            "has_images": False,
            "has_attachments": False,
            "modules_run": modules_run,
        }


def planner_decide(state: DASECState) -> Literal["headers", "urls", "ocr_images", "attachments", "join_modules"]:
    """
    Routeur conditionnel — décide quel module activer ensuite.
    LangGraph appelle ce routeur plusieurs fois via add_conditional_edges
    avec retour à ce même nœud (pattern fan-out géré par join explicite ci-dessous).
    """
    # Cette fonction sert de point de décision unique ; le fan-out réel
    # est géré par la liste de conditions retournées dans build_graph()
    pass  # remplacé par la logique inline dans build_graph


def node_headers(state: DASECState) -> DASECState:
    header_r = dasec.analyze_headers(state["parsed"])
    return {**state, "header_r": header_r, "modules_run": state["modules_run"] + ["headers"]}

def node_urls(state: DASECState) -> DASECState:
    url_r = dasec.analyze_urls(state["urls"])
    return {**state, "url_r": url_r, "modules_run": state["modules_run"] + ["urls"]}

def node_ocr_images(state: DASECState) -> DASECState:
    image_r = dasec.analyze_images(state["parsed"].get("images", []))
    return {**state, "image_r": image_r, "modules_run": state["modules_run"] + ["ocr_images"]}

def node_attachments(state: DASECState) -> DASECState:
    attach_r = dasec.analyze_attachments(state["parsed"].get("attachments", []))
    return {**state, "attach_r": attach_r, "modules_run": state["modules_run"] + ["attachments"]}

def node_join_modules(state: DASECState) -> DASECState:
    """Remplit les modules non exécutés avec des valeurs vides (état cohérent pour le scoring)."""
    return {
        **state,
        "header_r": state.get("header_r", EMPTY_HEADER),
        "url_r":    state.get("url_r", {"score":0.0,"total_urls":0,"suspicious":[],"findings":[]}),
        "image_r":  state.get("image_r", EMPTY_IMAGE),
        "attach_r": state.get("attach_r", EMPTY_ATTACH),
        "modules_run": state["modules_run"] + ["join_modules"],
    }

def node_nlp_analysis(state: DASECState) -> DASECState:
    ocr_text = state.get("image_r", {}).get("ocr_text", "")
    full_text = state["full_text"] + " " + ocr_text
    text_r = dasec.analyze_text(full_text)
    return {**state, "text_r": text_r, "full_text": full_text,
            "modules_run": state["modules_run"] + ["nlp_analysis"]}

def node_rag_threatintel(state: DASECState) -> DASECState:
    rag_r = dasec.rag_search(state["full_text"])
    ti_r = dasec.threat_intel_check(
        urls=state["url_r"].get("suspicious", [])[:3],
        ips=state["header_r"].get("ips_found", [])[:3],
        hashes=[a["sha256"] for a in state["parsed"].get("attachments", [])][:2],
    )
    return {**state, "rag_r": rag_r, "ti_r": ti_r,
            "modules_run": state["modules_run"] + ["rag_threatintel"]}

def node_score_and_report(state: DASECState) -> DASECState:
    mode_score = "full" if state["mode"] == "eml" else "text_only"
    score_global = dasec.compute_global_score(
        state["header_r"], state["url_r"], state["image_r"], state["attach_r"],
        state["text_r"], state["ti_r"], mode=mode_score
    )
    report = dasec.generate_report(
        state["parsed"], state["header_r"], state["url_r"], state["image_r"],
        state["attach_r"], state["text_r"], state["rag_r"], state["ti_r"], score_global
    )
    return {**state, "score_global": score_global, "report": report,
            "modules_run": state["modules_run"] + ["score_and_report"]}

def node_finalize(state: DASECState) -> DASECState:
    sigma_r = dasec.generate_sigma_rules(
        state["report"], state["score_global"],
        {"from": state.get("sender",""), "subject": state.get("subject","")}
    )
    alert_sent = dasec.send_soc_alert(state["report"], state["parsed"], state["score_global"])

    result = {
        "score_global": state["score_global"], "verdict": state["report"].get("verdict","?"),
        "report": state["report"], "sigma_rules": sigma_r, "alert_sent": alert_sent,
        "modules_executed": state["modules_run"] + ["finalize"],
    }
    aid = dasec.save_analysis(
        result, mode=state["mode"], sender=state.get("sender",""),
        subject=state.get("subject",""), alert_sent=alert_sent
    )
    return {**state, "sigma_r": sigma_r, "alert_sent": alert_sent, "analysis_id": aid,
            "modules_run": state["modules_run"] + ["finalize"]}


# ══════════════════════════════════════════════════════════════
# PARTIE 3 — Construction du graphe avec fan-out conditionnel
# ══════════════════════════════════════════════════════════════

def route_planner(state: DASECState) -> List[str]:
    """
    Retourne la LISTE des modules à activer en parallèle logique
    (LangGraph exécute séquentiellement mais on simule le fan-out
    en chaînant — voir add_conditional_edges avec liste de destinations).
    """
    routes = []
    if state["mode"] == "eml":
        routes.append("headers")
    if state["urls"]:
        routes.append("urls")
    if state.get("has_images"):
        routes.append("ocr_images")
    if state.get("has_attachments"):
        routes.append("attachments")
    if not routes:
        routes.append("join_modules")  # rien à activer, va direct au join
    return routes

def build_graph():
    graph = StateGraph(DASECState)

    graph.add_node("parse_input",      node_parse_input)
    graph.add_node("headers",          node_headers)
    graph.add_node("urls",             node_urls)
    graph.add_node("ocr_images",       node_ocr_images)
    graph.add_node("attachments",      node_attachments)
    graph.add_node("join_modules",     node_join_modules)
    graph.add_node("nlp_analysis",     node_nlp_analysis)
    graph.add_node("rag_threatintel",  node_rag_threatintel)
    graph.add_node("score_and_report", node_score_and_report)
    graph.add_node("finalize",         node_finalize)

    graph.add_edge(START, "parse_input")

    # Fan-out conditionnel — chaque module activé mène à join_modules
    graph.add_conditional_edges(
        "parse_input",
        route_planner,
        {
            "headers":      "headers",
            "urls":         "urls",
            "ocr_images":   "ocr_images",
            "attachments":  "attachments",
            "join_modules": "join_modules",
        }
    )

    # Tous les modules conditionnels convergent vers join_modules
    for module in ["headers", "urls", "ocr_images", "attachments"]:
        graph.add_edge(module, "join_modules")

    graph.add_edge("join_modules",     "nlp_analysis")
    graph.add_edge("nlp_analysis",     "rag_threatintel")
    graph.add_edge("rag_threatintel",  "score_and_report")
    graph.add_edge("score_and_report", "finalize")
    graph.add_edge("finalize",         END)

    return graph.compile()

dasec_graph = build_graph()
print("✅ LangGraph DASEC Planner compilé")

In [ ]:
import json

# Test 1 — email texte simple, pas d'images ni pièces jointes
result1 = dasec_graph.invoke({
    "mode": "text",
    "raw_text": "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender": "security@paypa1.com",
})
print("Modules exécutés :", result1["modules_run"])
print("Score :", result1["score_global"], "| Verdict :", result1["report"]["verdict"])

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE K — RAPPORT PDF AUTOMATIQUE (ReportLab)
# Transforme un résultat d'analyse DASEC en rapport PDF structuré
# Autonome — ne nécessite ni serveur actif ni GPU
# ══════════════════════════════════════════════════════════════

!pip install reportlab -q

from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, HRFlowable, KeepTogether
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from datetime import datetime
import os

# ══════════════════════════════════════════════════════════════
# PARTIE 1 — Styles DASEC (palette cohérente avec le projet)
# ══════════════════════════════════════════════════════════════

VERDICT_COLORS = {
    "PHISHING": colors.HexColor("#A32D2D"),
    "SUSPECT":  colors.HexColor("#854F0B"),
    "LEGITIME": colors.HexColor("#3B6D11"),
    "ERREUR":   colors.HexColor("#5F5E5A"),
}

def _build_styles():
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name="DasecTitle", parent=styles["Title"],
        fontSize=22, textColor=colors.HexColor("#042C53"),
        spaceAfter=4, alignment=TA_LEFT,
    ))
    styles.add(ParagraphStyle(
        name="DasecSubtitle", parent=styles["Normal"],
        fontSize=11, textColor=colors.HexColor("#5F5E5A"),
        spaceAfter=18,
    ))
    styles.add(ParagraphStyle(
        name="SectionHeading", parent=styles["Heading2"],
        fontSize=13, textColor=colors.HexColor("#042C53"),
        spaceBefore=18, spaceAfter=8,
        borderPadding=(0, 0, 4, 0),
    ))
    styles.add(ParagraphStyle(
        name="BodyDasec", parent=styles["Normal"],
        fontSize=10, leading=15, textColor=colors.HexColor("#2C2C2A"),
    ))
    styles.add(ParagraphStyle(
        name="Mono", parent=styles["Normal"],
        fontName="Courier", fontSize=8.5, leading=11,  # 12 → 11, plus compact
        textColor=colors.HexColor("#2C2C2A"),
        backColor=colors.HexColor("#F1EFE8"),
        spaceAfter=0,  # évite l'espacement Paragraph par défaut entre lignes
    ))
    styles.add(ParagraphStyle(
        name="VerdictBig", parent=styles["Normal"],
        fontSize=20, leading=24, alignment=TA_CENTER,
    ))
    styles.add(ParagraphStyle(
        name="ScoreLabel", parent=styles["Normal"],
        fontSize=9, alignment=TA_CENTER, textColor=colors.HexColor("#5F5E5A"),
    ))
    return styles

STYLES = _build_styles()

# ══════════════════════════════════════════════════════════════
# PARTIE 2 — Composants visuels
# ══════════════════════════════════════════════════════════════

def _score_bar_table(score: float, verdict: str):
    """Barre de score visuelle horizontale (0-1) avec couleur selon verdict."""
    width_total = 14 * cm
    filled = max(0.02, min(score, 1.0)) * width_total
    empty  = width_total - filled
    color  = VERDICT_COLORS.get(verdict, colors.grey)

    data = [["", ""]]
    t = Table(data, colWidths=[filled, empty], rowHeights=[0.5*cm])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (0,0), color),
        ("BACKGROUND", (1,0), (1,0), colors.HexColor("#D3D1C7")),
        ("LINEBELOW",  (0,0), (-1,-1), 0, colors.white),
    ]))
    return t

def _verdict_header(report: dict, score_global: float):
    """En-tête verdict avec score visuel."""
    verdict = report.get("verdict", "INCONNU")
    confiance = report.get("confiance", "N/A")
    color = VERDICT_COLORS.get(verdict, colors.grey)

    verdict_style = ParagraphStyle(
        name="VerdictColored", parent=STYLES["VerdictBig"], textColor=color
    )
    elements = [
        Paragraph(f"<b>{verdict}</b>", verdict_style),
        Spacer(1, 6),
        _score_bar_table(score_global, verdict),
        Spacer(1, 4),
        Paragraph(
            f"Score global : <b>{score_global:.2f}</b> / 1.00 &nbsp;&nbsp;|&nbsp;&nbsp; "
            f"Confiance : <b>{confiance}</b>",
            STYLES["ScoreLabel"]
        ),
    ]
    return elements

def _findings_table(rows: list, headers: list, col_widths=None):
    """Table générique avec en-tête stylé DASEC."""
    if not rows:
        return Paragraph("<i>Aucune donnée</i>", STYLES["BodyDasec"])

    data = [headers] + rows
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#042C53")),
        ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
        ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE",   (0,0), (-1,-1), 8.5),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#F1EFE8")]),
        ("GRID", (0,0), (-1,-1), 0.5, colors.HexColor("#D3D1C7")),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("LEFTPADDING", (0,0), (-1,-1), 6),
        ("RIGHTPADDING", (0,0), (-1,-1), 6),
        ("TOPPADDING", (0,0), (-1,-1), 5),
        ("BOTTOMPADDING", (0,0), (-1,-1), 5),
    ]))
    return t

# ══════════════════════════════════════════════════════════════
# PARTIE 3 — Générateur principal
# ══════════════════════════════════════════════════════════════

def generate_pdf_report(result: dict, output_path: str = None) -> str:
    """
    Génère un rapport PDF complet depuis le JSON retourné par
    /analyze/eml ou /analyze/text.
    Retourne le chemin du fichier PDF créé.
    """
    if output_path is None:
        aid = result.get("analysis_id", "unknown")[:8]
        output_path = f"/content/DASEC_report_{aid}.pdf"

    report       = result.get("report", {})
    sigma_r      = result.get("sigma_rules", {})
    score_global = result.get("score_global", 0.0)
    email_meta   = result.get("email_meta", {})
    modules      = result.get("modules", {})
    nlp          = modules.get("nlp", result.get("nlp", {}))
    urls_mod     = modules.get("urls", result.get("urls", {}))
    headers_mod  = modules.get("headers", {})
    attach_mod   = modules.get("attachments", {})
    rag          = modules.get("rag", result.get("rag", {}))
    ti           = modules.get("threat_intel", result.get("threat_intel", {}))

    doc = SimpleDocTemplate(
        output_path, pagesize=letter,
        topMargin=2*cm, bottomMargin=2*cm, leftMargin=2*cm, rightMargin=2*cm,
        title="DASEC — Rapport d'analyse phishing",
    )

    story = []

    # ── En-tête document ──────────────────────────────────────
    story.append(Paragraph("DASEC", STYLES["DasecTitle"]))
    story.append(Paragraph(
        f"Rapport d'analyse automatisée de phishing — "
        f"généré le {datetime.now().strftime('%d/%m/%Y à %H:%M')}",
        STYLES["DasecSubtitle"]
    ))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#D3D1C7")))
    story.append(Spacer(1, 12))

    # ── Verdict ────────────────────────────────────────────────
    story.extend(_verdict_header(report, score_global))
    story.append(Spacer(1, 16))

    # ── Métadonnées email ──────────────────────────────────────
    story.append(Paragraph("Métadonnées de l'email", STYLES["SectionHeading"]))
    meta_rows = [
        ["Expéditeur", email_meta.get("from", "N/A")],
        ["Sujet",      email_meta.get("subject", "N/A")],
        ["Date",       email_meta.get("date", "N/A")],
        ["Vecteur d'attaque", report.get("vecteur_attaque", "N/A")],
    ]
    t = Table(meta_rows, colWidths=[4*cm, 12*cm])
    t.setStyle(TableStyle([
        ("FONTNAME", (0,0), (0,-1), "Helvetica-Bold"),
        ("FONTSIZE", (0,0), (-1,-1), 9.5),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 12))

    # ── Explication & recommandation ───────────────────────────
    story.append(Paragraph("Analyse", STYLES["SectionHeading"]))
    story.append(Paragraph(report.get("explication", "N/A"), STYLES["BodyDasec"]))
    story.append(Spacer(1, 8))
    reco_style = ParagraphStyle(
        name="Reco", parent=STYLES["BodyDasec"],
        backColor=colors.HexColor("#FAEEDA"), borderPadding=8,
    )
    story.append(Paragraph(
        f"<b>Recommandation :</b> {report.get('recommandation', 'N/A')}", reco_style
    ))
    story.append(Spacer(1, 12))

    # ── Threat Intelligence ─────────────────────────────────────
    story.append(Paragraph("Threat Intelligence", STYLES["SectionHeading"]))
    ti_rows = [
        ["Technique MITRE ATT&CK", report.get("technique_attck", "N/A")],
        ["Campagne probable",      report.get("campagne_probable", "N/A")],
    ]
    if rag.get("apt_groups"):
        top_apt = rag["apt_groups"][0]
        ti_rows.append(["Groupe APT le plus proche",
                        f"{top_apt.get('name','')} ({top_apt.get('group_id','')})"])
    t = Table(ti_rows, colWidths=[5*cm, 11*cm])
    t.setStyle(TableStyle([
        ("FONTNAME", (0,0), (0,-1), "Helvetica-Bold"),
        ("FONTSIZE", (0,0), (-1,-1), 9.5),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 10))

    if ti.get("iocs_detected"):
        story.append(Paragraph("IoCs confirmés (VirusTotal / AbuseIPDB) :", STYLES["BodyDasec"]))
        for ioc in ti["iocs_detected"][:10]:
            story.append(Paragraph(f"&bull; {ioc}", STYLES["BodyDasec"]))
        story.append(Spacer(1, 8))

    # ── Indicateurs clés ─────────────────────────────────────────
    story.append(Paragraph("Indicateurs détectés", STYLES["SectionHeading"]))
    indicateurs = report.get("indicateurs_cles", [])
    if indicateurs:
        rows = [[f"{i+1}", ind] for i, ind in enumerate(indicateurs)]
        story.append(_findings_table(rows, ["#", "Indicateur"], [1.2*cm, 14.8*cm]))
    else:
        story.append(Paragraph("Aucun indicateur spécifique détecté.", STYLES["BodyDasec"]))
    story.append(Spacer(1, 12))

    # ── IoCs structurés ────────────────────────────────────────
    iocs = report.get("iocs", {})
    if any(iocs.get(k) for k in ("urls_suspectes", "ips_malveillantes", "hashes")):
        story.append(Paragraph("Indicateurs de compromission (IoCs)", STYLES["SectionHeading"]))
        ioc_rows = []
        for url in iocs.get("urls_suspectes", []):
            ioc_rows.append(["URL", url])
        for ip in iocs.get("ips_malveillantes", []):
            ioc_rows.append(["IP", ip])
        for h in iocs.get("hashes", []):
            ioc_rows.append(["Hash SHA256", h])
        story.append(_findings_table(ioc_rows, ["Type", "Valeur"], [3*cm, 13*cm]))
        story.append(Spacer(1, 12))

    # ── Scores par module ────────────────────────────────────────
    story.append(Paragraph("Détail des scores par module", STYLES["SectionHeading"]))
    score_rows = [
        ["NLP (DistilBERT + TF-IDF)", f"{nlp.get('score', 0):.3f}"],
        ["URLs",                      f"{urls_mod.get('score', 0):.3f}"],
        ["Threat Intelligence",       f"{ti.get('score', 0):.3f}"],
    ]
    if headers_mod:
        score_rows.insert(0, ["Headers SMTP (SPF/DKIM/DMARC)", f"{headers_mod.get('score', 0):.3f}"])
    if attach_mod:
        score_rows.append(["Pièces jointes", f"{attach_mod.get('score', 0):.3f}"])
    story.append(_findings_table(score_rows, ["Module", "Score (0-1)"], [11*cm, 5*cm]))
    story.append(Spacer(1, 14))

    # ── Règles Sigma ─────────────────────────────────────────────

    if sigma_r.get("rules_count", 0) > 0:
        story.append(PageBreak())
        story.append(Paragraph("Règles Sigma générées", STYLES["SectionHeading"]))
        story.append(Paragraph(
            f"{sigma_r['rules_count']} règle(s) prête(s) à déployer dans un SIEM "
            f"(Splunk, Elastic, Microsoft Sentinel, QRadar).",
            STYLES["BodyDasec"]
        ))
        story.append(Spacer(1, 8))

        combined_yaml = sigma_r.get("combined_yaml", "")
        for line in combined_yaml.split("\n"):
            # Échappement HTML d'abord
            safe_line = (line.replace("&", "&amp;")
                             .replace("<", "&lt;")
                             .replace(">", "&gt;"))
            # Préserve l'indentation : remplace les espaces de tête par des &nbsp;
            stripped = safe_line.lstrip(" ")
            leading_spaces = len(safe_line) - len(stripped)
            if leading_spaces > 0:
                safe_line = ("&nbsp;" * leading_spaces) + stripped
            story.append(Paragraph(safe_line if safe_line.strip() else "&nbsp;", STYLES["Mono"]))
        story.append(Spacer(1, 12))


    # ── Pied de page / disclaimer ───────────────────────────────
    story.append(Spacer(1, 20))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#D3D1C7")))
    disclaimer_style = ParagraphStyle(
        name="Disclaimer", parent=STYLES["Normal"],
        fontSize=8, textColor=colors.HexColor("#888780"),
    )
    story.append(Spacer(1, 6))
    story.append(Paragraph(
        "Rapport généré automatiquement par DASEC (Detection Automatisée et "
        "Sécurisée des E-mails Compromis). Ce rapport est destiné à assister "
        "l'analyste SOC dans sa prise de décision et ne remplace pas une "
        "investigation manuelle pour les cas à fort enjeu.",
        disclaimer_style
    ))

    doc.build(story)
    return output_path


# ══════════════════════════════════════════════════════════════
# PARTIE 4 — Test avec un résultat d'analyse simulé
# (n'a pas besoin du serveur actif — juste un dict Python)
# ══════════════════════════════════════════════════════════════

test_result = {
    "score_global": 0.6775,
    "verdict": "PHISHING",
    "analysis_id": "39c2b19b-55ca-4f89-a895-ef988e510376",
    "email_meta": {
        "from": "security@paypa1.com",
        "subject": "Security Alert",
        "date": "Sat, 20 Jun 2026 10:00:00 +0000",
    },
    "modules": {
        "nlp": {"score": 0.92, "bert_score": 0.95, "lr_score": 0.87},
        "urls": {"score": 0.55, "total_urls": 1, "suspicious": ["http://secure-login.tk/verify"]},
        "threat_intel": {
            "score": 0.85,
            "iocs_detected": ["URL malveillante (VT 9/92) : http://secure-login.tk/verify"],
        },
        "rag": {
            "apt_groups": [{"name": "Threat Group-1314", "group_id": "G0028"}],
        },
    },
    "report": {
        "verdict": "PHISHING",
        "confiance": "haute",
        "campagne_probable": "Threat Group-1314",
        "technique_attck": "T1556 - Modify Authentication Process",
        "vecteur_attaque": "lien",
        "indicateurs_cles": [
            "TLD suspect : secure-login.tk",
            "URL malveillante (VT 9/92) : http://secure-login.tk/verify",
        ],
        "explication": (
            "L'email contient un lien suspect (http://secure-login.tk/verify) qui a "
            "été détecté comme malveillant par VirusTotal (9/92). De plus, le TLD "
            "'secure-login.tk' est suspect."
        ),
        "recommandation": "Vérifier l'authenticité de l'email et ne pas cliquer sur le lien suspect.",
        "iocs": {
            "urls_suspectes": ["http://secure-login.tk/verify"],
            "ips_malveillantes": [],
            "hashes": [],
        },
    },
    "sigma_rules": {
        "rules_count": 2,
        "combined_yaml": (
            "title: Accès à un domaine de phishing détecté par DASEC\n"
            "id: 91aa2a4e-ea62-40f6-99da-31f4e133156f\n"
            "status: experimental\n"
            "logsource:\n  category: proxy\n"
            "detection:\n  selection:\n    c-uri|contains: secure-login.tk\n"
            "  condition: selection\n"
            "level: high\n"
            "---\n"
            "title: Email de phishing détecté par DASEC\n"
            "logsource:\n  category: mail\n"
            "level: high"
        ),
    },
}

pdf_path = generate_pdf_report(test_result)
print(f"✅ PDF généré : {pdf_path}")
print(f"   Taille : {os.path.getsize(pdf_path) / 1024:.1f} Ko")

✅ PDF généré : /content/DASEC_report_39c2b19b.pdf
   Taille : 5.6 Ko


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE F — FASTAPI v5.0 (+ Export PDF natif, fix indentation Sigma)
# Ajoute : /report/pdf/{analysis_id} · ReportLab natif · Sigma indenté
# ══════════════════════════════════════════════════════════════

!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart pyyaml reportlab -q
!pip install python-whois python-magic oletools pytesseract Pillow -q
!pip install python-Levenshtein slack-sdk google-generativeai groq -q
!pip install chromadb sentence-transformers -q
!apt-get install -y tesseract-ocr libmagic1 -q

import os, sys, time, asyncio, threading, subprocess
from google.colab import userdata
from pyngrok import ngrok

def _get_secret(name):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
        return val or ""
    except Exception:
        return os.getenv(name, "")

NGROK_TOKEN    = _get_secret("NGROK_TOKEN")
GEMINI_API_KEY = _get_secret("GEMINI_API_KEY")
GROQ_API_KEY   = _get_secret("GROQ_API_KEY")
SLACK_TOKEN    = _get_secret("SLACK_TOKEN")
VT_API_KEY     = _get_secret("VT_API_KEY")
ABUSEIPDB_KEY  = _get_secret("ABUSEIPDB_KEY")

if not NGROK_TOKEN:
    raise ValueError("❌ NGROK_TOKEN manquant dans les Secrets Colab")

if not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError(
        "❌ Google Drive non monté.\n"
        "   Exécute d'abord : from google.colab import drive; drive.mount('/content/drive')"
    )

try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)
print("✅ Dépendances installées, Drive vérifié, port libéré")

# ══════════════════════════════════════════════════════════════
api_code = r'''
import re, os, sys, json, pickle, hashlib, time, uuid, yaml, sqlite3
import torch
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from email import policy
from email.parser import BytesParser
from io import BytesIO
from typing import Optional
from datetime import datetime, timezone

from fastapi import FastAPI, UploadFile, File, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import pytesseract
from PIL import Image
import Levenshtein
import whois

from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
BASE_DIR        = "/content/drive/MyDrive/phishing_dasec"
VECTORSTORE_DIR = f"{BASE_DIR}/vectorstore"
IOC_CACHE_DIR   = f"{BASE_DIR}/ioc_cache"
DB_PATH         = f"{BASE_DIR}/dasec.db"
PDF_OUTPUT_DIR  = "/tmp/dasec_reports"
os.makedirs(PDF_OUTPUT_DIR, exist_ok=True)

SLACK_TOKEN     = os.getenv("SLACK_TOKEN", "")
SLACK_CHANNEL   = "#soc-alerts"
SOC_THRESHOLD   = 0.65

KNOWN_BRANDS = [
    "paypal","microsoft","google","apple","amazon","netflix",
    "facebook","instagram","twitter","linkedin","dhl","fedex",
    "bnpparibas","societegenerale","creditagricole","orange","sfr",
    "docusign","dropbox","wellsfargo","chase","bankofamerica",
]
SUSPICIOUS_TLDS = {
    ".tk",".xyz",".click",".top",".link",".info",
    ".biz",".ru",".cn",".pw",".ga",".ml",".cf",".gq",
}

# ══════════════════════════════════════════════════════════════
# SQLITE
# ══════════════════════════════════════════════════════════════
def init_db():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS analyses (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id     TEXT    UNIQUE NOT NULL,
            analyzed_at     TEXT    NOT NULL,
            mode            TEXT    NOT NULL,
            score_global    REAL    NOT NULL,
            verdict         TEXT    NOT NULL,
            confidence      TEXT,
            sender          TEXT,
            subject         TEXT,
            technique_attck TEXT,
            campaign        TEXT,
            vecteur_attaque TEXT,
            sigma_count     INTEGER DEFAULT 0,
            alert_sent      INTEGER DEFAULT 0,
            report_json     TEXT    NOT NULL
        );
        CREATE TABLE IF NOT EXISTS iocs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            analysis_id TEXT NOT NULL,
            ioc_type    TEXT NOT NULL,
            ioc_value   TEXT NOT NULL,
            source      TEXT,
            detected_at TEXT NOT NULL,
            FOREIGN KEY (analysis_id) REFERENCES analyses(analysis_id)
        );
        CREATE INDEX IF NOT EXISTS idx_analyses_verdict ON analyses(verdict);
        CREATE INDEX IF NOT EXISTS idx_analyses_score   ON analyses(score_global);
        CREATE INDEX IF NOT EXISTS idx_analyses_date    ON analyses(analyzed_at);
        CREATE INDEX IF NOT EXISTS idx_iocs_analysis_id ON iocs(analysis_id);
        CREATE INDEX IF NOT EXISTS idx_iocs_type        ON iocs(ioc_type);
    """)
    con.commit()
    con.close()

init_db()

def save_analysis(result: dict, mode: str = "text", sender: str = "",
                  subject: str = "", alert_sent: bool = False) -> str:
    analysis_id = str(uuid.uuid4())
    now = datetime.now(timezone.utc).isoformat()
    report  = result.get("report", {})
    sigma_r = result.get("sigma_rules", {})
    iocs    = report.get("iocs", {})
    ti      = result.get("threat_intel", result.get("modules", {}).get("threat_intel", {}))

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("""
        INSERT OR IGNORE INTO analyses
          (analysis_id, analyzed_at, mode, score_global, verdict, confidence,
           sender, subject, technique_attck, campaign, vecteur_attaque,
           sigma_count, alert_sent, report_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        analysis_id, now, mode, result.get("score_global", 0.0),
        report.get("verdict", result.get("verdict", "UNKNOWN")),
        report.get("confiance", ""), sender, subject,
        report.get("technique_attck", ""), report.get("campagne_probable", ""),
        report.get("vecteur_attaque", ""), sigma_r.get("rules_count", 0),
        int(alert_sent), json.dumps(result, ensure_ascii=False),
    ))

    rows = []
    for url in iocs.get("urls_suspectes", []):
        rows.append((analysis_id, "url", url, "pipeline", now))
    for ip in iocs.get("ips_malveillantes", []):
        rows.append((analysis_id, "ip", ip, "AbuseIPDB", now))
    for h in iocs.get("hashes", []):
        rows.append((analysis_id, "hash", h, "VirusTotal", now))
    for ioc_str in ti.get("iocs_detected", []):
        src = "VT" if "VT" in ioc_str else ("AbuseIPDB" if "AbuseIPDB" in ioc_str else "TI")
        rows.append((analysis_id, "ioc_external", ioc_str[:200], src, now))
    if rows:
        cur.executemany(
            "INSERT INTO iocs (analysis_id, ioc_type, ioc_value, source, detected_at) VALUES (?,?,?,?,?)",
            rows
        )
    con.commit()
    con.close()
    return analysis_id

def get_recent_analyses(limit: int = 20, verdict_filter: str = None) -> list:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    if verdict_filter:
        cur.execute("""SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                       sender, subject, technique_attck, campaign, sigma_count, alert_sent
                       FROM analyses WHERE verdict = ? ORDER BY analyzed_at DESC LIMIT ?""",
                   (verdict_filter.upper(), limit))
    else:
        cur.execute("""SELECT analysis_id, analyzed_at, mode, score_global, verdict,
                       sender, subject, technique_attck, campaign, sigma_count, alert_sent
                       FROM analyses ORDER BY analyzed_at DESC LIMIT ?""", (limit,))
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows

def get_analysis_by_id(analysis_id: str):
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM analyses WHERE analysis_id = ?", (analysis_id,))
    row = cur.fetchone()
    con.close()
    if not row:
        return None
    d = dict(row)
    try:
        d["report_full"] = json.loads(d.pop("report_json"))
    except Exception:
        d["report_full"] = {}
    return d

def get_iocs_by_analysis(analysis_id: str) -> list:
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("""SELECT ioc_type, ioc_value, source, detected_at FROM iocs
                   WHERE analysis_id = ? ORDER BY ioc_type""", (analysis_id,))
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows

def get_stats() -> dict:
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("SELECT COUNT(*) FROM analyses")
    total = cur.fetchone()[0]
    cur.execute("SELECT verdict, COUNT(*) FROM analyses GROUP BY verdict")
    by_verdict = dict(cur.fetchall())
    cur.execute("SELECT AVG(score_global) FROM analyses")
    avg_score = round(cur.fetchone()[0] or 0.0, 4)
    cur.execute("SELECT COUNT(*) FROM iocs")
    total_iocs = cur.fetchone()[0]
    cur.execute("""SELECT technique_attck, COUNT(*) c FROM analyses
                   WHERE technique_attck != '' GROUP BY technique_attck
                   ORDER BY c DESC LIMIT 5""")
    top_techniques = cur.fetchall()
    cur.execute("""SELECT campaign, COUNT(*) c FROM analyses
                   WHERE campaign != '' GROUP BY campaign
                   ORDER BY c DESC LIMIT 5""")
    top_campaigns = cur.fetchall()
    con.close()
    return {
        "total_analyses": total, "by_verdict": by_verdict, "avg_score": avg_score,
        "total_iocs": total_iocs,
        "top_techniques": [{"technique": t, "count": c} for t, c in top_techniques],
        "top_campaigns": [{"campaign": c, "count": n} for c, n in top_campaigns],
    }

# ══════════════════════════════════════════════════════════════
# PDF — ReportLab natif (fix indentation Sigma)
# ══════════════════════════════════════════════════════════════
_VERDICT_COLORS = {
    "PHISHING": colors.HexColor("#A32D2D"),
    "SUSPECT":  colors.HexColor("#854F0B"),
    "LEGITIME": colors.HexColor("#3B6D11"),
    "ERREUR":   colors.HexColor("#5F5E5A"),
}

def _pdf_styles():
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="DasecTitle", parent=styles["Title"],
        fontSize=22, textColor=colors.HexColor("#042C53"), spaceAfter=4, alignment=TA_LEFT))
    styles.add(ParagraphStyle(name="DasecSubtitle", parent=styles["Normal"],
        fontSize=11, textColor=colors.HexColor("#5F5E5A"), spaceAfter=18))
    styles.add(ParagraphStyle(name="SectionHeading", parent=styles["Heading2"],
        fontSize=13, textColor=colors.HexColor("#042C53"), spaceBefore=18, spaceAfter=8))
    styles.add(ParagraphStyle(name="BodyDasec", parent=styles["Normal"],
        fontSize=10, leading=15, textColor=colors.HexColor("#2C2C2A")))
    styles.add(ParagraphStyle(name="Mono", parent=styles["Normal"],
        fontName="Courier", fontSize=8.5, leading=11,
        textColor=colors.HexColor("#2C2C2A"), backColor=colors.HexColor("#F1EFE8"),
        spaceAfter=0))
    styles.add(ParagraphStyle(name="VerdictBig", parent=styles["Normal"],
        fontSize=20, leading=24, alignment=TA_CENTER))
    styles.add(ParagraphStyle(name="ScoreLabel", parent=styles["Normal"],
        fontSize=9, alignment=TA_CENTER, textColor=colors.HexColor("#5F5E5A")))
    return styles

_STYLES = _pdf_styles()

def _score_bar(score, verdict):
    width_total = 14 * cm
    filled = max(0.02, min(score, 1.0)) * width_total
    empty  = width_total - filled
    color  = _VERDICT_COLORS.get(verdict, colors.grey)
    t = Table([["", ""]], colWidths=[filled, empty], rowHeights=[0.5*cm])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (0,0), color),
        ("BACKGROUND", (1,0), (1,0), colors.HexColor("#D3D1C7")),
    ]))
    return t

def _findings_table(rows, headers, col_widths=None):
    if not rows:
        return Paragraph("<i>Aucune donnée</i>", _STYLES["BodyDasec"])
    data = [headers] + rows
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#042C53")),
        ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
        ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE",   (0,0), (-1,-1), 8.5),
        ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#F1EFE8")]),
        ("GRID", (0,0), (-1,-1), 0.5, colors.HexColor("#D3D1C7")),
        ("VALIGN", (0,0), (-1,-1), "TOP"),
        ("LEFTPADDING", (0,0), (-1,-1), 6), ("RIGHTPADDING", (0,0), (-1,-1), 6),
        ("TOPPADDING", (0,0), (-1,-1), 5), ("BOTTOMPADDING", (0,0), (-1,-1), 5),
    ]))
    return t

def generate_pdf_report(result: dict, output_path: str = None) -> str:
    if output_path is None:
        aid = result.get("analysis_id", "unknown")[:8]
        output_path = f"{PDF_OUTPUT_DIR}/DASEC_report_{aid}.pdf"

    report       = result.get("report", {})
    sigma_r      = result.get("sigma_rules", {})
    score_global = result.get("score_global", 0.0)
    email_meta   = result.get("email_meta", {})
    modules      = result.get("modules", {})
    nlp          = modules.get("nlp", result.get("nlp", {}))
    urls_mod     = modules.get("urls", result.get("urls", {}))
    headers_mod  = modules.get("headers", {})
    attach_mod   = modules.get("attachments", {})
    rag          = modules.get("rag", result.get("rag", {}))
    ti           = modules.get("threat_intel", result.get("threat_intel", {}))

    doc = SimpleDocTemplate(output_path, pagesize=letter,
        topMargin=2*cm, bottomMargin=2*cm, leftMargin=2*cm, rightMargin=2*cm,
        title="DASEC — Rapport d'analyse phishing")
    story = []

    story.append(Paragraph("DASEC", _STYLES["DasecTitle"]))
    story.append(Paragraph(
        f"Rapport d'analyse automatisée de phishing — généré le "
        f"{datetime.now().strftime('%d/%m/%Y à %H:%M')}", _STYLES["DasecSubtitle"]))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#D3D1C7")))
    story.append(Spacer(1, 12))

    verdict = report.get("verdict", "INCONNU")
    color = _VERDICT_COLORS.get(verdict, colors.grey)
    verdict_style = ParagraphStyle(name="VerdictColored", parent=_STYLES["VerdictBig"], textColor=color)
    story.append(Paragraph(f"<b>{verdict}</b>", verdict_style))
    story.append(Spacer(1, 6))
    story.append(_score_bar(score_global, verdict))
    story.append(Spacer(1, 4))
    story.append(Paragraph(
        f"Score global : <b>{score_global:.2f}</b> / 1.00 &nbsp;&nbsp;|&nbsp;&nbsp; "
        f"Confiance : <b>{report.get('confiance','N/A')}</b>", _STYLES["ScoreLabel"]))
    story.append(Spacer(1, 16))

    story.append(Paragraph("Métadonnées de l'email", _STYLES["SectionHeading"]))
    meta_rows = [
        ["Expéditeur", email_meta.get("from", "N/A")],
        ["Sujet",      email_meta.get("subject", "N/A")],
        ["Date",       email_meta.get("date", "N/A")],
        ["Vecteur d'attaque", report.get("vecteur_attaque", "N/A")],
    ]
    t = Table(meta_rows, colWidths=[4*cm, 12*cm])
    t.setStyle(TableStyle([
        ("FONTNAME", (0,0), (0,-1), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,-1), 9.5),
        ("VALIGN", (0,0), (-1,-1), "TOP"), ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 12))

    story.append(Paragraph("Analyse", _STYLES["SectionHeading"]))
    story.append(Paragraph(report.get("explication", "N/A"), _STYLES["BodyDasec"]))
    story.append(Spacer(1, 8))
    reco_style = ParagraphStyle(name="Reco", parent=_STYLES["BodyDasec"],
        backColor=colors.HexColor("#FAEEDA"), borderPadding=8)
    story.append(Paragraph(f"<b>Recommandation :</b> {report.get('recommandation','N/A')}", reco_style))
    story.append(Spacer(1, 12))

    story.append(Paragraph("Threat Intelligence", _STYLES["SectionHeading"]))
    ti_rows = [
        ["Technique MITRE ATT&CK", report.get("technique_attck", "N/A")],
        ["Campagne probable",      report.get("campagne_probable", "N/A")],
    ]
    if rag.get("apt_groups"):
        top_apt = rag["apt_groups"][0]
        ti_rows.append(["Groupe APT le plus proche",
                        f"{top_apt.get('name','')} ({top_apt.get('group_id','')})"])
    t = Table(ti_rows, colWidths=[5*cm, 11*cm])
    t.setStyle(TableStyle([
        ("FONTNAME", (0,0), (0,-1), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,-1), 9.5),
        ("VALIGN", (0,0), (-1,-1), "TOP"), ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 10))

    if ti.get("iocs_detected"):
        story.append(Paragraph("IoCs confirmés (VirusTotal / AbuseIPDB) :", _STYLES["BodyDasec"]))
        for ioc in ti["iocs_detected"][:10]:
            story.append(Paragraph(f"&bull; {ioc}", _STYLES["BodyDasec"]))
        story.append(Spacer(1, 8))

    story.append(Paragraph("Indicateurs détectés", _STYLES["SectionHeading"]))
    indicateurs = report.get("indicateurs_cles", [])
    if indicateurs:
        rows = [[f"{i+1}", ind] for i, ind in enumerate(indicateurs)]
        story.append(_findings_table(rows, ["#", "Indicateur"], [1.2*cm, 14.8*cm]))
    else:
        story.append(Paragraph("Aucun indicateur spécifique détecté.", _STYLES["BodyDasec"]))
    story.append(Spacer(1, 12))

    iocs = report.get("iocs", {})
    if any(iocs.get(k) for k in ("urls_suspectes","ips_malveillantes","hashes")):
        story.append(Paragraph("Indicateurs de compromission (IoCs)", _STYLES["SectionHeading"]))
        ioc_rows = []
        for url in iocs.get("urls_suspectes", []): ioc_rows.append(["URL", url])
        for ip in iocs.get("ips_malveillantes", []): ioc_rows.append(["IP", ip])
        for h in iocs.get("hashes", []): ioc_rows.append(["Hash SHA256", h])
        story.append(_findings_table(ioc_rows, ["Type", "Valeur"], [3*cm, 13*cm]))
        story.append(Spacer(1, 12))

    story.append(Paragraph("Détail des scores par module", _STYLES["SectionHeading"]))
    score_rows = [
        ["NLP (DistilBERT + TF-IDF)", f"{nlp.get('score', 0):.3f}"],
        ["URLs",                      f"{urls_mod.get('score', 0):.3f}"],
        ["Threat Intelligence",       f"{ti.get('score', 0):.3f}"],
    ]
    if headers_mod:
        score_rows.insert(0, ["Headers SMTP (SPF/DKIM/DMARC)", f"{headers_mod.get('score', 0):.3f}"])
    if attach_mod:
        score_rows.append(["Pièces jointes", f"{attach_mod.get('score', 0):.3f}"])
    story.append(_findings_table(score_rows, ["Module", "Score (0-1)"], [11*cm, 5*cm]))
    story.append(Spacer(1, 14))

    if sigma_r.get("rules_count", 0) > 0:
        story.append(PageBreak())
        story.append(Paragraph("Règles Sigma générées", _STYLES["SectionHeading"]))
        story.append(Paragraph(
            f"{sigma_r['rules_count']} règle(s) prête(s) à déployer dans un SIEM "
            f"(Splunk, Elastic, Microsoft Sentinel, QRadar).", _STYLES["BodyDasec"]))
        story.append(Spacer(1, 8))
        combined_yaml = sigma_r.get("combined_yaml", "")
        for line in combined_yaml.split("\n"):
            safe_line = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
            stripped = safe_line.lstrip(" ")
            leading_spaces = len(safe_line) - len(stripped)
            if leading_spaces > 0:
                safe_line = ("&nbsp;" * leading_spaces) + stripped
            story.append(Paragraph(safe_line if safe_line.strip() else "&nbsp;", _STYLES["Mono"]))
        story.append(Spacer(1, 12))

    story.append(Spacer(1, 20))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#D3D1C7")))
    disclaimer_style = ParagraphStyle(name="Disclaimer", parent=_STYLES["Normal"],
        fontSize=8, textColor=colors.HexColor("#888780"))
    story.append(Spacer(1, 6))
    story.append(Paragraph(
        "Rapport généré automatiquement par DASEC (Detection Automatisée et "
        "Sécurisée des E-mails Compromis). Ce rapport est destiné à assister "
        "l'analyste SOC dans sa prise de décision et ne remplace pas une "
        "investigation manuelle pour les cas à fort enjeu.", disclaimer_style))

    doc.build(story)
    return output_path

# ══════════════════════════════════════════════════════════════
# CHARGEMENT MODÈLES
# ══════════════════════════════════════════════════════════════
print("⏳ Chargement des modèles...")

with open(f"{BASE_DIR}/models/tfidf_lr_pipeline.pkl", "rb") as f:
    artifacts = pickle.load(f)
tfidf    = artifacts["tfidf"]
lr_model = artifacts["lr_model"]

BERT_PATH  = f"{BASE_DIR}/models/distilbert-5000/final"
tokenizer  = DistilBertTokenizerFast.from_pretrained(BERT_PATH)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_PATH)
bert_model.eval()

embedder      = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=VECTORSTORE_DIR)

def embed_fn(texts):
    return embedder.encode(texts).tolist()

def _get_col(name):
    try:
        return chroma_client.get_collection(name)
    except Exception:
        return None

col_techniques = _get_col("mitre_techniques")
col_apt        = _get_col("mitre_apt_groups")
col_ti         = _get_col("threat_intel")
col_live       = _get_col("live_ioc_feeds")
RAG_OK         = all([col_techniques, col_apt, col_ti])

import json as _json
def _load_set(path):
    try:
        with open(path) as f:
            return set(_json.load(f))
    except Exception:
        return set()

urlhaus_set   = _load_set(f"{IOC_CACHE_DIR}/urlhaus.json")
openphish_set = _load_set(f"{IOC_CACHE_DIR}/openphish.json")

llm_client  = None
llm_backend = None
GROQ_KEY   = os.getenv("GROQ_API_KEY", "")
GEMINI_KEY = os.getenv("GEMINI_API_KEY", "")

if GROQ_KEY:
    try:
        from groq import Groq
        llm_client  = Groq(api_key=GROQ_KEY)
        llm_backend = "groq"
        print("✅ LLM : Groq")
    except Exception as e:
        print(f"⚠️ Groq : {e}")

if not llm_client and GEMINI_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_KEY)
        llm_client  = genai.GenerativeModel("gemini-2.5-flash")
        llm_backend = "gemini"
        print("✅ LLM : Gemini")
    except Exception as e:
        print(f"⚠️ Gemini : {e}")

if not llm_client:
    print("⚠️ Aucun LLM — mode rapport structuré")

print(f"✅ Chargé | RAG={'OK' if RAG_OK else 'KO'} | LLM={llm_backend or 'none'} "
      f"| URLHaus={len(urlhaus_set):,} | OpenPhish={len(openphish_set):,} | DB={DB_PATH}")

# ══════════════════════════════════════════════════════════════
# THREAT INTEL
# ══════════════════════════════════════════════════════════════
VT_API_KEY    = os.getenv("VT_API_KEY", "")
ABUSEIPDB_KEY = os.getenv("ABUSEIPDB_KEY", "")
_ti_cache     = {}
_ti_executor  = ThreadPoolExecutor(max_workers=4)
VT_BASE_URL    = "https://www.virustotal.com/api/v3"
ABUSE_BASE_URL = "https://api.abuseipdb.com/api/v2/check"

def _vt_check_hash(sha256, timeout=5):
    cache_key = f"vt_hash:{sha256}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        r = _rq.get(f"{VT_BASE_URL}/files/{sha256}", headers={"x-apikey": VT_API_KEY}, timeout=timeout)
        if r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt", "malicious_count": 0, "total_engines": 0}
        elif r.status_code == 200:
            data  = r.json()["data"]["attributes"]
            stats = data.get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0, "names": data.get("names", [])[:3]}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _vt_check_url(target_url, timeout=5):
    cache_key = f"vt_url:{target_url}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not VT_API_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq, base64
        url_id = base64.urlsafe_b64encode(target_url.encode()).decode().strip("=")
        r = _rq.get(f"{VT_BASE_URL}/urls/{url_id}", headers={"x-apikey": VT_API_KEY}, timeout=timeout)
        if r.status_code == 200:
            stats = r.json()["data"]["attributes"].get("last_analysis_stats", {})
            malicious, suspicious = stats.get("malicious", 0), stats.get("suspicious", 0)
            result = {"checked": True, "malicious": (malicious+suspicious) > 0,
                      "malicious_count": malicious, "suspicious_count": suspicious,
                      "total_engines": sum(stats.values()) if stats else 0}
        elif r.status_code == 404:
            result = {"checked": True, "malicious": False, "reason": "not_in_vt"}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def _abuseipdb_check(ip, timeout=5):
    cache_key = f"abuse:{ip}"
    if cache_key in _ti_cache: return _ti_cache[cache_key]
    if not ABUSEIPDB_KEY: return {"checked": False, "malicious": False, "reason": "no_api_key"}
    try:
        import requests as _rq
        r = _rq.get(ABUSE_BASE_URL, headers={"Key": ABUSEIPDB_KEY, "Accept": "application/json"},
                    params={"ipAddress": ip, "maxAgeInDays": 90}, timeout=timeout)
        if r.status_code == 200:
            data  = r.json()["data"]
            score = data.get("abuseConfidenceScore", 0)
            result = {"checked": True, "malicious": score >= 50, "abuse_score": score,
                      "total_reports": data.get("totalReports", 0), "country": data.get("countryCode", ""),
                      "isp": data.get("isp", ""), "is_tor": data.get("isTor", False)}
        else:
            result = {"checked": False, "malicious": False, "reason": f"http_{r.status_code}"}
        _ti_cache[cache_key] = result
        return result
    except Exception as e:
        return {"checked": False, "malicious": False, "reason": str(e)[:80]}

def threat_intel_check(urls=None, ips=None, hashes=None):
    urls, ips, hashes = urls or [], ips or [], hashes or []
    iocs_detected, details = [], {"urls": [], "ips": [], "hashes": []}
    apis_used = {"virustotal": bool(VT_API_KEY), "abuseipdb": bool(ABUSEIPDB_KEY)}
    score = 0.0

    def _run(fn, arg, timeout=6):
        try: return _ti_executor.submit(fn, arg).result(timeout=timeout)
        except Exception as e: return {"checked": False, "malicious": False, "reason": str(e)[:80]}

    for h in hashes[:2]:
        r = _run(_vt_check_hash, h)
        details["hashes"].append({"hash": h, **r})
        if r.get("malicious"):
            score += 0.95
            iocs_detected.append(f"Hash malveillant (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {h[:16]}...")

    for u in urls[:3]:
        r = _run(_vt_check_url, u)
        details["urls"].append({"url": u, **r})
        if r.get("malicious"):
            score += 0.85
            iocs_detected.append(f"URL malveillante (VT {r.get('malicious_count',0)}/{r.get('total_engines',0)}) : {u[:50]}")

    for ip in ips[:3]:
        r = _run(_abuseipdb_check, ip)
        details["ips"].append({"ip": ip, **r})
        if r.get("malicious"):
            score += 0.70
            iocs_detected.append(f"IP malveillante (AbuseIPDB {r.get('abuse_score',0)}%) : {ip}")

    return {"score": round(min(score, 1.0), 4), "iocs_detected": iocs_detected, "details": details, "apis_used": apis_used}

# ══════════════════════════════════════════════════════════════
# UTILITAIRES
# ══════════════════════════════════════════════════════════════
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_urls(text):
    return list(set(re.findall(r"https?://[^\s<>\"\']+|www\.[^\s<>\"\']+", text)))

def get_domain(url):
    m = re.search(r"(?:https?://)?(?:www\.)?([^/\s]+)", url)
    return m.group(1).lower() if m else ""

def parse_eml(raw_bytes):
    msg = BytesParser(policy=policy.default).parsebytes(raw_bytes)
    body_text, body_html = "", ""
    images, attachments  = [], []
    for part in msg.walk():
        ct = part.get_content_type()
        cd = str(part.get("Content-Disposition", ""))
        if ct == "text/plain" and "attachment" not in cd:
            body_text += part.get_content() or ""
        elif ct == "text/html" and "attachment" not in cd:
            body_html += str(part.get_content() or "")
        elif ct.startswith("image/"):
            images.append({"content_type": ct, "data": part.get_payload(decode=True), "name": part.get_filename("image")})
        elif "attachment" in cd or part.get_filename():
            payload = part.get_payload(decode=True)
            if payload:
                attachments.append({"filename": part.get_filename("unknown"), "content_type": ct,
                                    "data": payload, "md5": hashlib.md5(payload).hexdigest(),
                                    "sha256": hashlib.sha256(payload).hexdigest(), "size": len(payload)})
    body_html_clean = re.sub(r"<[^>]+>", " ", body_html)
    full_text = f"{body_text} {body_html_clean}".strip()
    return {"headers": dict(msg.items()), "from": msg.get("From", ""), "reply_to": msg.get("Reply-To", ""),
            "return_path": msg.get("Return-Path", ""), "subject": msg.get("Subject", ""), "date": msg.get("Date", ""),
            "message_id": msg.get("Message-ID", ""), "x_mailer": msg.get("X-Mailer", ""), "body_text": body_text,
            "body_html": body_html, "full_text": full_text, "images": images, "attachments": attachments,
            "urls_in_html": extract_urls(body_html), "urls_in_text": extract_urls(body_text)}

def analyze_headers(parsed):
    headers, findings, score = parsed["headers"], [], 0.0
    auth = str(headers.get("Authentication-Results", "")).lower()
    spf  = str(headers.get("Received-SPF", "")).lower()
    dkim = "dkim-signature" in {k.lower() for k in headers}
    spf_pass, dkim_pass, dmarc_pass = ("pass" in spf or "pass" in auth), (dkim and "dkim=pass" in auth), ("dmarc=pass" in auth)
    if not spf_pass: findings.append("SPF absent ou échoué"); score += 0.20
    if not dkim_pass: findings.append("DKIM absent ou échoué"); score += 0.20
    if not dmarc_pass: findings.append("DMARC absent ou échoué"); score += 0.10
    from_addr, reply_to = parsed["from"], parsed["reply_to"]
    if reply_to and from_addr and reply_to != from_addr:
        findings.append(f"Reply-To ≠ From : {reply_to}"); score += 0.20
    return_path = parsed["return_path"]
    if return_path and from_addr:
        fd, rd = get_domain(from_addr), get_domain(return_path)
        if fd and rd and fd != rd:
            findings.append(f"Return-Path ({rd}) ≠ From ({fd})"); score += 0.15
    mailer = parsed["x_mailer"].lower()
    for sm in ["mailchimp","sendgrid","phpmailer","massmailer","bulk"]:
        if sm in mailer:
            findings.append(f"X-Mailer suspect : {parsed['x_mailer']}"); score += 0.10; break
    msg_id = parsed["message_id"]
    if not msg_id or not re.match(r"<.+@.+>", msg_id):
        findings.append("Message-ID absent ou invalide"); score += 0.10
    received  = str(headers.get("Received", ""))
    ips_found = list(set(re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", received)))
    return {"score": round(min(score, 1.0), 3), "spf_pass": spf_pass, "dkim_pass": dkim_pass,
            "dmarc_pass": dmarc_pass, "findings": findings, "ips_found": ips_found[:5]}

_whois_executor = ThreadPoolExecutor(max_workers=2)
def _whois_safe(domain):
    try: return _whois_executor.submit(whois.whois, domain).result(timeout=3)
    except Exception: return {}

def analyze_urls(urls):
    findings, suspicious, score = [], [], 0.0
    for url in urls[:10]:
        domain = get_domain(url)
        if not domain: continue
        for tld in SUSPICIOUS_TLDS:
            if domain.endswith(tld):
                findings.append(f"TLD suspect : {domain}"); score += 0.15; suspicious.append(url); break
        domain_base = domain.split(".")[0]
        for brand in KNOWN_BRANDS:
            if 0 < Levenshtein.distance(domain_base, brand) <= 2:
                findings.append(f"Typosquatting : {domain} ≈ {brand}"); score += 0.25; suspicious.append(url); break
        if len(domain) > 40: findings.append(f"Domaine trop long : {domain}"); score += 0.10
        if domain.count("-") >= 3: findings.append(f"Beaucoup de tirets : {domain}"); score += 0.10
        if re.match(r"https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", url):
            findings.append(f"IP directe : {url[:60]}"); score += 0.30; suspicious.append(url)
        url_norm = url.lower()
        if url_norm in urlhaus_set or url in urlhaus_set:
            findings.append(f"URLHaus blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)
        if url_norm in openphish_set or url in openphish_set:
            findings.append(f"OpenPhish blacklist : {url[:60]}"); score += 0.90; suspicious.append(url)
        try:
            w = _whois_safe(domain)
            if w and w.get("creation_date"):
                creation = w["creation_date"]
                if isinstance(creation, list): creation = creation[0]
                if creation and (datetime.now() - creation).days < 30:
                    findings.append(f"Domaine très récent : {domain}"); score += 0.30; suspicious.append(url)
        except Exception: pass
    return {"score": round(min(score, 1.0), 3), "total_urls": len(urls),
            "suspicious": list(set(suspicious))[:5], "findings": findings}

def analyze_images(images):
    ocr_texts, findings, score = [], [], 0.0
    if len(images) > 3:
        findings.append(f"{len(images)} images — contournement filtre"); score += 0.15
    for img_data in images[:5]:
        try:
            raw = img_data["data"]
            if not raw: continue
            img = Image.open(BytesIO(raw))
            w, h = img.size
            if w <= 3 and h <= 3:
                findings.append("Pixel tracking"); score += 0.20; continue
            text = pytesseract.image_to_string(img, lang="eng+fra")
            if text.strip():
                ocr_texts.append(text.strip())
                kw = [k for k in ["verify","account","password","click","urgent","suspended","login","security","bank","credit card"] if k in text.lower()]
                if kw: findings.append(f"Mots-clés phishing image : {kw}"); score += 0.20
        except Exception: pass
    return {"score": round(min(score, 1.0), 3), "ocr_text": " ".join(ocr_texts)[:500], "images_count": len(images), "findings": findings}

def analyze_attachments(attachments):
    findings, suspicious_hashes, score = [], [], 0.0
    DANGEROUS_EXT = {".exe",".bat",".cmd",".ps1",".vbs",".js",".jar",".scr",".pif",".com",".msi"}
    DOUBLE_EXT    = re.compile(r"\.(pdf|docx?|xlsx?|zip)\.(exe|bat|cmd|scr)$", re.I)
    for att in attachments:
        fname, ct = att["filename"].lower(), att["content_type"]
        for ext in DANGEROUS_EXT:
            if fname.endswith(ext):
                findings.append(f"Extension dangereuse : {att['filename']}"); score += 0.40; break
        if DOUBLE_EXT.search(fname):
            findings.append(f"Double extension : {att['filename']}"); score += 0.50
        try:
            import magic
            real_mime = magic.from_buffer(att["data"][:1024], mime=True)
            if real_mime and real_mime not in ct:
                findings.append(f"MIME réel ({real_mime}) ≠ déclaré ({ct})"); score += 0.35
        except Exception: pass
        if fname.endswith((".doc",".docx",".xls",".xlsx",".ppt",".pptx")):
            try:
                from oletools.olevba import VBA_Parser
                vba = VBA_Parser(att["filename"], data=att["data"])
                if vba.detect_vba_macros():
                    findings.append(f"Macros VBA : {att['filename']}"); score += 0.45
            except Exception: pass
        suspicious_hashes.append({"filename": att["filename"], "md5": att["md5"], "sha256": att["sha256"], "size": att["size"]})
    return {"score": round(min(score, 1.0), 3), "attachments_count": len(attachments), "suspicious_hashes": suspicious_hashes, "findings": findings}

def analyze_text(text):
    text_clean = clean_text(text)
    if not text_clean.strip():
        return {"bert_score":0.0,"lr_score":0.0,"score":0.0,"keywords":[]}
    inputs = tokenizer(text_clean, truncation=True, padding=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    bert_score = float(torch.softmax(logits, dim=1).numpy()[0][1])
    lr_score   = float(lr_model.predict_proba(tfidf.transform([text_clean]))[0][1])
    keywords = [kw for kw in ["verify","account","password","click here","urgent","confirm","suspended","login",
                "update","security","limited","expire","winner","prize","claim","free","bank","credit card","wire transfer"]
                if kw in text.lower()]
    return {"bert_score": round(bert_score, 4), "lr_score": round(lr_score, 4),
            "score": round(bert_score * 0.6 + lr_score * 0.4, 4), "keywords": keywords}

def rag_search(text):
    if not RAG_OK:
        return {"techniques":[],"apt_groups":[],"campaigns":[],"live_iocs":[]}
    q_emb = embed_fn([text[:500]])
    def _query(col, n):
        if not col: return None
        try: return col.query(query_embeddings=q_emb, n_results=n)
        except Exception: return None
    def _parse(res, fields):
        if not res or not res["ids"][0]: return []
        out = []
        for i in range(len(res["ids"][0])):
            item = {"similarity": round(1 - res["distances"][0][i], 3)}
            for f in fields: item[f] = res["metadatas"][0][i].get(f, "")
            out.append(item)
        return out
    return {
        "techniques": _parse(_query(col_techniques, 3), ["tech_id","name","tactics"]),
        "apt_groups": _parse(_query(col_apt, 2),        ["group_id","name","aliases"]),
        "campaigns":  _parse(_query(col_ti, 2),         ["campaign","actor","technique","severity"]),
        "live_iocs":  _parse(_query(col_live, 2),       ["url","attack_type","brand_target"]) if col_live else [],
    }

def _call_llm(prompt):
    if llm_backend == "groq":
        resp = llm_client.chat.completions.create(model="llama-3.1-8b-instant",
            messages=[{"role":"user","content":prompt}], temperature=0.1, max_tokens=800)
        return resp.choices[0].message.content
    elif llm_backend == "gemini":
        return llm_client.generate_content(prompt).text
    return ""

def generate_report(parsed, header_r, url_r, image_r, attach_r, text_r, rag_r, ti_r, score_global):
    top_tech     = rag_r["techniques"][0] if rag_r["techniques"] else {}
    top_campaign = rag_r["campaigns"][0]  if rag_r["campaigns"]  else {}
    top_apt      = rag_r["apt_groups"][0] if rag_r["apt_groups"] else {}
    all_findings = (header_r["findings"] + url_r["findings"] + image_r["findings"] + attach_r["findings"])
    iocs_detected = ti_r.get("iocs_detected", [])

    if not llm_client:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {"verdict": verdict, "confiance": "haute" if score_global >= 0.75 else "moyenne",
                "score_final": score_global, "campagne_probable": top_campaign.get("campaign","Inconnue"),
                "technique_attck": f"{top_tech.get('tech_id','')} — {top_tech.get('name','N/A')}",
                "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
                "indicateurs_cles": all_findings[:5],
                "explication": f"Score {score_global:.2f}. {len(all_findings)} indicateurs. LLM indisponible.",
                "recommandation": "Bloquer et quarantaine." if score_global >= 0.65 else "Vérification manuelle.",
                "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                         "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
                "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else ""}

    urls_json   = _json.dumps(url_r["suspicious"])
    ips_json    = _json.dumps(header_r["ips_found"])
    hashes_json = _json.dumps([h["sha256"] for h in attach_r["suspicious_hashes"]])
    ti_block    = f"IoCs confirmés externes : {iocs_detected}" if iocs_detected else "Aucun IoC externe confirmé"

    prompt = f"""Tu es un analyste SOC expert en détection de phishing.

=== EMAIL ===
De      : {parsed.get("from","")}
Sujet   : {parsed.get("subject","")}
Extrait : {parsed.get("full_text","")[:300]}

=== SCORES ===
Headers : {header_r["score"]} | SPF={header_r["spf_pass"]} DKIM={header_r["dkim_pass"]} DMARC={header_r["dmarc_pass"]}
URLs    : {url_r["score"]} ({url_r["total_urls"]} URLs, {len(url_r["suspicious"])} suspectes)
OCR     : {image_r["score"]} | Attachments : {attach_r["score"]}
NLP     : BERT={text_r["bert_score"]} LR={text_r["lr_score"]}
TI      : {ti_r.get("score",0)} | Global : {score_global}

=== INDICATEURS ===
{chr(10).join(all_findings) if all_findings else "Aucun"}

=== THREAT INTEL EXTERNE ===
{ti_block}

=== RAG ===
MITRE   : {top_tech.get("tech_id","")} {top_tech.get("name","")} (sim={top_tech.get("similarity",0)})
APT     : {top_apt.get("name","")} (sim={top_apt.get("similarity",0)})
Campagne: {top_campaign.get("campaign","")} — {top_campaign.get("actor","")} (sim={top_campaign.get("similarity",0)})

Réponds STRICTEMENT avec un JSON valide, sans texte avant/après, sans markdown :

{{"verdict": "PHISHING", "confiance": "haute", "score_final": {score_global}, "campagne_probable": "nom ou Aucune", "technique_attck": "T#### - nom", "vecteur_attaque": "lien", "indicateurs_cles": ["max 5, mentionne les hits VT/AbuseIPDB si presents"], "explication": "2-3 phrases SOC", "recommandation": "action", "iocs": {{"urls_suspectes": {urls_json}, "ips_malveillantes": {ips_json}, "hashes": {hashes_json}}}, "sigma_hint": "valeur"}}"""

    try:
        raw = _call_llm(prompt)
        raw = re.sub(r"```json|```", "", raw).strip()
        m   = re.search(r"\{.*\}", raw, re.DOTALL)
        return _json.loads(m.group() if m else raw)
    except Exception as e:
        verdict = ("PHISHING" if score_global >= 0.65 else "SUSPECT" if score_global >= 0.35 else "LEGITIME")
        return {"verdict": verdict, "confiance": "moyenne", "score_final": score_global,
                "explication": f"Erreur LLM : {str(e)[:100]}",
                "campagne_probable": top_campaign.get("campaign","N/A"),
                "technique_attck": f"{top_tech.get('tech_id','')} {top_tech.get('name','')}",
                "vecteur_attaque": "lien" if url_r["suspicious"] else "texte seul",
                "indicateurs_cles": all_findings[:5], "recommandation": "Vérification manuelle",
                "iocs": {"urls_suspectes": url_r["suspicious"], "ips_malveillantes": header_r["ips_found"],
                         "hashes": [h["sha256"] for h in attach_r["suspicious_hashes"]]},
                "sigma_hint": url_r["suspicious"][0] if url_r["suspicious"] else ""}

def _mitre_to_tag(technique_attck):
    m = re.search(r"T(\d{4})(?:\.(\d{3}))?", technique_attck or "")
    if not m: return "attack.t1566"
    base = f"attack.t{m.group(1)}"
    return f"{base}.{m.group(2)}" if m.group(2) else base

def _tactic_tag(tech_id):
    TACTIC_MAP = {"T1566":"attack.initial_access","T1598":"attack.reconnaissance",
                  "T1556":"attack.credential_access","T1539":"attack.credential_access",
                  "T1204":"attack.execution","T1027":"attack.defense_evasion"}
    base = tech_id.split(".")[0] if tech_id else ""
    return TACTIC_MAP.get(base, "attack.initial_access")

def _level_from_score(score):
    if score >= 0.85: return "critical"
    if score >= 0.65: return "high"
    if score >= 0.40: return "medium"
    if score >= 0.20: return "low"
    return "informational"

def generate_sigma_rules(report, score_global, email_meta=None):
    email_meta = email_meta or {}
    rules = []
    level = _level_from_score(score_global)
    tech_id = (report.get("technique_attck","") or "").split(" ")[0].strip("—-").strip()
    tag_technique, tag_tactic = _mitre_to_tag(tech_id), _tactic_tag(tech_id)
    rule_date = datetime.now().strftime("%Y/%m/%d")
    iocs = report.get("iocs", {})
    urls_susp, ips_mal, hashes_mal = iocs.get("urls_suspectes",[]), iocs.get("ips_malveillantes",[]), iocs.get("hashes",[])
    common_tags = ["attack.phishing", tag_tactic, tag_technique]
    common_refs = [f"https://attack.mitre.org/techniques/{tech_id}"] if tech_id else []

    if urls_susp:
        domains = list(set([re.sub(r"^https?://","",u).split("/")[0] for u in urls_susp]))
        rules.append({"type":"proxy_network","data":{
            "title": f"Accès à un domaine de phishing détecté par DASEC ({report.get('campagne_probable','Inconnue')})",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Connexion vers domaine malveillant. Score : {score_global:.2f}. Vecteur : {report.get('vecteur_attaque','lien')}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"proxy"},
            "detection":{"selection":{"c-uri|contains": domains if len(domains)>1 else domains[0]},"condition":"selection"},
            "falsepositives":["Domaine réutilisé légitimement","Faux positif AV tiers"],"level":level}})

    sender, subject = email_meta.get("from",""), email_meta.get("subject","")
    if sender or subject:
        selection = {}
        if sender: selection["sender|contains"] = sender.split("@")[-1] if "@" in sender else sender
        if subject: selection["subject|contains"] = subject[:60]
        rules.append({"type":"mail_gateway","data":{
            "title": f"Email de phishing détecté par DASEC — {report.get('campagne_probable','Inconnue')}",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Caractéristiques phishing : {', '.join(report.get('indicateurs_cles',[])[:3]) or 'analyse NLP'}. MITRE : {report.get('technique_attck','N/A')}.",
            "references": common_refs, "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags,
            "logsource":{"category":"mail"}, "detection":{"selection":selection,"condition":"selection"},
            "falsepositives":["Email légitime — vérifier SPF/DKIM/DMARC"],"level":level}})

    if hashes_mal:
        rules.append({"type":"file_hash","data":{
            "title":"Pièce jointe malveillante détectée par DASEC (hash confirmé VirusTotal)",
            "id": str(uuid.uuid4()), "status":"experimental",
            "description": f"Hash confirmé malveillant par VirusTotal. Score : {score_global:.2f}.",
            "references": common_refs + ["https://www.virustotal.com"],
            "author":"DASEC Automated Pipeline","date":rule_date,"tags":common_tags+["attack.execution"],
            "logsource":{"category":"file_event"}, "detection":{"selection":{"Hashes|contains":hashes_mal},"condition":"selection"},
            "falsepositives":["Aucun connu"],"level":"critical"}})

    yaml_blocks = []
    for r in rules:
        clean = {k:v for k,v in r["data"].items() if v is not None}
        yaml_str = yaml.dump(clean, sort_keys=False, allow_unicode=True, default_flow_style=False, width=100)
        yaml_blocks.append(yaml_str)
        r["yaml"] = yaml_str
    combined = "\n---\n".join(yaml_blocks) if yaml_blocks else "# Aucune règle générée"
    return {"rules_count": len(rules), "rules": [{"type":r["type"],"yaml":r["yaml"]} for r in rules], "combined_yaml": combined}

def send_soc_alert(report, parsed, score):
    if not SLACK_TOKEN or score < SOC_THRESHOLD: return False
    try:
        from slack_sdk import WebClient
        client = WebClient(token=SLACK_TOKEN)
        verdict = report.get("verdict","?")
        emoji = "🚨" if verdict == "PHISHING" else "⚠️"
        msg = (f"{emoji} *DASEC — {verdict}*\n>Score : {score:.2f} | Confiance : {report.get('confiance','?')}\n"
               f">De : {parsed.get('from','')}\n>Sujet : {parsed.get('subject','')}\n"
               f">Campagne : {report.get('campagne_probable','N/A')}\n>Technique : {report.get('technique_attck','N/A')}\n"
               f">Action : {report.get('recommandation','N/A')}\n>Indicateurs : {', '.join(report.get('indicateurs_cles',[])[:3])}")
        client.chat_postMessage(channel=SLACK_CHANNEL, text=msg)
        return True
    except Exception as e:
        print(f"Slack error: {e}"); return False

def compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full"):
    if mode == "text_only":
        return round(text_r["score"]*0.50 + url_r["score"]*0.35 + ti_r.get("score",0.0)*0.10 +
                    (len(url_r["suspicious"])>0)*0.05, 4)
    return round(header_r["score"]*0.25 + text_r["score"]*0.20 + url_r["score"]*0.20 +
                attach_r["score"]*0.10 + ti_r.get("score",0.0)*0.15 + image_r["score"]*0.05 +
                (len(url_r["suspicious"])>0)*0.05, 4)

# ══════════════════════════════════════════════════════════════
# FASTAPI
# ══════════════════════════════════════════════════════════════
app = FastAPI(title="DASEC — Phishing Detection API",
             description="Headers+URLs+OCR+Attachments+NLP+RAG+LLM+ThreatIntel+Sigma+SOC+SQLite+PDF", version="5.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class TextRequest(BaseModel):
    text: str
    subject: Optional[str] = ""
    sender: Optional[str] = ""

EMPTY_HEADER = {"score":0.0,"spf_pass":None,"dkim_pass":None,"dmarc_pass":None,"findings":[],"ips_found":[]}
EMPTY_IMAGE  = {"score":0.0,"ocr_text":"","images_count":0,"findings":[]}
EMPTY_ATTACH = {"score":0.0,"attachments_count":0,"suspicious_hashes":[],"findings":[]}

@app.get("/")
def root():
    return {"message":"DASEC v5.0","status":"running","rag":RAG_OK,"llm_backend":llm_backend or "none"}

@app.get("/health")
def health():
    return {
        "status":"ok","models":["distilbert","lr","sentence-transformers"],
        "rag_collections":{
            "mitre_techniques": col_techniques.count() if col_techniques else 0,
            "mitre_apt_groups": col_apt.count() if col_apt else 0,
            "threat_intel": col_ti.count() if col_ti else 0,
            "live_ioc_feeds": col_live.count() if col_live else 0,
        },
        "ioc_feeds":{"urlhaus":len(urlhaus_set),"openphish":len(openphish_set)},
        "threat_intel_apis":{"virustotal":bool(VT_API_KEY),"abuseipdb":bool(ABUSEIPDB_KEY)},
        "llm_backend":llm_backend or "none","soc_alert":bool(SLACK_TOKEN),"soc_threshold":SOC_THRESHOLD,
        "db_path": DB_PATH, "pdf_output_dir": PDF_OUTPUT_DIR,
    }

@app.post("/analyze/eml")
async def analyze_eml(file: UploadFile = File(...)):
    if not file.filename.endswith(".eml"):
        raise HTTPException(400, "Fichier .eml requis")
    raw = await file.read()
    parsed = parse_eml(raw)
    all_urls = list(set(parsed["urls_in_html"] + parsed["urls_in_text"]))
    header_r = analyze_headers(parsed)
    url_r    = analyze_urls(all_urls)
    image_r  = analyze_images(parsed["images"])
    attach_r = analyze_attachments(parsed["attachments"])
    full_text = parsed["full_text"] + " " + image_r["ocr_text"]
    text_r = analyze_text(full_text)
    rag_r  = rag_search(full_text)
    ti_r = threat_intel_check(urls=url_r["suspicious"][:3], ips=header_r["ips_found"][:3],
                              hashes=[a["sha256"] for a in parsed["attachments"]][:2])
    score_global = compute_global_score(header_r, url_r, image_r, attach_r, text_r, ti_r, mode="full")
    report = generate_report(parsed, header_r, url_r, image_r, attach_r, text_r, rag_r, ti_r, score_global)
    sigma_r = generate_sigma_rules(report, score_global, {"from":parsed["from"],"subject":parsed["subject"]})
    alert_sent = send_soc_alert(report, parsed, score_global)
    result = {"score_global": score_global, "verdict": report.get("verdict","?"),
              "modules":{"headers":header_r,"urls":url_r,"images":image_r,"attachments":attach_r,
                        "nlp":text_r,"rag":rag_r,"threat_intel":ti_r},
              "report": report, "sigma_rules": sigma_r, "alert_sent": alert_sent,
              "email_meta":{"from":parsed["from"],"subject":parsed["subject"],"date":parsed["date"]}}
    aid = save_analysis(result, mode="eml", sender=parsed["from"], subject=parsed["subject"], alert_sent=alert_sent)
    result["analysis_id"] = aid
    return result

@app.post("/analyze/text")
async def analyze_text_endpoint(req: TextRequest):
    text_r = analyze_text(req.text)
    url_r  = analyze_urls(extract_urls(req.text))
    rag_r  = rag_search(req.text)
    ti_r   = threat_intel_check(urls=url_r["suspicious"][:2])
    score_global = compute_global_score(EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, ti_r, mode="text_only")
    report = generate_report({"from":req.sender,"subject":req.subject,"full_text":req.text},
                             EMPTY_HEADER, url_r, EMPTY_IMAGE, EMPTY_ATTACH, text_r, rag_r, ti_r, score_global)
    sigma_r = generate_sigma_rules(report, score_global, {"from":req.sender,"subject":req.subject})
    result = {"score_global": score_global, "verdict": report.get("verdict","?"),
              "nlp": text_r, "urls": url_r, "rag": rag_r, "threat_intel": ti_r,
              "report": report, "sigma_rules": sigma_r,
              "email_meta": {"from": req.sender, "subject": req.subject, "date": ""}}
    aid = save_analysis(result, mode="text", sender=req.sender or "", subject=req.subject or "", alert_sent=False)
    result["analysis_id"] = aid
    return result

@app.get("/history")
def history(limit: int = Query(20, ge=1, le=100), verdict: str = Query(None)):
    return get_recent_analyses(limit=limit, verdict_filter=verdict)

@app.get("/history/{analysis_id}")
def history_detail(analysis_id: str):
    row = get_analysis_by_id(analysis_id)
    if not row:
        raise HTTPException(404, f"Analyse {analysis_id} introuvable")
    row["iocs_detail"] = get_iocs_by_analysis(analysis_id)
    return row

@app.get("/stats")
def stats():
    return get_stats()

@app.get("/report/pdf/{analysis_id}")
def report_pdf(analysis_id: str):
    """Génère et retourne le rapport PDF d'une analyse déjà loggée en DB."""
    row = get_analysis_by_id(analysis_id)
    if not row:
        raise HTTPException(404, f"Analyse {analysis_id} introuvable")

    result = row["report_full"]
    result["analysis_id"] = analysis_id
    pdf_path = generate_pdf_report(result)

    return FileResponse(
        path=pdf_path,
        media_type="application/pdf",
        filename=f"DASEC_report_{analysis_id[:8]}.pdf",
        headers={"Content-Disposition": f'attachment; filename="DASEC_report_{analysis_id[:8]}.pdf"'},
    )
'''

with open("/content/api_v2.py", "w") as f:
    f.write(api_code)
print("✅ api_v2.py v5.0 écrit (Sigma + ThreatIntel + SQLite + PDF natifs)")

# ══════════════════════════════════════════════════════════════
sys.path.insert(0, "/content")
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print(f"\n🌍 API : {PUBLIC_URL}")
print(f"📖 Docs : {PUBLIC_URL}/docs")
print(f"🔎 Health : {PUBLIC_URL}/health")
print(f"📜 History : {PUBLIC_URL}/history")
print(f"📊 Stats : {PUBLIC_URL}/stats")
print(f"📄 PDF : {PUBLIC_URL}/report/pdf/{{analysis_id}}")

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    import uvicorn
    config = uvicorn.Config("api_v2:app", host="0.0.0.0", port=8000, log_level="warning", loop="asyncio")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(4)

print("\n✅ Serveur DASEC v5.0 démarré — Sigma + ThreatIntel + SQLite + PDF natifs !")

In [ ]:
import requests, json, time

r = requests.post(f"{PUBLIC_URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
result = r.json()
aid = result["analysis_id"]
print("Score :", result["score_global"], "| Verdict:", result["verdict"], "| ID:", aid)

# Téléchargement du PDF directement depuis l'API
pdf = requests.get(f"{PUBLIC_URL}/report/pdf/{aid}")
with open(f"/content/test_report_{aid[:8]}.pdf", "wb") as f:
    f.write(pdf.content)
print(f"✅ PDF téléchargé : {pdf.status_code}, {len(pdf.content)} octets")

In [ ]:
import requests, json, time

r = requests.post(f"{PUBLIC_URL}/analyze/text", json={
    "text":    "URGENT verify your account password now http://secure-login.tk/verify",
    "subject": "Security Alert",
    "sender":  "security@paypa1.com",
})
result = r.json()
aid = result["analysis_id"]
print("Score :", result["score_global"], "| Verdict:", result["verdict"], "| ID:", aid)

# Téléchargement du PDF directement depuis l'API
pdf = requests.get(f"{PUBLIC_URL}/report/pdf/{aid}")
with open(f"/content/test_report_{aid[:8]}.pdf", "wb") as f:
    f.write(pdf.content)
print(f"✅ PDF téléchargé : {pdf.status_code}, {len(pdf.content)} octets")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE DOCKER — Création de la structure depuis Colab
# ══════════════════════════════════════════════════════════════

import os

DOCKER_DIR = "/content/drive/MyDrive/phishing_dasec/docker"
os.makedirs(DOCKER_DIR, exist_ok=True)

# ── requirements.txt ─────────────────────────────────────────
with open(f"{DOCKER_DIR}/requirements.txt", "w") as f:
    f.write("""fastapi==0.111.0
uvicorn==0.30.1
python-multipart==0.0.9
pyyaml==6.0.1
reportlab==4.2.0
torch==2.3.0+cpu
transformers==4.41.2
accelerate==0.30.0
sentence-transformers==3.0.0
chromadb==0.5.0
scikit-learn==1.5.0
python-Levenshtein==0.25.1
python-whois==0.9.4
pytesseract==0.3.13
Pillow==10.3.0
oletools==0.60.1
python-magic==0.4.27
slack-sdk==3.29.0
google-generativeai==0.7.2
groq==0.9.0
requests==2.32.3
numpy==1.26.4
langgraph==0.1.5
""")
print("✅ requirements.txt créé")

# ── Dockerfile ────────────────────────────────────────────────
with open(f"{DOCKER_DIR}/Dockerfile", "w") as f:
    f.write("""FROM python:3.11-slim

RUN apt-get update && apt-get install -y --no-install-recommends \\
    tesseract-ocr \\
    tesseract-ocr-eng \\
    tesseract-ocr-fra \\
    libmagic1 \\
    libgomp1 \\
    curl \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY api_v2.py .

ENV PYTHONUNBUFFERED=1 \\
    PYTHONDONTWRITEBYTECODE=1 \\
    HF_HUB_OFFLINE=1 \\
    TRANSFORMERS_OFFLINE=1

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "api_v2:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
""")
print("✅ Dockerfile créé")

# ── docker-compose.yml ────────────────────────────────────────
with open(f"{DOCKER_DIR}/docker-compose.yml", "w") as f:
    f.write("""version: \"3.9\"

services:
  dasec-api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: dasec-api
    ports:
      - \"8000:8000\"
    environment:
      - GROQ_API_KEY=${GROQ_API_KEY}
      - GEMINI_API_KEY=${GEMINI_API_KEY}
      - VT_API_KEY=${VT_API_KEY}
      - ABUSEIPDB_KEY=${ABUSEIPDB_KEY}
      - SLACK_TOKEN=${SLACK_TOKEN}
    volumes:
      - ./models:/app/models:ro
      - ./data/vectorstore:/app/data/vectorstore:ro
      - ./data/ioc_cache:/app/data/ioc_cache:ro
      - ./data/dasec.db:/app/data/dasec.db:rw
      - ./data/reports:/tmp/dasec_reports:rw
    restart: unless-stopped
    healthcheck:
      test: [\"CMD\", \"curl\", \"-f\", \"http://localhost:8000/health\"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 60s
""")
print("✅ docker-compose.yml créé")

# ── .dockerignore ─────────────────────────────────────────────
with open(f"{DOCKER_DIR}/.dockerignore", "w") as f:
    f.write("""__pycache__/
*.pyc
*.pyo
.env
.git
*.ipynb
*.csv
*.png
mlflow/
data/raw/
data/processed/
""")
print("✅ .dockerignore créé")

# ── api_v2.py adapté Docker (chemins via variables d'env) ─────
# Lire le fichier existant et remplacer les chemins Colab
with open("/content/api_v2.py", "r") as f:
    api_code = f.read()

api_code = api_code.replace(
    'BASE_DIR        = "/content/drive/MyDrive/phishing_dasec"',
    'BASE_DIR        = os.getenv("BASE_DIR", "/app")'
).replace(
    'VECTORSTORE_DIR = f"{BASE_DIR}/vectorstore"',
    'VECTORSTORE_DIR = os.getenv("VECTORSTORE_DIR", "/app/data/vectorstore")'
).replace(
    'IOC_CACHE_DIR   = f"{BASE_DIR}/ioc_cache"',
    'IOC_CACHE_DIR   = os.getenv("IOC_CACHE_DIR", "/app/data/ioc_cache")'
).replace(
    'DB_PATH         = f"{BASE_DIR}/dasec.db"',
    'DB_PATH         = os.getenv("DB_PATH", "/app/data/dasec.db")'
).replace(
    'PDF_OUTPUT_DIR  = "/tmp/dasec_reports"',
    'PDF_OUTPUT_DIR  = os.getenv("PDF_OUTPUT_DIR", "/tmp/dasec_reports")'
).replace(
    'BERT_PATH  = f"{BASE_DIR}/models/distilbert-5000/final"',
    'BERT_PATH  = os.getenv("BERT_PATH", "/app/models/distilbert-5000/final")'
).replace(
    'with open(f"{BASE_DIR}/models/tfidf_lr_pipeline.pkl", "rb") as f:',
    'with open(os.getenv("TFIDF_PATH", "/app/models/tfidf_lr_pipeline.pkl"), "rb") as f:'
)

with open(f"{DOCKER_DIR}/api_v2.py", "w") as f:
    f.write(api_code)
print("✅ api_v2.py adapté Docker créé")

# ── .env.example ─────────────────────────────────────────────
with open(f"{DOCKER_DIR}/.env.example", "w") as f:
    f.write("""# Copie ce fichier en .env et remplis tes clés
GEMINI_API_KEY=
GROQ_API_KEY=
VT_API_KEY=
ABUSEIPDB_KEY=
SLACK_TOKEN=
""")
print("✅ .env.example créé")

# ── Résumé ────────────────────────────────────────────────────
print(f"\n📦 Structure créée dans {DOCKER_DIR} :")
for f in os.listdir(DOCKER_DIR):
    size = os.path.getsize(f"{DOCKER_DIR}/{f}")
    print(f"   {f:30} {size:6} octets")

print("""
📥 Étapes suivantes sur ton PC :
   1. Télécharge le dossier docker/ depuis Drive
   2. Télécharge aussi models/ et data/ depuis Drive
   3. Place tout dans le même dossier :
      dasec/
      ├── Dockerfile
      ├── docker-compose.yml
      ├── requirements.txt
      ├── api_v2.py
      ├── .env          ← crée depuis .env.example
      ├── models/
      │   ├── distilbert-5000/final/
      │   └── tfidf_lr_pipeline.pkl
      └── data/
          ├── vectorstore/
          ├── ioc_cache/
          └── dasec.db
   4. docker-compose up -d
   5. curl http://localhost:8000/health
""")